# Data Understanding

## Import Library

In [ ]:
!pip install -q ftfy
!pip install langdetect

import warnings
import re
from ftfy import fix_text
from langdetect import detect, DetectorFactory

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Menonaktifkan warning yang tidak diperlukan
warnings.filterwarnings("ignore")

# Menampilkan seluruh kolom ketika DataFrame ditampilkan
pd.set_option("display.max_columns", None)

# Mengatur lebar maksimum tampilan kolom
pd.set_option("display.max_colwidth", 100)

# Format angka desimal agar lebih mudah dibaca
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


## Load Dataset

In [ ]:
df_books = pd.read_csv(
    "GoodReads_100k_books.csv",
    dtype={
        "isbn": "string",
        "isbn13": "string"
    },
    low_memory=False
)

print("Dataset berhasil dimuat.")

Dataset berhasil dimuat.


## Characteristic Dataset

In [ ]:
df_books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   author        100000 non-null  object 
 1   bookformat    96772 non-null   object 
 2   desc          93228 non-null   object 
 3   genre         89533 non-null   object 
 4   img           96955 non-null   object 
 5   isbn          85518 non-null   string 
 6   isbn13        88565 non-null   string 
 7   link          100000 non-null  object 
 8   pages         100000 non-null  int64  
 9   rating        100000 non-null  float64
 10  reviews       100000 non-null  int64  
 11  title         99999 non-null   object 
 12  totalratings  100000 non-null  int64  
dtypes: float64(1), int64(3), object(7), string(2)
memory usage: 9.9+ MB


## Summary Statistics

In [ ]:
df_books.describe()

,pages,rating,reviews,totalratings
count,"100,000.00","100,000.00","100,000.00","100,000.00"
mean,255.01,3.83,181.53,"2,990.76"
std,367.91,0.62,"1,449.45","36,353.38"
min,0.00,0.00,0.00,0.00
25%,135.00,3.66,3.00,31.00
50%,240.00,3.91,15.00,146.00
75%,336.00,4.14,67.00,744.00
max,"70,000.00",5.00,"158,776.00","3,819,326.00"


## Check Data Type

In [ ]:
print(df_books.dtypes)

author                  object
bookformat              object
desc                    object
genre                   object
img                     object
isbn            string[python]
isbn13          string[python]
link                    object
pages                    int64
rating                 float64
reviews                  int64
title                   object
totalratings             int64
dtype: object


## Data Dimensions

In [ ]:
number_of_rows, number_of_columns = df_books.shape

print("Dimensi dataset")
print("-" * 40)
print(f"Jumlah baris : {number_of_rows:,}")
print(f"Jumlah kolom : {number_of_columns}")
print(f"Shape dataset: {df_books.shape}")

Dimensi dataset
----------------------------------------
Jumlah baris : 100,000
Jumlah kolom : 13
Shape dataset: (100000, 13)


## Number of Missing Values

In [ ]:
print("\nMissing Values Dataset:")
print(df_books.isnull().sum())


Missing Values Dataset:
author              0
bookformat       3228
desc             6772
genre           10467
img              3045
isbn            14482
isbn13          11435
link                0
pages               0
rating              0
reviews             0
title               1
totalratings        0
dtype: int64


## Encoding Correction

In [ ]:
# ============================================================
# PERBAIKAN ENCODING TEKS
# ============================================================
# Memperbaiki karakter mojibake atau encoding rusak pada
# kolom-kolom teks sebelum proses filtering dan analisis.

text_columns_to_fix = [
    "title",
    "author",
    "desc",
    "genre",
    "bookformat"
]

for column in text_columns_to_fix:
    df_books[column] = df_books[column].apply(
        lambda value: fix_text(value)
        if isinstance(value, str)
        else value
    )

## Duplicate

In [ ]:
print("\nDuplikat Dataset:", df_books.duplicated().sum())


Duplikat Dataset: 0


### isbn

In [ ]:
duplicate_isbn_count = (
    df_books.loc[df_books["isbn"].notna(), "isbn"]
    .duplicated()
    .sum()
)

print(f"Jumlah ISBN duplikat: {duplicate_isbn_count}")

Jumlah ISBN duplikat: 0


### isbn13

In [ ]:
duplicate_isbn13_count = (
    df_books.loc[df_books["isbn13"].notna(), "isbn13"]
    .duplicated()
    .sum()
)

print(f"Jumlah ISBN-13 duplikat: {duplicate_isbn13_count}")

Jumlah ISBN-13 duplikat: 87840


### link

In [ ]:
duplicate_link_count = (
    df_books.loc[df_books["link"].notna(), "link"]
    .duplicated()
    .sum()
)

print(f"Jumlah link duplikat: {duplicate_link_count}")

Jumlah link duplikat: 0


### desc

In [ ]:
dup_desc = df_books['desc'].duplicated().sum()
print("Total duplikat pada kolom 'desc'  :", dup_desc)

Total duplikat pada kolom 'desc'  : 7500


In [ ]:
duplicate_desc_count = (
    df_books.loc[df_books["desc"].notna(), "desc"]
    .duplicated()
    .sum()
)

print("Duplikat desc tanpa nilai kosong :", duplicate_desc_count)

Duplikat desc tanpa nilai kosong : 729


### title

In [ ]:
dup_title = df_books['title'].duplicated().sum()
print("Total duplikat pada kolom 'title' :", dup_title)

Total duplikat pada kolom 'title' : 2412


In [ ]:
dup_title = (
    df_books.loc[df_books["title"].notna(), "title"]
    .duplicated()
    .sum()
)

print("Duplikat title tanpa nilai kosong:", dup_title)

Duplikat title tanpa nilai kosong: 2412


### title - author - bookformat

In [ ]:
duplicate_book_count = df_books.duplicated(
    subset=["title", "author", "bookformat"]
).sum()

print(
    f"Jumlah duplikat berdasarkan title, author, dan bookformat: "
    f"{duplicate_book_count}"
)

Jumlah duplikat berdasarkan title, author, dan bookformat: 77


## Dataset Scope Filtering

In [ ]:
# 2. Hapus nilai null/kosong pada kolom genre
clean_genres = df_books['genre'].dropna()

# 3. Pecah string berdasarkan koma, hilangkan spasi tambahan, dan jadikan satu series
all_genres = clean_genres.str.split(',').explode().str.strip()

# 4. Filter string kosong (jika ada sisa koma kosong)
all_genres = all_genres[all_genres != '']

# 5. Dapatkan daftar genre unik dan jumlah totalnya
unique_genres = all_genres.unique()
total_unique = len(unique_genres)

print(f"Total Genre Unik: {total_unique}")
print("\nDaftar Genre Unik:")
print(unique_genres)

# 6. (Opsional) Melihat frekuensi setiap genre dari yang terbanyak
print("\nFrekuensi Genre Terbanyak:")
print(all_genres.value_counts().head(20))

Total Genre Unik: 1182

Daftar Genre Unik:
['History' 'Military History' 'Civil War' ... 'Vampire Hunters'
 '1864 Shenandoah Campaign' 'Nairobi']

Frekuensi Genre Terbanyak:
genre
Romance                34324
Fantasy                30798
Fiction                29743
Nonfiction             29446
Historical             18183
Childrens              17304
History                15477
Cultural               14640
Sequential Art         13687
Mystery                13385
Religion               12323
Literature             11422
Paranormal             10748
Science                10676
Science Fiction        10568
Young Adult            10337
Contemporary            8809
European Literature     7773
Historical Fiction      7618
Comics                  7606
Name: count, dtype: int64


In [ ]:
# ============================================================
# FINAL GENRE MAPPING
# ============================================================

genre_mapping = {
    "Self Development": {
        "core": [
            "Personal Development",
            "Growth Mindset"
        ],
        "optional": [
            "Self Help",
            "Human Development",
            "Inspirational",
            "How To",
            "Journaling",
            "Communication",
            "Relationships",
            "Leadership",
            "Spirituality",
        ]
    },

    "Technology": {
        "core": [
            "Computer Science",
            "Programming",
            "Programming Languages",
            "Software",
            "Coding",
            "Artificial Intelligence",
            "Algorithms",
            "Informatics",
            "Computation",
            "Computer Reference"
        ],
        "optional": [
            "Technology",
            "Computers",
            "Information Science",
            "Internet",
            "Web",
            "Website Design",
            "Usability",
            "Virtual Reality",
            "Game Design",
            "Engineering",
            "Electrical Engineering",
            "Technical",
            "Hackers",
        ]
    },

    "Career Development": {
        "core": [
            "Human Resources",
            "Recruitment"
        ],
        "optional": [
            "Leadership",
            "Management",
            "Managers",
            "Entrepreneurship",
            "Business",
            "Buisness",
            "Communication",
            "Finance",
            "Financial Management",
            "Accounting",
            "College",
            "Grad School",
            "Writing",
            "Journalism"
        ]
    },

    "Productivity": {
        "core": [
            "Productivity"
        ],
        "optional": [
            "Self Help",
            "Personal Development",
            "Growth Mindset",
            "How To",
            "Journaling",
            "Management",
            "Leadership"
        ]
    },

    "Psychology": {
        "core": [
            "Psychology",
            "Psychoanalysis",
            "Psychiatry",
            "Mental Health",
            "Mental Illness",
            "Counselling",
            "Neuroscience"
        ],
        "optional": [
            "Brain",
            "Emotion",
            "Human Development",
            "Relationships",
            "Aspergers",
            "Parenting",
        ]
    }
}

In [ ]:
fiction_genres = [
    "Fiction",
    "Adult Fiction",
    "Science Fiction",
    "Fantasy",
    "Romance",
    "Historical Fiction",
    "Young Adult",
    "Paranormal",
    "Mystery",
    "Thriller",
    "Horror",
    "Adventure",
    "Graphic Novels",
    "Comics",
    "Novels",
    "Womens Fiction",
    "Christian Fiction",
    "Contemporary Romance",
    "Historical Romance",
    "Paranormal Romance",
    "Fantasy Romance",
    "Romantic Suspense",
    "Chick Lit",
    "Dystopia",
    "Urban Fantasy",
    "Magical Realism",
    "Ghost Stories",
    "Supernatural",
    "Speculative Fiction"
]

In [ ]:
# ============================================================
# KEYWORDS FOR OPTIONAL GENRE VALIDATION
# ============================================================

category_keywords = {
    "Self Development": [
        "self development",
        "personal development",
        "self improvement",
        "personal growth",
        "growth mindset",
        "self confidence",
        "self awareness",
        "self esteem",
        "positive thinking",
        "personal transformation",
        "improve yourself",
        "life improvement",
        "build confidence",
        "motivation and success"
    ],

    "Technology": [
        "computer science",
        "information technology",
        "software development",
        "software engineering",
        "computer programming",
        "programming language",
        "artificial intelligence",
        "machine learning",
        "data science",
        "database",
        "algorithm",
        "web development",
        "website development",
        "cybersecurity",
        "computer network",
        "user experience",
        "virtual reality",
        "software design",
        "coding"
    ],

    "Career Development": [
        "career development",
        "career growth",
        "career planning",
        "career success",
        "professional development",
        "professional growth",
        "professional skills",
        "job search",
        "job interview",
        "resume writing",
        "career transition",
        "career change",
        "workplace skills",
        "leadership development",
        "management skills",
        "professional communication",
        "work performance",
        "employee development"
    ],

    "Productivity": [
        "personal productivity",
        "work productivity",
        "productive habits",
        "time management",
        "task management",
        "procrastination",
        "deep work",
        "getting things done",
        "work efficiency",
        "focus and concentration",
        "priority management",
        "workflow optimization",
        "work smarter"
    ],

    "Psychology": [
        "psychology",
        "psychological",
        "human behavior",
        "human behaviour",
        "cognitive psychology",
        "social psychology",
        "developmental psychology",
        "personality psychology",
        "mental health",
        "mental illness",
        "emotional regulation",
        "cognitive behavior",
        "cognitive behaviour",
        "brain and behavior",
        "brain and behaviour",
        "psychotherapy",
        "counselling psychology",
        "counseling psychology",
        "psychoanalysis",
        "psychiatry"
    ]
}

In [ ]:
self_development_excluded_genres = {
    "Cookbooks",
    "Cooking",
    "Diets",
    "Nutrition",
    "Personal Finance",
    "Finance",
    "Occult",
    "Witchcraft",
    "Magick",
    "Art Design"
}

In [ ]:
self_development_excluded_normalized = {
    genre.lower()
    for genre in self_development_excluded_genres
}

In [ ]:
# ============================================================
# 4. PARSE GENRES
# ============================================================

def parse_genres(genre_text):
    if pd.isna(genre_text):
        return []

    genres = [
        genre.strip()
        for genre in str(genre_text).split(",")
        if genre.strip()
    ]

    return list(dict.fromkeys(genres))


df_books["genre_list"] = (
    df_books["genre"]
    .apply(parse_genres)
)

In [ ]:
# Genre lowercase digunakan untuk proses pencocokan.
# Kolom genre_list asli tetap dipertahankan untuk tampilan.

df_books["genre_list_normalized"] = (
    df_books["genre_list"]
    .apply(
        lambda genres: {
            genre.strip().lower()
            for genre in genres
        }
    )
)

In [ ]:
# ============================================================
# 5. TEXT FOR FILTERING KEYWORD VALIDATION
# ============================================================

df_books["text_for_filter"] = (
    df_books["title"].fillna("").astype(str)
    + " "
    + df_books["desc"].fillna("").astype(str)
).str.lower()

In [ ]:
# ============================================================
# 6. NONFICTION FILTER
# ============================================================

fiction_genres_normalized = {
    genre.strip().lower()
    for genre in fiction_genres
}

def is_nonfiction_book(genre_list):
    book_genres = {
        str(genre).strip().lower()
        for genre in genre_list
    }

    has_fiction_genre = bool(
        book_genres.intersection(fiction_genres_normalized)
    )

    return not has_fiction_genre


df_books["is_nonfiction"] = df_books["genre_list"].apply(
    is_nonfiction_book
)

In [ ]:
print(df_books["is_nonfiction"].value_counts())

is_nonfiction
True     54530
False    45470
Name: count, dtype: int64


In [ ]:
# # Daftar judul yang ingin dicek
# check_titles = [
#     "The Game",
#     "The Reproductive System",
#     "Just Good Friends",
#     "Glow",
#     "The Daughter She Used To Be"
# ]

# # Filter dataframe
# df_check = df_books[
#     df_books["title"].isin(check_titles)
# ][
#     ["title", "genre", "is_nonfiction"]
# ]

# # Tentukan nama file output terpisah
# nama_file_check = "output_check_titles.md"

# # Buka file baru untuk menulis output
# with open(nama_file_check, "w", encoding="utf-8") as f:
#     f.write("## Hasil Pengecekan Judul Buku\n\n")

#     # Menggunakan to_markdown() agar format tabel di Markdown rapi
#     f.write(df_check.to_markdown(index=False))

# print(f"File markdown terpisah berhasil dibuat dan disimpan sebagai: {nama_file_check}")

In [ ]:
# ============================================================
# KEYWORD MATCHING FUNCTION
# ============================================================

def contains_keyword(text, keyword):
    pattern = (
        r"\b"
        + re.escape(keyword.lower())
        + r"\b"
    )

    return bool(
        re.search(pattern, str(text).lower())
    )


# Memeriksa apakah minimal satu keyword kategori ditemukan.
def has_any_keyword(text, keywords):
    return any(
        contains_keyword(text, keyword)
        for keyword in keywords
    )

In [ ]:
# ============================================================
# 7. TWO-STAGE CATEGORY FILTER
# ============================================================

def assign_categories(row):
    if not row["is_nonfiction"]:
        return []

    book_genres = row["genre_list_normalized"]
    book_text = row["text_for_filter"]

    matched_categories = []

    for category, genre_rules in genre_mapping.items():
        core_genres = {
            genre.lower()
            for genre in genre_rules["core"]
        }

        optional_genres = {
            genre.lower()
            for genre in genre_rules["optional"]
        }

        has_core_genre = bool(
            book_genres.intersection(core_genres)
        )

        has_optional_genre = bool(
            book_genres.intersection(optional_genres)
        )

        has_relevant_keyword = has_any_keyword(
            book_text,
            category_keywords[category]
        )

        # Exclusion hanya diterapkan pada Self Development.
        if category == "Self Development":
            has_excluded_genre = bool(
                book_genres.intersection(
                    self_development_excluded_normalized
                )
            )

            # Buku dengan genre penghambat hanya diterima
            # apabila tetap memiliki keyword pengembangan diri.
            if (
                has_excluded_genre
                and not has_relevant_keyword
            ):
                continue

        # Genre inti merupakan bukti kategori yang kuat.
        if has_core_genre:
            matched_categories.append(category)

        # Genre opsional memerlukan dukungan isi teks.
        elif (
            has_optional_genre
            and has_relevant_keyword
        ):
            matched_categories.append(category)

    return matched_categories

df_books["target_categories"] = df_books.apply(
    assign_categories,
    axis=1
)

In [ ]:
# ============================================================
# 8. FILTER RELEVANT BOOKS
# ============================================================

df_books_filtered = df_books[
    df_books["target_categories"].str.len() > 0
].copy()

df_books_filtered.reset_index(
    drop=True,
    inplace=True
)

print("Jumlah data sebelum filter :", len(df_books))
print("Jumlah data setelah filter :", len(df_books_filtered))

Jumlah data sebelum filter : 100000
Jumlah data setelah filter : 4173


In [ ]:
fiction_books_remaining = df_books_filtered[
    df_books_filtered["is_nonfiction"] == False
]

print(
    "Jumlah buku berindikasi fiksi yang masih masuk hasil filter:",
    len(fiction_books_remaining)
)

Jumlah buku berindikasi fiksi yang masih masuk hasil filter: 0


In [ ]:
problematic_titles = [
    "The Game",
    "The Reproductive System",
    "Just Good Friends",
    "Glow",
    "The Daughter She Used To Be",
    "Forgiven"
]

remaining_problematic_books = df_books_filtered[
    df_books_filtered["title"].isin(problematic_titles)
]

print(
    remaining_problematic_books[
        ["title", "genre", "target_categories"]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [title, genre, target_categories]
Index: []


In [ ]:
# ============================================================
# 9. CATEGORY DISTRIBUTION
# ============================================================

category_distribution = (
    df_books_filtered["target_categories"]
    .explode()
    .value_counts()
    .reindex(genre_mapping.keys(), fill_value=0)
)

print("Distribusi data berdasarkan kategori:")
print(category_distribution)

Distribusi data berdasarkan kategori:
target_categories
Self Development       787
Technology            1088
Career Development      55
Productivity           267
Psychology            2809
Name: count, dtype: int64


In [ ]:
# ============================================================
# 10. CATEGORY DISTRIBUTION TABLE
# ============================================================

category_distribution_table = (
    category_distribution
    .rename_axis("category")
    .reset_index(name="book_count")
)

category_distribution_table["percentage"] = (
    category_distribution_table["book_count"]
    / len(df_books_filtered)
    * 100
).round(2)

print(category_distribution_table.to_string(index=False))

          category  book_count  percentage
  Self Development         787       18.86
        Technology        1088       26.07
Career Development          55        1.32
      Productivity         267        6.40
        Psychology        2809       67.31


In [ ]:
# ============================================================
# 11. NUMBER OF CATEGORIES PER BOOK
# ============================================================

category_count_per_book = (
    df_books_filtered["target_categories"]
    .str.len()
    .value_counts()
    .sort_index()
)

print("Jumlah kategori yang dimiliki setiap buku:")
print(category_count_per_book)

Jumlah kategori yang dimiliki setiap buku:
target_categories
1    3483
2     549
3     139
4       2
Name: count, dtype: int64


In [ ]:
print(
    "Jumlah buku unik yang masuk lima kategori:",
    len(df_books_filtered)
)

print(
    "Jumlah keseluruhan label kategori:",
    df_books_filtered["target_categories"]
    .str.len()
    .sum()
)

Jumlah buku unik yang masuk lima kategori: 4173
Jumlah keseluruhan label kategori: 5006


In [ ]:
# # ============================================================
# # 12. CHECK BOOKS WITH MANY CATEGORIES
# # ============================================================

# # Filter buku dengan kategori >= 4
# books_with_many_categories = df_books_filtered[
#     df_books_filtered["target_categories"].str.len() >= 4
# ]

# # Tentukan nama file output
# nama_file_many_cat = "output_many_categories.md"

# # Buka file dalam mode write ('w')
# with open(nama_file_many_cat, "w", encoding="utf-8") as f:
#     f.write("## Buku dengan Banyak Kategori (>= 4)\n\n")

#     # Menuliskan data maksimal 30 baris ke dalam format markdown
#     f.write(
#         books_with_many_categories[
#             [
#                 "title",
#                 "author",
#                 "genre",
#                 "target_categories"
#             ]
#         ].head(30).to_markdown(index=False)
#     )

# print(f"File markdown terpisah berhasil dibuat dan disimpan sebagai: {nama_file_many_cat}")

In [ ]:
# # ============================================================
# # 13. SAMPLE BOOKS PER CATEGORY
# # ============================================================

# # Tentukan nama file output
# nama_file_sample = "output_sample_per_category.md"

# # Buka file dalam mode write ('w')
# with open(nama_file_sample, "w", encoding="utf-8") as f:
#     f.write("# 13. Sample Books per Category\n\n")

#     for category in genre_mapping.keys():
#         category_books = df_books_filtered[
#             df_books_filtered["target_categories"].apply(
#                 lambda categories: category in categories
#             )
#         ]

#         sample_size = min(70, len(category_books))

#         # Pengecekan agar tidak mencoba memproses kategori yang kosong
#         if sample_size > 0:
#             category_sample = category_books[
#                 [
#                     "title",
#                     "author",
#                     "genre",
#                     "target_categories"
#                 ]
#             ].sample(
#                 n=sample_size,
#                 random_state=42
#             )

#             # Mengganti baris pemisah (===) dengan Heading Markdown (##)
#             f.write(f"## KATEGORI: {category}\n\n")

#             # Menggunakan to_markdown agar tampil sebagai tabel rapi
#             f.write(category_sample.to_markdown(index=False))

#             # Memberikan jarak antar kategori dan garis pemisah markdown (---)
#             f.write("\n\n---\n\n")

# print(f"File markdown terpisah berhasil dibuat dan disimpan sebagai: {nama_file_sample}")

In [ ]:
books_with_five_categories = df_books_filtered[
    df_books_filtered["target_categories"].str.len() == 5
]

print(
    books_with_five_categories[
        ["title", "author", "genre", "target_categories"]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [title, author, genre, target_categories]
Index: []


In [ ]:
books_with_four_categories = df_books_filtered[
    df_books_filtered["target_categories"].str.len() == 4
]

print(
    books_with_four_categories[
        ["title", "author", "genre", "target_categories"]
    ].head(20).to_string(index=False)
)

                                                                                                                                              title                             author                                                                                                                                                            genre                                                target_categories
                                                                                                    Personal Kanban: Mapping Work | Navigating Life Jim  Benson,Tonianne DeMaria Barry Productivity,Nonfiction,Business,Self Help,Business,Management,Self Help,Personal Development,Leadership,Computer Science,Software,Psychology,Science,Technology         [Self Development, Technology, Productivity, Psychology]
The 3 Secrets to Effective Time Investment: Achieve More Success with Less Stress: Foreword by Cal Newport, Author of So Good They Can't Ignore You           Elizabeth Grace Saunders

In [ ]:
print(df_books_filtered.columns.tolist())

['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings', 'genre_list', 'genre_list_normalized', 'text_for_filter', 'is_nonfiction', 'target_categories']


In [ ]:
df_books_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4173 entries, 0 to 4172
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   author                 4173 non-null   object 
 1   bookformat             4112 non-null   object 
 2   desc                   4083 non-null   object 
 3   genre                  4173 non-null   object 
 4   img                    4147 non-null   object 
 5   isbn                   3876 non-null   string 
 6   isbn13                 3907 non-null   string 
 7   link                   4173 non-null   object 
 8   pages                  4173 non-null   int64  
 9   rating                 4173 non-null   float64
 10  reviews                4173 non-null   int64  
 11  title                  4173 non-null   object 
 12  totalratings           4173 non-null   int64  
 13  genre_list             4173 non-null   object 
 14  genre_list_normalized  4173 non-null   object 
 15  text

## EDA

In [ ]:
df_books_eda = df_books_filtered.copy()

# Panjang deskripsi dalam jumlah karakter
df_books_eda["desc_length"] = (
    df_books_eda["desc"]
    .fillna("")
    .astype(str)
    .str.len()
)

# Panjang judul dalam jumlah karakter
df_books_eda["title_length"] = (
    df_books_eda["title"]
    .fillna("")
    .astype(str)
    .str.len()
)

# Jumlah genre asli setiap buku
df_books_eda["genre_count"] = (
    df_books_eda["genre_list"]
    .str.len()
)

# Jumlah kategori penelitian setiap buku
df_books_eda["target_category_count"] = (
    df_books_eda["target_categories"]
    .str.len()
)

NameError: name 'df_books_filtered' is not defined

In [ ]:
category_distribution = (
    df_books_eda["target_categories"]
    .explode()
    .value_counts()
)

plt.figure(figsize=(10, 5))

category_distribution.plot(
    kind="bar"
)

plt.title("Distribusi Buku Berdasarkan Kategori")
plt.xlabel("Kategori")
plt.ylabel("Jumlah Buku")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
print(category_distribution)

In [ ]:
bookformat_distribution = (
    df_books_eda["bookformat"]
    .fillna("Unknown")
    .value_counts()
    .head(15)
)

plt.figure(figsize=(10, 6))

bookformat_distribution.sort_values().plot(
    kind="barh"
)

plt.title("Distribusi Format Buku")
plt.xlabel("Jumlah Buku")
plt.ylabel("Format Buku")
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
top_authors = (
    df_books_eda["author"]
    .fillna("Unknown")
    .value_counts()
    .head(15)
)

plt.figure(figsize=(10, 6))

top_authors.sort_values().plot(
    kind="barh"
)

plt.title("15 Penulis dengan Jumlah Buku Terbanyak")
plt.xlabel("Jumlah Buku")
plt.ylabel("Penulis")
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df_books_eda["rating"].dropna(),
    bins=20,
    edgecolor="black"
)

plt.title("Distribusi Rating Buku")
plt.xlabel("Rating")
plt.ylabel("Jumlah Buku")
plt.tight_layout()
plt.show()

In [ ]:
data_rating = df_books_eda["rating"].dropna()
mean_rating = data_rating.mean()
median_rating = data_rating.median()

plt.figure(figsize=(10, 5))

plt.hist(
    data_rating,
    bins=20,
    edgecolor="black",
    color="lightgreen",
    alpha=0.8
)

plt.axvline(mean_rating, color='red', linestyle='dashed', linewidth=2, label=f'Rata-rata: {mean_rating:.2f}')
plt.axvline(median_rating, color='blue', linestyle='solid', linewidth=2, label=f'Median: {median_rating:.2f}')

plt.title("Distribusi Rating Buku", fontsize=14, pad=15)
plt.xlabel("Rating")
plt.ylabel("Jumlah Buku")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
print(
    df_books_eda["rating"]
    .describe()
    .round(2)
)

NameError: name 'df_books_eda' is not defined

In [ ]:
zero_rating_books = df_books_eda[
    df_books_eda["rating"] == 0
]

print(
    zero_rating_books[
        ["title", "rating", "reviews", "totalratings"]
    ].to_string(index=False)
)

NameError: name 'df_books_eda' is not defined

In [ ]:
pages_upper_limit = df_books_eda["pages"].quantile(0.99)

plt.figure(figsize=(10, 5))

plt.hist(
    df_books_eda.loc[
        df_books_eda["pages"] <= pages_upper_limit,
        "pages"
    ].dropna(),
    bins=30,
    edgecolor="black"
)

plt.title("Distribusi Jumlah Halaman Buku")
plt.xlabel("Jumlah Halaman")
plt.ylabel("Jumlah Buku")
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
# # Filter buku dengan 0 halaman
# zero_page_books = df_books_eda[
#     df_books_eda["pages"] == 0
# ]

# # Hitung total buku dengan 0 halaman
# jumlah_buku = len(zero_page_books)

# # Tentukan nama file output
# nama_file_zero = "output_zero_pages.md"

# # Buka file untuk ditulis
# with open(nama_file_zero, "w", encoding="utf-8") as f:
#     f.write("# Data Buku dengan 0 Halaman\n\n")

#     # Menuliskan jumlah buku
#     f.write(f"**Jumlah buku dengan 0 halaman:** {jumlah_buku}\n\n")

#     # Menuliskan SEMUA data ke dalam format tabel Markdown
#     # Perhatikan: .head(20) dihilangkan agar semua data tercetak
#     # dan .to_string() diganti dengan .to_markdown()
#     f.write(
#         zero_page_books[
#             ["title", "bookformat", "pages"]
#         ].to_markdown(index=False)
#     )

# print(f"File markdown berhasil dibuat dan disimpan sebagai: {nama_file_zero}")

In [ ]:
plt.figure(figsize=(10, 4))

plt.boxplot(
    df_books_eda["pages"].dropna(),
    vert=False
)

plt.title("Boxplot Jumlah Halaman Buku")
plt.xlabel("Jumlah Halaman")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# # 1. Definisikan dan buat variabel datanya terlebih dahulu
# page_outliers = df_books_eda[
#     df_books_eda["pages"] > 617
# ].sort_values(
#     "pages",
#     ascending=False
# )

# # 2. Tentukan nama file output
# nama_file_outliers = "output_page_outliers.md"

# # 3. Buka file dalam mode write dan gunakan variabel yang sudah dibuat di atas
# with open(nama_file_outliers, "w", encoding="utf-8") as f:
#     f.write("# Data Outlier: Buku dengan Halaman > 617\n\n")

#     # Gunakan .to_markdown() dan tanpa .head()
#     f.write(
#         page_outliers[
#             ["title", "bookformat", "pages"]
#         ].to_markdown(index=False)
#     )

# print(f"File markdown berhasil dibuat: {nama_file_outliers}")

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    np.log1p(df_books_eda["reviews"]),
    bins=30,
    edgecolor="black"
)

plt.title("Distribusi Jumlah Ulasan Buku")
plt.xlabel("Log(1 + Jumlah Ulasan)")
plt.ylabel("Jumlah Buku")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    np.log1p(df_books_eda["totalratings"]),
    bins=30,
    edgecolor="black"
)

plt.title("Distribusi Total Pemberi Rating")
plt.xlabel("Log(1 + Total Pemberi Rating)")
plt.ylabel("Jumlah Buku")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    df_books_eda["desc_length"],
    bins=30,
    edgecolor="black"
)

plt.title("Distribusi Panjang Deskripsi Buku")
plt.xlabel("Panjang Deskripsi dalam Karakter")
plt.ylabel("Jumlah Buku")
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
empty_description_count = (
    df_books_eda["desc_length"] == 0
).sum()

print(
    "Jumlah buku tanpa deskripsi:",
    empty_description_count
)

NameError: name 'df_books_eda' is not defined

In [ ]:
genre_count_distribution = (
    df_books_eda["genre_count"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 5))

genre_count_distribution.plot(
    kind="bar"
)

plt.title("Distribusi Jumlah Genre per Buku")
plt.xlabel("Jumlah Genre")
plt.ylabel("Jumlah Buku")
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
df_books_category = df_books_eda.explode(
    "target_categories"
).copy()

category_order = list(genre_mapping.keys())

rating_by_category = [
    df_books_category.loc[
        df_books_category["target_categories"] == category,
        "rating"
    ].dropna()
    for category in category_order
]

plt.figure(figsize=(11, 6))

plt.boxplot(
    rating_by_category,
    labels=category_order
)

plt.title("Distribusi Rating Berdasarkan Kategori")
plt.xlabel("Kategori")
plt.ylabel("Rating")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
pages_by_category = [
    df_books_category.loc[
        df_books_category["target_categories"] == category,
        "pages"
    ].dropna()
    for category in category_order
]

plt.figure(figsize=(11, 6))

plt.boxplot(
    pages_by_category,
    labels=category_order,
    showfliers=False
)

plt.title("Distribusi Jumlah Halaman Berdasarkan Kategori")
plt.xlabel("Kategori")
plt.ylabel("Jumlah Halaman")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

NameError: name 'category_order' is not defined

In [ ]:
average_rating_by_category = (
    df_books_category
    .groupby("target_categories")["rating"]
    .mean()
    .reindex(category_order)
)

plt.figure(figsize=(10, 5))

average_rating_by_category.plot(
    kind="bar"
)

plt.title("Rata-rata Rating Berdasarkan Kategori")
plt.xlabel("Kategori")
plt.ylabel("Rata-rata Rating")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 5)
plt.tight_layout()
plt.show()

NameError: name 'df_books_category' is not defined

In [ ]:
median_totalratings_by_category = (
    df_books_category
    .groupby("target_categories")["totalratings"]
    .median()
    .reindex(category_order)
)

plt.figure(figsize=(10, 5))

median_totalratings_by_category.plot(
    kind="bar"
)

plt.title("Median Total Pemberi Rating Berdasarkan Kategori")
plt.xlabel("Kategori")
plt.ylabel("Median Total Pemberi Rating")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

NameError: name 'df_books_category' is not defined

### Correlation Analysis

In [ ]:
numeric_columns = [
    "pages",
    "rating",
    "reviews",
    "totalratings",
    "desc_length",
    "title_length",
    "genre_count",
    "target_category_count"
]

correlation_matrix = (
    df_books_eda[numeric_columns]
    .corr(method="spearman")
)

print(correlation_matrix.round(2))

NameError: name 'df_books_eda' is not defined

In [ ]:
# 1. Menghitung matriks korelasi (Kode asli Anda)
correlation_matrix = (
    df_books_eda[numeric_columns]
    .corr(method="spearman")
)

# 2. Mengatur ukuran kanvas visualisasi
plt.figure(figsize=(10, 8))

# 3. Membuat heatmap
sns.heatmap(
    correlation_matrix,
    annot=True,          # Menampilkan angka korelasi di dalam kotak
    cmap="Blues",        # Menggunakan gradasi warna biru murni
    fmt=".2f",           # Membulatkan angka menjadi 2 desimal (sama seperti .round(2))
    linewidths=0.5,      # Memberikan garis putih pemisah antar kotak agar rapi
    vmin=-1, vmax=1      # Rentang korelasi selalu dari -1 hingga 1
)

# 4. Menambahkan judul dan menampilkan plot
plt.title("Spearman Correlation Matrix", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

NameError: name 'df_books_eda' is not defined

In [ ]:
# plt.figure(figsize=(10, 8))

# plt.imshow(
#     correlation_matrix,
#     aspect="auto"
# )

# plt.colorbar(
#     label="Koefisien Korelasi Spearman"
# )

# plt.xticks(
#     range(len(numeric_columns)),
#     numeric_columns,
#     rotation=45,
#     ha="right"
# )

# plt.yticks(
#     range(len(numeric_columns)),
#     numeric_columns
# )

# plt.title("Korelasi Atribut Numerik")
# plt.tight_layout()
# plt.show()

### Outlier Detection

In [ ]:
def count_outliers_iqr(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (dataframe[column] < lower_bound)
        | (dataframe[column] > upper_bound)
    ).sum()

    return lower_bound, upper_bound, outlier_count

In [ ]:
outlier_columns = [
    "pages",
    "rating",
    "reviews",
    "totalratings"
]

for column in outlier_columns:
    lower_bound, upper_bound, outlier_count = (
        count_outliers_iqr(
            df_books_eda,
            column
        )
    )

    print(f"\nKolom: {column}")
    print(f"Batas bawah : {lower_bound:.2f}")
    print(f"Batas atas  : {upper_bound:.2f}")
    print(f"Jumlah outlier: {outlier_count}")

## Data Cleaning

In [ ]:
# Menghapus deskripsi kosong (NaN), string kosong, dan deskripsi yang HANYA berupa URL
# Menghapus deskripsi kosong, string kosong, dan deskripsi berupa URL
# df_books_clean = df_books_filtered[
#     df_books_filtered["desc"].notna()
#     & df_books_filtered["desc"].str.strip().ne("")
#     & ~df_books_filtered["desc"].str.strip().str.match(r'^https?://\S+$',na=False)
# ].copy()
df_books_clean = df_books_filtered[
    df_books_filtered["desc"].notna()
    & df_books_filtered["desc"].str.strip().ne("")
].copy()

# Mengubah nilai nol yang menandakan data tidak tersedia.
df_books_clean["rating"] = (
    df_books_clean["rating"]
    .replace(0, np.nan)
)

df_books_clean["pages"] = (
    df_books_clean["pages"]
    .replace(0, np.nan)
)

# Standardisasi format buku.
df_books_clean["bookformat"] = (
    df_books_clean["bookformat"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .str.title()
)

# Gambar kosong disimpan sebagai string kosong untuk deployment.
df_books_clean["img"] = (
    df_books_clean["img"]
    .fillna("")
)

df_books_clean.reset_index(
    drop=True,
    inplace=True
)

print("Jumlah data sebelum penanganan:", len(df_books_filtered))
print("Jumlah dataset bersih:", len(df_books_clean))

Jumlah data sebelum penanganan: 4173
Jumlah dataset bersih: 4083


# Data Preparation

### Handling Duplicate

In [ ]:
duplicate_books = df_books_clean.duplicated(
    subset=["title", "author", "bookformat"]
).sum()

print(
    "Jumlah duplikat title, author, dan bookformat:",
    duplicate_books
)

Jumlah duplikat title, author, dan bookformat: 1


In [ ]:
# # 1. Definisikan variabel datanya
# duplicate_book_data = df_books_clean[
#     df_books_clean.duplicated(
#         subset=["title", "author", "bookformat"],
#         keep=False
#     )
# ].sort_values(
#     ["title", "author", "bookformat"]
# )

# # 2. Tentukan nama file output
# nama_file_duplicates = "output_duplicate_books.md"

# # 3. Buka file dalam mode write
# with open(nama_file_duplicates, "w", encoding="utf-8") as f:
#     f.write("# Data Buku Duplikat\n\n")

#     # Gunakan .to_markdown() dan tanpa .head()
#     f.write(
#         duplicate_book_data[
#             [
#                 "title",
#                 "author",
#                 "bookformat",
#                 "isbn",
#                 "isbn13",
#                 "pages",
#                 "rating",
#                 "link"
#             ]
#         ].to_markdown(index=False)
#     )

# print(f"File markdown berhasil dibuat: {nama_file_duplicates}")

In [ ]:
duplicate_isbn = (
    df_books_clean.loc[
        df_books_clean["isbn"].notna(),
        "isbn"
    ]
    .duplicated()
    .sum()
)

duplicate_isbn13 = (
    df_books_clean.loc[
        df_books_clean["isbn13"].notna(),
        "isbn13"
    ]
    .duplicated()
    .sum()
)

print("Jumlah ISBN duplikat   :", duplicate_isbn)
print("Jumlah ISBN-13 duplikat:", duplicate_isbn13)

Jumlah ISBN duplikat   : 0
Jumlah ISBN-13 duplikat: 3668


In [ ]:
print(
    "Jumlah ISBN-13 non-null:",
    df_books_clean["isbn13"].notna().sum()
)

print(
    "Jumlah ISBN-13 unik:",
    df_books_clean["isbn13"].nunique(dropna=True)
)

Jumlah ISBN-13 non-null: 3841
Jumlah ISBN-13 unik: 173


In [ ]:
# # 1. Hitung frekuensi seluruh ISBN13 tanpa batasan .head()
# isbn13_frequency = (
#     df_books_clean["isbn13"]
#     .dropna()
#     .value_counts()
# )

# # 2. Ubah hasil Series menjadi DataFrame agar tampilan tabel Markdown rapi
# df_isbn_freq = isbn13_frequency.reset_index()
# # Ubah nama kolom agar lebih jelas
# df_isbn_freq.columns = ["isbn13", "frequency"]

# # 3. Tentukan nama file output
# nama_file_isbn = "output_isbn13_frequency.md"

# # 4. Buka file dalam mode write
# with open(nama_file_isbn, "w", encoding="utf-8") as f:
#     f.write("# Frekuensi Kemunculan ISBN13\n\n")

#     # Cetak ke dalam format Markdown tanpa membatasi baris
#     f.write(df_isbn_freq.to_markdown(index=False))

# print(f"File markdown berhasil dibuat: {nama_file_isbn}")

In [ ]:
print(
    df_books_clean[
        ["title", "isbn", "isbn13"]
    ].head(30).to_string(index=False)
)

                                                                                                                          title       isbn      isbn13
                                                                       Genuine Happiness: Meditation as the Path to Fulfillment 047146984X    9.78E+12
                                                                                          Happiness: Lessons from a New Science  143037013    9.78E+12
                                                                                         The Principles of Psychology: Volume 1  486203816    9.78E+12
                                                                 Character Strengths and Virtues: A Handbook and Classification  195167015    9.78E+12
                                                                                                          Irrational Exuberance  767923634    9.78E+12
                                                                             The Transsexual E

In [ ]:
isbn13_text = (
    df_books_clean["isbn13"]
    .dropna()
    .astype(str)
    .str.strip()
)

valid_isbn13_format = isbn13_text.str.fullmatch(r"\d{13}")

print("ISBN-13 dengan format 13 digit :", valid_isbn13_format.sum())
print("ISBN-13 dengan format tidak valid:", (~valid_isbn13_format).sum())

ISBN-13 dengan format 13 digit : 0
ISBN-13 dengan format tidak valid: 3841


In [ ]:
# # 1. Definisikan variabel datanya (sesuai kode asli Anda)
# invalid_isbn13_values = df_books_clean.loc[
#     df_books_clean["isbn13"].notna()
#     & ~df_books_clean["isbn13"]
#         .astype(str)
#         .str.strip()
#         .str.fullmatch(r"\d{13}"),
#     ["title", "isbn13"]
# ]

# # 2. Tentukan nama file output
# nama_file_invalid = "output_invalid_isbn13.md"

# # 3. Buka file dalam mode write
# with open(nama_file_invalid, "w", encoding="utf-8") as f:
#     f.write("# Data ISBN13 Tidak Valid\n\n")

#     # Gunakan .to_markdown() dan pastikan .head() dihapus agar semua data diekstrak
#     f.write(
#         invalid_isbn13_values.to_markdown(index=False)
#     )

# print(f"File markdown berhasil dibuat: {nama_file_invalid}")

In [ ]:
df_books_clean = df_books_clean.rename(
    columns={"isbn13": "isbn13_raw_invalid"}
)

In [ ]:
# ============================================================
# NORMALISASI DAN VALIDASI ISBN-10
# ============================================================

def normalize_isbn10(value):
    if pd.isna(value):
        return np.nan

    isbn = re.sub(
        r"[^0-9Xx]",
        "",
        str(value).strip()
    ).upper()

    # Menambahkan nol di depan untuk ISBN numerik
    # yang kehilangan leading zero saat dibaca sebagai angka
    if isbn.isdigit() and 7 <= len(isbn) < 10:
        isbn = isbn.zfill(10)

    return isbn

def is_valid_isbn10(isbn):
    if pd.isna(isbn):
        return False

    if not re.fullmatch(r"\d{9}[\dX]", isbn):
        return False

    total = 0

    for position, character in enumerate(isbn):
        value = 10 if character == "X" else int(character)
        weight = 10 - position
        total += weight * value

    return total % 11 == 0

# Membuat ISBN-10 yang sudah dinormalisasi
df_books_clean["isbn10_clean"] = (
    df_books_clean["isbn"]
    .apply(normalize_isbn10)
)

# Menandai apakah ISBN-10 lolos checksum.
df_books_clean["isbn10_valid"] = (
    df_books_clean["isbn10_clean"]
    .apply(is_valid_isbn10)
)

print(
    df_books_clean["isbn10_valid"]
    .value_counts()
)

isbn10_valid
True     3795
False     288
Name: count, dtype: int64


In [ ]:
# ============================================================
# KONVERSI ISBN-10 MENJADI ISBN-13
# ============================================================

def convert_isbn10_to_isbn13(isbn10):
    if not is_valid_isbn10(isbn10):
        return np.nan

    isbn12 = "978" + isbn10[:9]

    weighted_sum = sum(
        int(digit) * (1 if index % 2 == 0 else 3)
        for index, digit in enumerate(isbn12)
    )

    check_digit = (10 - weighted_sum % 10) % 10

    return isbn12 + str(check_digit)

# Membentuk ISBN-13 hanya dari ISBN-10 yang valid.
df_books_clean["isbn13_clean"] = (
    df_books_clean["isbn10_clean"]
    .apply(convert_isbn10_to_isbn13)
)

In [ ]:
valid_generated_isbn13 = (
    df_books_clean["isbn13_clean"]
    .dropna()
    .astype(str)
    .str.fullmatch(r"\d{13}")
    .sum()
)

print(
    "Jumlah ISBN-13 yang berhasil dibentuk:",
    valid_generated_isbn13
)

Jumlah ISBN-13 yang berhasil dibentuk: 3795


In [ ]:
# ============================================================
# MENAMPILKAN PERBANDINGAN ALUR TRANSFORMASI ISBN
# ============================================================

# 1. Tentukan kolom-kolom yang ingin kita bandingkan jejaknya
kolom_jejak_isbn = [
    "isbn",          # ISBN-10 awal (Mentah)
    "isbn10_clean",  # ISBN-10 setelah dibersihkan dan ditambah leading zero
    "isbn10_valid",  # Status validasi checksum (True/False)
    "isbn13_clean"   # Hasil konversi ke ISBN-13
]

# 2. Filter data agar kita hanya melihat baris yang memang memiliki ISBN awal
# dan ambil 10 data pertama sebagai sampel perbandingan
df_perbandingan = (
    df_books_clean.dropna(subset=["isbn"])[kolom_jejak_isbn]
    .head(10)
)

# 3. Tampilkan dalam format tabel Markdown
print("### Alur Perubahan Data ISBN\n")
print(df_perbandingan.to_markdown(index=False))

### Alur Perubahan Data ISBN

| isbn       | isbn10_clean   | isbn10_valid   |   isbn13_clean |
|:-----------|:---------------|:---------------|---------------:|
| 047146984X | 047146984X     | True           |  9780471469841 |
| 143037013  | 0143037013     | True           |  9780143037019 |
| 486203816  | 0486203816     | True           |  9780486203812 |
| 195167015  | 0195167015     | True           |  9780195167016 |
| 767923634  | 0767923634     | True           |  9780767923637 |
| 807762725  | 0807762725     | True           |  9780807762721 |
| 098243877X | 098243877X     | True           |  9780982438770 |
| 743243366  | 0743243366     | True           |  9780743243360 |
| 786869143  | 0786869143     | True           |  9780786869145 |
| 785274324  | 0785274324     | True           |  9780785274322 |


In [ ]:
# # 1. Pastikan variabel invalid_isbn10_data sudah didefinisikan (sesuai kode Anda)
# invalid_isbn10_data = df_books_clean[
#     ~df_books_clean["isbn10_valid"]
# ]

# # 2. Tentukan nama file output
# nama_file_invalid_isbn10 = "output_invalid_isbn10.md"

# # 3. Buka file dalam mode write
# with open(nama_file_invalid_isbn10, "w", encoding="utf-8") as f:
#     f.write("# Data ISBN Tidak Valid (Berdasarkan Pengecekan ISBN10)\n\n")

#     # TAMBAHKAN .fillna("") SEBELUM .to_markdown()
#     f.write(
#         invalid_isbn10_data[
#             [
#                 "title",
#                 "isbn",
#                 "isbn10_clean",
#                 "isbn13_clean"
#             ]
#         ].fillna("").to_markdown(index=False)
#     )

# print(f"File markdown berhasil dibuat: {nama_file_invalid_isbn10}")

In [ ]:
missing_isbn_count = df_books_clean["isbn"].isna().sum()

invalid_non_missing_isbn_count = (
    df_books_clean["isbn"].notna()
    & ~df_books_clean["isbn10_valid"]
).sum()

print("ISBN kosong                    :", missing_isbn_count)
print("ISBN terisi tetapi tidak valid :", invalid_non_missing_isbn_count)

ISBN kosong                    : 274
ISBN terisi tetapi tidak valid : 14


In [ ]:
duplicate_isbn13_clean = (
    df_books_clean.loc[
        df_books_clean["isbn13_clean"].notna(),
        "isbn13_clean"
    ]
    .duplicated()
    .sum()
)

print(
    "Jumlah duplikat ISBN-13 hasil konversi:",
    duplicate_isbn13_clean
)

Jumlah duplikat ISBN-13 hasil konversi: 0


In [ ]:
# Validasi hasil konversi.
print(
    "Jumlah ISBN-13 non-null:",
    df_books_clean["isbn13_clean"].notna().sum()
)

print(
    "Jumlah ISBN-13 unik:",
    df_books_clean["isbn13_clean"].nunique(dropna=True)
)

Jumlah ISBN-13 non-null: 3795
Jumlah ISBN-13 unik: 3795


### Handling Missing Values

In [ ]:
missing_values_clean = df_books_clean.isna().sum()

print(missing_values_clean[missing_values_clean > 0])

isbn                  274
isbn13_raw_invalid    242
pages                 175
rating                  4
isbn10_clean          274
isbn13_clean          288
dtype: int64


In [ ]:
empty_description_count = (
    df_books_clean["desc"].isna()
    | df_books_clean["desc"].str.strip().eq("")
).sum()

print("Jumlah deskripsi kosong:", empty_description_count)

Jumlah deskripsi kosong: 0


## Final Validation

In [ ]:
print("Dimensi final:", df_books_clean.shape)
print("\nMissing values final:")
print(
    df_books_clean
    .isna()
    .sum()
    .loc[
        lambda values: values > 0
    ]
)

print(
    "\nDeskripsi kosong:",
    (
        df_books_clean["desc"].isna()
        | df_books_clean["desc"]
            .str.strip()
            .eq("")
    ).sum()
)
print(
    "Rating di luar rentang 1–5:",
    ((df_books_clean["rating"] < 1)
     | (df_books_clean["rating"] > 5)).sum()
)

print(
    "Jumlah halaman negatif:",
    (df_books_clean["pages"] < 0).sum()
)

print(
    "Jumlah reviews negatif:",
    (df_books_clean["reviews"] < 0).sum()
)

print(
    "Jumlah totalratings negatif:",
    (df_books_clean["totalratings"] < 0).sum()
)

Dimensi final: (4083, 21)

Missing values final:
isbn                  274
isbn13_raw_invalid    242
pages                 175
rating                  4
isbn10_clean          274
isbn13_clean          288
dtype: int64

Deskripsi kosong: 0
Rating di luar rentang 1–5: 0
Jumlah halaman negatif: 0
Jumlah reviews negatif: 0
Jumlah totalratings negatif: 0


In [ ]:
final_category_distribution = (
    df_books_clean["target_categories"]
    .explode()
    .value_counts()
    .reindex(genre_mapping.keys(), fill_value=0)
)

print(final_category_distribution)

target_categories
Self Development       772
Technology            1064
Career Development      53
Productivity           258
Psychology            2751
Name: count, dtype: int64


In [ ]:
# # 1. Definisikan variabel datanya
# invalid_isbn_data = df_books_clean[
#     df_books_clean["isbn"].notna()
#     & ~df_books_clean["isbn10_valid"]
# ]

# # 2. Tentukan nama file output
# nama_file_invalid_isbn = "output_invalid_isbn.md"

# # 3. Buka file dalam mode write
# with open(nama_file_invalid_isbn, "w", encoding="utf-8") as f:
#     f.write("# Data ISBN Tidak Valid\n\n")

#     # Cetak ke dalam format Markdown
#     # Menggunakan .fillna("") sebagai langkah preventif agar tidak terjadi error tabulate
#     f.write(
#         invalid_isbn_data[
#             ["title", "isbn", "isbn10_clean"]
#         ].fillna("").to_markdown(index=False)
#     )

# print(f"File markdown berhasil dibuat: {nama_file_invalid_isbn}")

In [ ]:
# ============================================================
# PEMILIHAN TEKS UNTUK SENTENCE-BERT
# ============================================================
# model_text menggabungkan informasi semantik utama buku:
# judul, genre, dan deskripsi.

df_books_clean["genre_text"] = (
    df_books_clean["genre_list"]
    .apply(
        lambda genres: ", ".join(genres)
    )
)

df_books_clean["model_text"] = (
    "Title: "
    + df_books_clean["title"].fillna("").str.strip()
    + ". Genre: "
    + df_books_clean["genre_text"].fillna("").str.strip()
    + ". Description: "
    + df_books_clean["desc"].fillna("").str.strip()
)

In [ ]:
# ============================================================
# PEMBERSIHAN TEKS RINGAN
# ============================================================
# Membersihkan HTML, URL, dan spasi berlebih tanpa menghilangkan
# struktur semantik kalimat yang diperlukan Sentence-BERT.

def clean_text_for_sbert(text):
    text = str(text)

    # Menghapus tag HTML.
    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    # Menghapus URL.
    text = re.sub(
        r"http\S+|www\.\S+",
        " ",
        text
    )

    # Menyamakan whitespace.
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df_books_clean["model_text"] = (
    df_books_clean["model_text"]
    .apply(clean_text_for_sbert)
)

In [ ]:
empty_model_text_count = (
    df_books_clean["model_text"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Jumlah model_text kosong:",
    empty_model_text_count
)

df_books_clean["model_text_length"] = (
    df_books_clean["model_text"]
    .str.len()
)

print(
    df_books_clean["model_text_length"]
    .describe()
    .round(2)
)

Jumlah model_text kosong: 0
count   4,083.00
mean    1,296.77
std       712.61
min        69.00
25%       816.50
50%     1,204.00
75%     1,643.00
max     9,056.00
Name: model_text_length, dtype: float64


## Melihat isi teks mentah yang tersembunyi

In [ ]:
# Menentukan judul yang ingin dicari
judul_cari = 'Learn SQL the Hard Way'

# Memfilter data
anomali = df_books_clean[df_books_clean['title'].str.contains(judul_cari, case=False, na=False)]

# Melakukan pengecekan apakah data ditemukan
if not anomali.empty:
    print(f"--- Data Ditemukan untuk '{judul_cari}' ---")
    print("Isi desc mentah :", repr(anomali['desc'].values[0]))
    print("Panjang karakter:", len(anomali['desc'].values[0]))
else:
    print(f"Informasi: Buku dengan judul yang mengandung '{judul_cari}' tidak ditemukan di dalam dataset.")

--- Data Ditemukan untuk 'Learn SQL the Hard Way' ---
Isi desc mentah : 'http://sql.learncodethehardway.org/book/'
Panjang karakter: 40


In [ ]:
# 1. Memfilter baris di mana deskripsi HANYA berisi URL
# Regex '^https?://\S+$' artinya:
# Dimulai dengan http atau https, diikuti karakter tanpa spasi hingga akhir.
kondisi_url = df_books_clean['desc'].astype(str).str.strip().str.match(r'^https?://\S+$')

df_anomali_url = df_books_clean[kondisi_url]

# 2. Memilih kolom yang relevan dan merapikan namanya
df_tampil = df_anomali_url[['title', 'genre_text', 'desc']].rename(columns={
    'title': 'Judul Buku',
    'genre_text': 'Genre',
    'desc': 'Isi Deskripsi Mentah'
})

# 3. Mencetak hasil ke dalam format tabel Markdown eksternal
print(f"### Ditemukan {len(df_tampil)} buku dengan deskripsi berupa URL\n")
print(df_tampil.to_markdown(index=False))

### Ditemukan 3 buku dengan deskripsi berupa URL

| Judul Buku                         | Genre                                                                                                                 | Isi Deskripsi Mentah                     |
|:-----------------------------------|:----------------------------------------------------------------------------------------------------------------------|:-----------------------------------------|
| Paperless: A MacSparky Field Guide | Nonfiction, Productivity, Science, Technology, Business, Self Help, Personal Development, Computer Science, Computers | http://itunes.apple.com/us/book/paper... |
| Learn SQL the Hard Way             | Computer Science, Programming                                                                                         | http://sql.learncodethehardway.org/book/ |
| Clean Ruby                         | Computer Science, Programming                                                                          

In [ ]:
# ============================================================
# PEMERIKSAAN KARAKTER RUSAK
# ============================================================

replacement_character_count = (
    df_books_clean["model_text"]
    .str.contains("�", regex=False, na=False)
    .sum()
)

print(
    "Jumlah model_text yang masih mengandung karakter �:",
    replacement_character_count
)

Jumlah model_text yang masih mengandung karakter �: 27


In [ ]:
df_books_clean["model_text"] = (
    df_books_clean["model_text"]
    .str.replace("�", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
# 1. Tentukan nama file output
nama_file_model_text = "output_model_text.md"

# 2. Buka file dalam mode write
with open(nama_file_model_text, "w", encoding="utf-8") as f:
    f.write("# Data Title, Genre, dan Model Text\n\n")

    # 3. Tulis data ke dalam format tabel Markdown
    # .head(5) dihapus untuk mengambil semua data
    # .fillna("") ditambahkan untuk mencegah error "boolean value of NA is ambiguous"
    f.write(
        df_books_clean[
            [
                "title",
                "genre",
                "model_text"
            ]
        ].fillna("").to_markdown(index=False)
    )

print(f"File markdown berhasil dibuat: {nama_file_model_text}")

File markdown berhasil dibuat: output_model_text.md


In [ ]:
# 1. Hitung panjang karakter dari model_text
df_books_clean["model_text_length"] = df_books_clean["model_text"].str.len()

# 2. Ambil otomatis 5 data dengan teks TERPENDEK agar rapi di laporan
df_singkat = df_books_clean.nsmallest(25, "model_text_length")

# 3. Pilih kolom pembentuk dan hasil akhirnya, lalu ubah nama kolom agar formal
df_tampil = df_singkat[["title", "genre_text", "model_text"]].rename(columns={
    "title": "Judul Buku Asli",
    "genre_text": "Genre Asli",
    "model_text": "Hasil Penggabungan (model_text)"
})

# 4. Cetak ke dalam format Markdown
print(df_tampil.to_markdown(index=False))

| Judul Buku Asli                                                                                 | Genre Asli                                                                                                            | Hasil Penggabungan (model_text)                                                                                                                                                                        |
|:------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Clean Ruby                                                                                      | Computer Science, Programming                             

In [ ]:
# 1. Hitung panjang karakter (sama seperti sebelumnya)
df_books_clean["model_text_length"] = df_books_clean["model_text"].str.len()

# 2. FILTER UTAMA: Pastikan kita hanya mengambil buku yang deskripsinya TIDAK kosong
# Kita gunakan pengecekan panjang karakter desc > 0
df_buku_lengkap = df_books_clean[df_books_clean["desc"].fillna("").str.strip() != ""]

# 3. Baru kita cari 5 yang terpendek dari data yang sudah dipastikan lengkap
df_singkat = df_buku_lengkap.nsmallest(5, "model_text_length")

# 4. Pilih kolom dan ubah nama agar rapi
df_tampil = df_singkat[["title", "genre_text", "model_text"]].rename(columns={
    "title": "Judul Buku Asli",
    "genre_text": "Genre Asli",
    "model_text": "Hasil Penggabungan (model_text)"
})

# 5. Cetak
print(df_tampil.to_markdown(index=False))

| Judul Buku Asli        | Genre Asli                                                        | Hasil Penggabungan (model_text)                                                                                                         |
|:-----------------------|:------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------|
| Clean Ruby             | Computer Science, Programming                                     | Title: Clean Ruby. Genre: Computer Science, Programming. Description:                                                                   |
| Learn SQL the Hard Way | Computer Science, Programming                                     | Title: Learn SQL the Hard Way. Genre: Computer Science, Programming. Description:                                                       |
| Case Histories 1       | Psychology, Psychoanalysis, Nonfiction   

In [ ]:
# 1. Proses pembuatan identitas buku (Kode Asli Anda)
df_books_clean["book_id"] = [
    f"BOOK_{index:05d}"
    for index in range(1, len(df_books_clean) + 1)
]

# 2. Menampilkan 5 data awal dari hasil pembentukan ID
# Kita bisa menyandingkannya dengan kolom judul agar lebih jelas
kolom_tampil = ["book_id", "title", "model_text"] # Tambahkan kolom lain jika perlu

print("### Hasil Pembentukan Identitas Buku (book_id)\n")
print(df_books_clean[kolom_tampil].head(5).to_markdown(index=False))

### Hasil Pembentukan Identitas Buku (book_id)

| book_id    | title                                                          | model_text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [ ]:
df_books_clean.to_csv(
    "books_prepared.csv",
    index=False,
    encoding="utf-8"
)

print(
    "Dataset preparation berhasil disimpan:",
    df_books_clean.shape
)

Dataset preparation berhasil disimpan: (4083, 25)


In [ ]:

# # Mengatur seed agar hasil deteksi konsisten setiap kali kode dijalankan
# DetectorFactory.seed = 0

# # Fungsi untuk mendeteksi bahasa dengan penanganan error
# def deteksi_bahasa(teks):
#     try:
#         # Jika teks kosong atau hanya berisi spasi
#         if not teks or pd.isna(teks) or str(teks).strip() == "":
#             return "unknown"
#         # Deteksi bahasa
#         return detect(str(teks))
#     except:
#         # Menangkap error jika judul hanya berisi angka atau simbol
#         return "unknown"

# # 1. Buat kolom baru bernama 'language' untuk menyimpan kode bahasanya
# df_books_clean["language"] = df_books_clean["title"].apply(deteksi_bahasa)

# # 2. Menampilkan ada bahasa apa saja beserta jumlah bukunya
# print("Distribusi Bahasa pada Judul Buku:")
# distribusi_bahasa = df_books_clean["language"].value_counts().reset_index()
# distribusi_bahasa.columns = ["language", "count"]
# print(distribusi_bahasa.to_string(index=False))
# print("-" * 30)

# # 3. Filter data untuk mengambil buku yang BUKAN berbahasa Inggris ('en')
# # 'en' adalah kode standar langdetect untuk bahasa Inggris
# buku_non_inggris = df_books_clean[
#     (df_books_clean["language"] != "en") &
#     (df_books_clean["language"] != "unknown")
# ]

# # 4. Tentukan nama file output
# nama_file_non_inggris = "output_non_english_books.md"

# # 5. Simpan ke file Markdown
# with open(nama_file_non_inggris, "w", encoding="utf-8") as f:
#     f.write("# Data Buku Non-Bahasa Inggris\n\n")

#     # Menulis ringkasan distribusi bahasa di bagian atas file Markdown
#     f.write("### Distribusi Semua Bahasa:\n")
#     f.write(distribusi_bahasa.to_markdown(index=False) + "\n\n")

#     f.write("### Daftar Buku Non-Inggris:\n")
#     # Pastikan .fillna("") tetap digunakan untuk mencegah error tabulate
#     f.write(
#         buku_non_inggris[
#             ["title", "language"] # Tambahkan kolom lain di sini jika diperlukan
#         ].fillna("").to_markdown(index=False)
#     )

# print(f"\nFile markdown berhasil dibuat: {nama_file_non_inggris}")

# Modelling

In [ ]:
!pip install -q -U sentence-transformers

In [ ]:
# ============================================================
# IMPORT MODELLING LIBRARIES
# ============================================================

import gc
import time
import torch
import numpy as np
import pandas as pd

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
    util
)

In [ ]:
# ============================================================
# DEVICE CONFIGURATION
# ============================================================
# Menggunakan GPU jika tersedia.
# Jika GPU tidak tersedia, program otomatis menggunakan CPU.

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Perangkat yang digunakan:", device)

if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))

Perangkat yang digunakan: cuda
Nama GPU: Tesla T4


In [ ]:
df_books_model = pd.read_csv(
    "books_prepared.csv",
    low_memory=False
)

# ============================================================
# MODELLING DATA VALIDATION
# ============================================================

required_columns = [
    "book_id",
    "title",
    "genre_text",
    "desc",
    "model_text"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df_books_model.columns
]

if missing_columns:
    raise ValueError(
        f"Kolom modelling tidak ditemukan: {missing_columns}"
    )

if df_books_model["model_text"].isna().any():
    raise ValueError(
        "Masih terdapat model_text yang kosong."
    )

if df_books_model["book_id"].duplicated().any():
    raise ValueError(
        "Masih terdapat book_id yang duplikat."
    )

print("Jumlah buku untuk modelling:", len(df_books_model))
print("Seluruh data modelling valid.")

Jumlah buku untuk modelling: 4083
Seluruh data modelling valid.


In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

embedding_model_names = {
    "multilingual_minilm": (
        "sentence-transformers/"
        "paraphrase-multilingual-MiniLM-L12-v2"
    ),

    "multilingual_mpnet": (
        "sentence-transformers/"
        "paraphrase-multilingual-mpnet-base-v2"
    )
}

reranker_model_name = (
    "cross-encoder/"
    "mmarco-mMiniLMv2-L12-H384-v1"
)

top_k_candidates = 50
top_n_recommendations = 10

print("Embedding models:")
for model_key, model_name in embedding_model_names.items():
    print(f"- {model_key}: {model_name}")

print("\nReranker:")
print(reranker_model_name)

Embedding models:
- multilingual_minilm: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- multilingual_mpnet: sentence-transformers/paraphrase-multilingual-mpnet-base-v2

Reranker:
cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


In [ ]:
# ============================================================
# TOKEN LENGTH ANALYSIS FUNCTION
# ============================================================
# Menghitung panjang token seluruh model_text berdasarkan
# tokenizer masing-masing embedding model.

def analyze_token_length(
    dataframe,
    model_name,
    text_column="model_text"
):
    model = SentenceTransformer(
        model_name,
        device=device
    )

    tokenizer = model.tokenizer
    max_seq_length = model.max_seq_length

    token_counts = dataframe[text_column].apply(
        lambda text: len(
            tokenizer.encode(
                str(text),
                add_special_tokens=True,
                truncation=False
            )
        )
    )

    over_limit_count = (
        token_counts > max_seq_length
    ).sum()

    over_limit_percentage = (
        over_limit_count
        / len(dataframe)
        * 100
    )

    result = {
        "model_name": model_name,
        "max_seq_length": max_seq_length,
        "mean_tokens": token_counts.mean(),
        "median_tokens": token_counts.median(),
        "max_tokens": token_counts.max(),
        "texts_over_limit": over_limit_count,
        "percentage_over_limit": over_limit_percentage
    }

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return token_counts, result

In [ ]:
# ============================================================
# TOKEN ANALYSIS: MULTILINGUAL MINILM
# ============================================================

minilm_token_counts, minilm_token_summary = (
    analyze_token_length(
        dataframe=df_books_model,
        model_name=embedding_model_names[
            "multilingual_minilm"
        ]
    )
)

print("Analisis token MiniLM:")
for key, value in minilm_token_summary.items():
    print(f"{key}: {value}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (636 > 128). Running this sequence through the model will result in indexing errors


Analisis token MiniLM:
model_name: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
max_seq_length: 128
mean_tokens: 309.7114866519716
median_tokens: 288.0
max_tokens: 2295
texts_over_limit: 3605
percentage_over_limit: 88.29292187117316


In [ ]:
# ============================================================
# TOKEN ANALYSIS: MULTILINGUAL MPNET
# ============================================================

mpnet_token_counts, mpnet_token_summary = (
    analyze_token_length(
        dataframe=df_books_model,
        model_name=embedding_model_names[
            "multilingual_mpnet"
        ]
    )
)

print("Analisis token MPNet:")
for key, value in mpnet_token_summary.items():
    print(f"{key}: {value}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (636 > 128). Running this sequence through the model will result in indexing errors


Analisis token MPNet:
model_name: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
max_seq_length: 128
mean_tokens: 309.7114866519716
median_tokens: 288.0
max_tokens: 2295
texts_over_limit: 3605
percentage_over_limit: 88.29292187117316


In [ ]:
token_analysis_table = pd.DataFrame([
    minilm_token_summary,
    mpnet_token_summary
])

print(
    token_analysis_table.to_string(
        index=False
    )
)

                                                 model_name  max_seq_length  mean_tokens  median_tokens  max_tokens  texts_over_limit  percentage_over_limit
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2             128       309.71         288.00        2295              3605                  88.29
sentence-transformers/paraphrase-multilingual-mpnet-base-v2             128       309.71         288.00        2295              3605                  88.29


In [ ]:
# ============================================================
# CORPUS EMBEDDING FUNCTION
# ============================================================
# Mengubah seluruh model_text menjadi embedding.
#
# normalize_embeddings=True membuat setiap embedding memiliki
# panjang vektor 1, sehingga cosine similarity dapat dihitung
# secara efisien.

def create_corpus_embeddings(
    dataframe,
    model_name,
    batch_size=32
):
    model = SentenceTransformer(
        model_name,
        device=device
    )

    corpus_texts = (
        dataframe["model_text"]
        .astype(str)
        .tolist()
    )

    start_time = time.perf_counter()

    embeddings = model.encode(
        corpus_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    elapsed_time = (
        time.perf_counter()
        - start_time
    )

    return model, embeddings, elapsed_time

In [ ]:
# ============================================================
# CORPUS EMBEDDING: MULTILINGUAL MINILM
# ============================================================

minilm_model, minilm_embeddings, minilm_encoding_time = (
    create_corpus_embeddings(
        dataframe=df_books_model,
        model_name=embedding_model_names[
            "multilingual_minilm"
        ],
        batch_size=32
    )
)

print("Shape embedding MiniLM:", minilm_embeddings.shape)
print(
    "Waktu encoding MiniLM:",
    round(minilm_encoding_time, 2),
    "detik"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/128 [00:00<?, ?it/s]

Shape embedding MiniLM: (4083, 384)
Waktu encoding MiniLM: 11.35 detik


In [ ]:
np.save(
    "book_embeddings_multilingual_minilm.npy",
    minilm_embeddings
)

print("Embedding MiniLM berhasil disimpan.")

Embedding MiniLM berhasil disimpan.


In [ ]:
del minilm_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memori model MiniLM sudah dibersihkan.")

Memori model MiniLM sudah dibersihkan.


In [ ]:
# ============================================================
# CORPUS EMBEDDING: MULTILINGUAL MPNET
# ============================================================

mpnet_model, mpnet_embeddings, mpnet_encoding_time = (
    create_corpus_embeddings(
        dataframe=df_books_model,
        model_name=embedding_model_names[
            "multilingual_mpnet"
        ],
        batch_size=16
    )
)

print("Shape embedding MPNet:", mpnet_embeddings.shape)
print(
    "Waktu encoding MPNet:",
    round(mpnet_encoding_time, 2),
    "detik"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/256 [00:00<?, ?it/s]

Shape embedding MPNet: (4083, 768)
Waktu encoding MPNet: 30.35 detik


In [ ]:
np.save(
    "book_embeddings_multilingual_mpnet.npy",
    mpnet_embeddings
)

print("Embedding MPNet berhasil disimpan.")

Embedding MPNet berhasil disimpan.


In [ ]:
# ============================================================
# EMBEDDING TECHNICAL COMPARISON
# ============================================================

embedding_comparison = pd.DataFrame({
    "model": [
        "Multilingual MiniLM",
        "Multilingual MPNet"
    ],

    "number_of_books": [
        minilm_embeddings.shape[0],
        mpnet_embeddings.shape[0]
    ],

    "embedding_dimension": [
        minilm_embeddings.shape[1],
        mpnet_embeddings.shape[1]
    ],

    "encoding_time_seconds": [
        round(minilm_encoding_time, 2),
        round(mpnet_encoding_time, 2)
    ],

    "embedding_size_mb": [
        round(
            minilm_embeddings.nbytes
            / (1024 ** 2),
            2
        ),

        round(
            mpnet_embeddings.nbytes
            / (1024 ** 2),
            2
        )
    ]
})

print(
    embedding_comparison.to_string(
        index=False
    )
)

              model  number_of_books  embedding_dimension  encoding_time_seconds  embedding_size_mb
Multilingual MiniLM             4083                  384                  11.35               5.98
 Multilingual MPNet             4083                  768                  30.35              11.96


In [ ]:
embedding_comparison.to_csv(
    "embedding_model_comparison.csv",
    index=False
)

In [ ]:
del mpnet_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# LOAD MULTILINGUAL CROSS-ENCODER
# ============================================================

reranker_model = CrossEncoder(
    reranker_model_name,
    device=device
)

print("Cross-Encoder berhasil dimuat.")
print("Model:", reranker_model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Cross-Encoder berhasil dimuat.
Model: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


In [ ]:
# ============================================================
# QUERY CLEANING FUNCTION
# ============================================================

def clean_user_query(query):
    query = str(query)

    query = re.sub(
        r"<[^>]+>",
        " ",
        query
    )

    query = re.sub(
        r"http\S+|www\.\S+",
        " ",
        query
    )

    query = re.sub(
        r"\s+",
        " ",
        query
    )

    return query.strip()

In [ ]:
# ============================================================
# RETRIEVAL FUNCTION
# ============================================================
# Tahap pertama:
# 1. encode query menggunakan model embedding;
# 2. hitung cosine similarity query dengan seluruh buku;
# 3. ambil Top-K kandidat.

def retrieve_candidates(
    query,
    embedding_model,
    corpus_embeddings,
    dataframe,
    top_k=50
):
    clean_query = clean_user_query(query)

    if not clean_query:
        raise ValueError(
            "Query pengguna tidak boleh kosong."
        )

    query_embedding = embedding_model.encode(
        clean_query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.reshape(
        1,
        -1
    )

    cosine_scores = util.cos_sim(
        query_embedding,
        corpus_embeddings
    )[0].cpu().numpy()

    top_k = min(
        top_k,
        len(dataframe)
    )

    top_indices = np.argpartition(
        -cosine_scores,
        top_k - 1
    )[:top_k]

    top_indices = top_indices[
        np.argsort(
            -cosine_scores[top_indices]
        )
    ]

    candidates = dataframe.iloc[
        top_indices
    ].copy()

    candidates["corpus_index"] = top_indices

    candidates["retrieval_score"] = (
        cosine_scores[top_indices]
    )

    candidates["retrieval_rank"] = (
        np.arange(
            1,
            len(candidates) + 1
        )
    )

    return clean_query, candidates

In [ ]:
# ============================================================
# RERANKING FUNCTION
# ============================================================
# Cross-Encoder membaca query dan model_text kandidat secara
# bersamaan, lalu mengurutkan ulang kandidat berdasarkan skor
# relevansi yang lebih detail.

def rerank_candidates(
    query,
    candidates,
    reranker,
    top_n=10,
    batch_size=16
):
    query_document_pairs = [
        (
            query,
            document_text
        )
        for document_text
        in candidates["model_text"].astype(str)
    ]

    reranker_scores = reranker.predict(
        query_document_pairs,
        batch_size=batch_size,
        show_progress_bar=False
    )

    reranked_candidates = candidates.copy()

    reranked_candidates["reranker_score"] = (
        np.asarray(reranker_scores)
        .reshape(-1)
    )

    reranked_candidates = (
        reranked_candidates
        .sort_values(
            "reranker_score",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    reranked_candidates["final_rank"] = (
        np.arange(
            1,
            len(reranked_candidates) + 1
        )
    )

    return reranked_candidates

In [ ]:
# ============================================================
# COMPLETE RETRIEVE AND RERANK PIPELINE
# ============================================================

def recommend_books(
    query,
    embedding_model,
    corpus_embeddings,
    dataframe,
    reranker,
    top_k=50,
    top_n=10
):
    retrieval_start = time.perf_counter()

    clean_query, candidates = retrieve_candidates(
        query=query,
        embedding_model=embedding_model,
        corpus_embeddings=corpus_embeddings,
        dataframe=dataframe,
        top_k=top_k
    )

    retrieval_time = (
        time.perf_counter()
        - retrieval_start
    )

    reranking_start = time.perf_counter()

    final_results = rerank_candidates(
        query=clean_query,
        candidates=candidates,
        reranker=reranker,
        top_n=top_n
    )

    reranking_time = (
        time.perf_counter()
        - reranking_start
    )

    total_time = (
        retrieval_time
        + reranking_time
    )

    selected_columns = [
        "final_rank",
        "book_id",
        "title",
        "author",
        "genre_text",
        "target_categories",
        "rating",
        "totalratings",
        "retrieval_rank",
        "retrieval_score",
        "reranker_score"
    ]

    available_columns = [
        column
        for column in selected_columns
        if column in final_results.columns
    ]

    output = final_results[
        available_columns
    ].copy()

    timing = {
        "retrieval_time_seconds": retrieval_time,
        "reranking_time_seconds": reranking_time,
        "total_time_seconds": total_time
    }

    return output, candidates, timing

In [ ]:
minilm_model = SentenceTransformer(
    embedding_model_names[
        "multilingual_minilm"
    ],
    device=device
)

test_query = (
    "I am preparing for my first leadership role and want to learn "
    "how to manage a small team, communicate effectively, and "
    "handle conflict."
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
minilm_results, minilm_candidates, minilm_timing = (
    recommend_books(
        query=test_query,
        embedding_model=minilm_model,
        corpus_embeddings=minilm_embeddings,
        dataframe=df_books_model,
        reranker=reranker_model,
        top_k=top_k_candidates,
        top_n=top_n_recommendations
    )
)

# 1. Tentukan nama file output
nama_file_minilm = "output_minilm_recommendations.md"

# 2. Buka file dalam mode write
with open(nama_file_minilm, "w", encoding="utf-8") as f:

    # 3. Menulis judul dan tabel hasil rekomendasi
    f.write("# Hasil Rekomendasi Model MiniLM\n\n")
    f.write(
        minilm_results.fillna("").to_markdown(index=False)
    )

    # 4. Menulis judul untuk bagian waktu proses
    f.write("\n\n### Rincian Waktu Proses:\n")

    # 5. Melakukan iterasi (looping) pada dictionary minilm_timing
    # dan menulisnya sebagai bullet list di Markdown
    for key, value in minilm_timing.items():
        f.write(f"* **{key}**: {value:.4f} detik\n")

print(f"File markdown berhasil dibuat: {nama_file_minilm}")

File markdown berhasil dibuat: output_minilm_recommendations.md


In [ ]:
del minilm_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
mpnet_model = SentenceTransformer(
    embedding_model_names[
        "multilingual_mpnet"
    ],
    device=device
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
mpnet_results, mpnet_candidates, mpnet_timing = (
    recommend_books(
        query=test_query,
        embedding_model=mpnet_model,
        corpus_embeddings=mpnet_embeddings,
        dataframe=df_books_model,
        reranker=reranker_model,
        top_k=top_k_candidates,
        top_n=top_n_recommendations
    )
)

# 1. Tentukan nama file output
nama_file_mpnet = "output_mpnet_recommendations.md"

# 2. Buka file dalam mode write
with open(nama_file_mpnet, "w", encoding="utf-8") as f:

    # 3. Menulis judul dan tabel hasil rekomendasi
    f.write("# Hasil Rekomendasi Model MPNet\n\n")

    # Pastikan menggunakan .fillna("") untuk mencegah error jika ada data kosong
    f.write(
        mpnet_results.fillna("").to_markdown(index=False)
    )

    # 4. Menulis judul untuk bagian waktu proses
    f.write("\n\n### Rincian Waktu Proses:\n")

    # 5. Melakukan iterasi (looping) pada dictionary mpnet_timing
    # dan menulisnya sebagai bullet list di Markdown
    for key, value in mpnet_timing.items():
        f.write(f"* **{key}**: {value:.4f} detik\n")

print(f"File markdown berhasil dibuat: {nama_file_mpnet}")

File markdown berhasil dibuat: output_mpnet_recommendations.md


In [ ]:
# ============================================================
# QUERY-LEVEL MODEL COMPARISON
# ============================================================

query_model_comparison = pd.DataFrame({
    "model": [
        "Multilingual MiniLM",
        "Multilingual MPNet"
    ],

    "retrieval_time_seconds": [
        minilm_timing[
            "retrieval_time_seconds"
        ],
        mpnet_timing[
            "retrieval_time_seconds"
        ]
    ],

    "reranking_time_seconds": [
        minilm_timing[
            "reranking_time_seconds"
        ],
        mpnet_timing[
            "reranking_time_seconds"
        ]
    ],

    "total_time_seconds": [
        minilm_timing[
            "total_time_seconds"
        ],
        mpnet_timing[
            "total_time_seconds"
        ]
    ]
})

print(
    query_model_comparison
    .round(4)
    .to_string(index=False)
)

              model  retrieval_time_seconds  reranking_time_seconds  total_time_seconds
Multilingual MiniLM                    0.02                    0.47                0.49
 Multilingual MPNet                    0.02                    0.45                0.48


In [ ]:
minilm_top10_ids = set(
    minilm_results["book_id"]
)

mpnet_top10_ids = set(
    mpnet_results["book_id"]
)

common_books = (
    minilm_top10_ids
    .intersection(mpnet_top10_ids)
)

print(
    "Jumlah buku yang sama pada Top-10:",
    len(common_books)
)

print(
    "Persentase overlap Top-10:",
    round(
        len(common_books)
        / top_n_recommendations
        * 100,
        2
    ),
    "%"
)

Jumlah buku yang sama pada Top-10: 9
Persentase overlap Top-10: 90.0 %


In [ ]:
minilm_results.to_csv(
    "recommendation_minilm_test.csv",
    index=False,
    encoding="utf-8"
)

mpnet_results.to_csv(
    "recommendation_mpnet_test.csv",
    index=False,
    encoding="utf-8"
)

query_model_comparison.to_csv(
    "query_model_timing_comparison.csv",
    index=False
)

print("Seluruh hasil pengujian berhasil disimpan.")

Seluruh hasil pengujian berhasil disimpan.


# Evaluation

In [ ]:
# ============================================================
# STRATIFIED MULTILINGUAL EVALUATION QUERIES
# ============================================================
# Setiap information need mempunyai dua formulasi:
# 1. short  : query ringkas;
# 2. long   : query lebih deskriptif.
#
# Fungsi ini mengubah satu kebutuhan informasi menjadi
# dua record query evaluasi.

def create_query_pair(
    category,
    information_need_id,
    information_need,
    query_group,
    query_language_code,
    query_language,
    target_language_code,
    target_language,
    short_query,
    long_query,
    overlap_category=None
):
    return [
        {
            "category": category,
            "information_need_id": information_need_id,
            "information_need": information_need,
            "formulation_id": "F1",
            "query_length": "short",
            "query_group": query_group,
            "query_language_code": query_language_code,
            "query_language": query_language,
            "target_language_code": target_language_code,
            "target_language": target_language,
            "is_cross_lingual": (
                query_language_code != target_language_code
            ),
            "is_cross_category": (
                overlap_category is not None
            ),
            "overlap_category": overlap_category,
            "query": short_query
        },
        {
            "category": category,
            "information_need_id": information_need_id,
            "information_need": information_need,
            "formulation_id": "F2",
            "query_length": "long",
            "query_group": query_group,
            "query_language_code": query_language_code,
            "query_language": query_language,
            "target_language_code": target_language_code,
            "target_language": target_language,
            "is_cross_lingual": (
                query_language_code != target_language_code
            ),
            "is_cross_category": (
                overlap_category is not None
            ),
            "overlap_category": overlap_category,
            "query": long_query
        }
    ]

# ============================================================
# QUERY SPECIFICATIONS
# ============================================================
# Terdapat 10 information needs pada setiap kategori.
# Setiap kebutuhan menghasilkan dua formulasi query.

evaluation_query_pairs = []

In [ ]:
# ============================================================
# SELF DEVELOPMENT: 10 NEEDS × 2 FORMULATIONS
# ============================================================

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD01",
    information_need="Meningkatkan kepercayaan diri",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about building self-confidence and overcoming "
        "self-doubt."
    ),
    long_query=(
        "I am looking for an English book that can help me build "
        "self-confidence, stop doubting my abilities, and become "
        "more comfortable taking action."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD02",
    information_need="Membangun growth mindset",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Growth mindset and personal improvement books."
    ),
    long_query=(
        "Recommend an English book about developing a growth "
        "mindset, learning from mistakes, and continuously "
        "improving as a person."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD03",
    information_need="Memahami diri dan tujuan hidup",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about self-awareness and finding life purpose."
    ),
    long_query=(
        "I want an English book that helps me understand my "
        "strengths, values, identity, and long-term purpose in life."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD04",
    information_need="Mengatasi ketakutan akan kegagalan",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku untuk mengatasi rasa takut gagal."
    ),
    long_query=(
        "Saya mencari buku berbahasa Inggris yang membantu "
        "mengurangi ketakutan akan kegagalan, membangun keberanian, "
        "dan mulai mengambil tindakan."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD05",
    information_need="Membangun kebiasaan positif",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang kebiasaan positif dan transformasi diri."
    ),
    long_query=(
        "Saya ingin menemukan buku berbahasa Inggris yang "
        "menjelaskan cara membentuk kebiasaan positif, menghentikan "
        "pola buruk, dan melakukan perubahan diri secara konsisten."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD06",
    information_need="Membangun ketahanan pribadi",
    query_group="Secondary Language to English",
    query_language_code="de",
    query_language="German",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Bücher über persönliche Widerstandsfähigkeit."
    ),
    long_query=(
        "Ich suche ein englisches Buch darüber, wie man nach "
        "Rückschlägen wieder aufsteht, innere Stärke entwickelt "
        "und mit schwierigen Lebenssituationen umgeht."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD07",
    information_need="Penerimaan diri dan keberanian emosional",
    query_group="Secondary Language to English",
    query_language_code="ar",
    query_language="Arabic",
    target_language_code="en",
    target_language="English",
    short_query=(
        "كتب عن تقبل الذات والثقة بالنفس."
    ),
    long_query=(
        "أبحث عن كتاب باللغة الإنجليزية يساعدني على تقبل نفسي، "
        "والتغلب على الشعور بالنقص، وبناء الشجاعة والثقة الداخلية."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD08",
    information_need="Pengembangan diri dalam buku Prancis",
    query_group="English to Non-English",
    query_language_code="en",
    query_language="English",
    target_language_code="fr",
    target_language="French",
    short_query=(
        "French books about personal growth."
    ),
    long_query=(
        "Find a French-language book about improving self-esteem, "
        "developing personal potential, and becoming a more "
        "confident person."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD09",
    information_need="Zelfkennis dan persoonlijke groei",
    query_group="Non-English Monolingual",
    query_language_code="nl",
    query_language="Dutch",
    target_language_code="nl",
    target_language="Dutch",
    short_query=(
        "Boeken over zelfkennis en persoonlijke groei."
    ),
    long_query=(
        "Ik zoek een Nederlandstalig boek dat mij helpt mezelf "
        "beter te begrijpen, mijn sterke punten te herkennen en "
        "persoonlijk te groeien."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Self Development",
    information_need_id="SD10",
    information_need=(
        "Self-compassion dan pengelolaan emosi"
    ),
    query_group="Cross-Category",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Self-compassion and emotional regulation books."
    ),
    long_query=(
        "I need a book that combines personal development with "
        "psychology by teaching self-compassion, emotional "
        "regulation, and healthier ways of responding to failure."
    ),
    overlap_category="Psychology"
)

In [ ]:
# ============================================================
# TECHNOLOGY: 10 NEEDS × 2 FORMULATIONS
# ============================================================

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE01",
    information_need="Dasar machine learning",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Beginner books about machine learning."
    ),
    long_query=(
        "I am looking for an English introductory book that "
        "explains machine learning and artificial intelligence "
        "without requiring advanced prior knowledge."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE02",
    information_need="Pemrograman Python",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Python programming books for beginners."
    ),
    long_query=(
        "Recommend an English book for a beginner who wants to "
        "learn Python programming, write practical programs, and "
        "understand basic software development."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE03",
    information_need="Cybersecurity dan keamanan jaringan",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about cybersecurity and network security."
    ),
    long_query=(
        "I need an English book that introduces cybersecurity, "
        "computer network protection, common attacks, and methods "
        "for securing digital systems."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE04",
    information_need="Database dan pengolahan data",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang database dan pengolahan data."
    ),
    long_query=(
        "Saya mencari buku berbahasa Inggris yang menjelaskan "
        "perancangan database, pengelolaan data, SQL, dan sistem "
        "informasi untuk pemula."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE05",
    information_need="Software architecture dan clean code",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang clean code dan arsitektur perangkat lunak."
    ),
    long_query=(
        "Saya ingin menemukan buku berbahasa Inggris mengenai "
        "software engineering, clean code, pola desain, dan "
        "arsitektur sistem yang mudah dipelihara."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE06",
    information_need="Cloud dan distributed systems",
    query_group="Secondary Language to English",
    query_language_code="nl",
    query_language="Dutch",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Boeken over cloud computing en gedistribueerde systemen."
    ),
    long_query=(
        "Ik zoek een Engelstalig boek dat cloud computing, "
        "gedistribueerde systemen, schaalbaarheid en betrouwbare "
        "software-infrastructuur uitlegt."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE07",
    information_need="UX dan human-computer interaction",
    query_group="Secondary Language to English",
    query_language_code="fr",
    query_language="French",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Livres sur l'expérience utilisateur et l'ergonomie."
    ),
    long_query=(
        "Je cherche un livre en anglais sur l'expérience "
        "utilisateur, l'interaction homme-machine et la conception "
        "d'interfaces numériques faciles à utiliser."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE08",
    information_need="Etika kecerdasan buatan dalam bahasa Jerman",
    query_group="English to Non-English",
    query_language_code="en",
    query_language="English",
    target_language_code="de",
    target_language="German",
    short_query=(
        "German books about artificial intelligence ethics."
    ),
    long_query=(
        "Find a German-language book that discusses artificial "
        "intelligence, algorithmic bias, privacy, and the ethical "
        "impact of automated technology."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE09",
    information_need="الأمن السيبراني",
    query_group="Non-English Monolingual",
    query_language_code="ar",
    query_language="Arabic",
    target_language_code="ar",
    target_language="Arabic",
    short_query=(
        "كتب عربية عن الأمن السيبراني."
    ),
    long_query=(
        "أبحث عن كتاب باللغة العربية يشرح أساسيات الأمن "
        "السيبراني، وحماية الشبكات، والوقاية من الهجمات الرقمية."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Technology",
    information_need_id="TE10",
    information_need="Membangun karier di bidang teknologi",
    query_group="Cross-Category",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku untuk membangun karier di bidang teknologi."
    ),
    long_query=(
        "Saya mencari buku yang menggabungkan pembelajaran "
        "teknologi dengan pengembangan karier, termasuk keterampilan "
        "teknis, portofolio, dan persiapan bekerja di industri IT."
    ),
    overlap_category="Career Development"
)

In [ ]:
# ============================================================
# CAREER DEVELOPMENT: 10 NEEDS × 2 FORMULATIONS
# ============================================================

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD01",
    information_need="Persiapan wawancara kerja",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about job interview preparation."
    ),
    long_query=(
        "I need an English book that teaches how to prepare for "
        "job interviews, answer difficult questions, and make a "
        "strong first impression."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD02",
    information_need="Perencanaan karier",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Career planning and meaningful work books."
    ),
    long_query=(
        "Recommend an English book about identifying career goals, "
        "choosing meaningful work, and planning long-term "
        "professional development."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD03",
    information_need="Kepemimpinan dan pengelolaan tim",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Leadership and team management books."
    ),
    long_query=(
        "I am looking for an English book about becoming a better "
        "leader, motivating employees, managing conflict, and "
        "building an effective team."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD04",
    information_need="Komunikasi profesional",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang komunikasi profesional di tempat kerja."
    ),
    long_query=(
        "Saya mencari buku berbahasa Inggris untuk meningkatkan "
        "komunikasi profesional, kemampuan presentasi, dan hubungan "
        "dengan rekan kerja."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD05",
    information_need="Transisi karier",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku untuk melakukan transisi karier."
    ),
    long_query=(
        "Saya ingin menemukan buku berbahasa Inggris yang membantu "
        "menilai keterampilan saat ini, berpindah profesi, dan "
        "mempersiapkan karier baru."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD06",
    information_need="Negosiasi gaji dan promosi",
    query_group="Secondary Language to English",
    query_language_code="de",
    query_language="German",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Bücher über Gehaltsverhandlungen und Beförderungen."
    ),
    long_query=(
        "Ich suche ein englisches Buch darüber, wie man das Gehalt "
        "verhandelt, berufliche Leistungen sichtbar macht und sich "
        "auf eine Beförderung vorbereitet."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD07",
    information_need="Networking dan personal branding",
    query_group="Secondary Language to English",
    query_language_code="fr",
    query_language="French",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Livres sur le réseautage et la marque personnelle."
    ),
    long_query=(
        "Je cherche un livre en anglais qui explique comment créer "
        "un réseau professionnel, développer une marque personnelle "
        "et trouver de nouvelles opportunités de carrière."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD08",
    information_need="Kepemimpinan kerja dalam bahasa Belanda",
    query_group="English to Non-English",
    query_language_code="en",
    query_language="English",
    target_language_code="nl",
    target_language="Dutch",
    short_query=(
        "Dutch books about workplace leadership."
    ),
    long_query=(
        "Find a Dutch-language book about leading employees, "
        "managing workplace relationships, and developing as a "
        "professional manager."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD09",
    information_need="Perencanaan karier bahasa Indonesia",
    query_group="Non-English Monolingual",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="id",
    target_language="Indonesian",
    short_query=(
        "Buku Indonesia tentang perencanaan karier."
    ),
    long_query=(
        "Saya mencari buku berbahasa Indonesia tentang menentukan "
        "arah karier, mengenali kemampuan profesional, dan memilih "
        "pekerjaan yang sesuai."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Career Development",
    information_need_id="CD10",
    information_need="Karier dan produktivitas kerja",
    query_group="Cross-Category",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Career growth and workplace productivity books."
    ),
    long_query=(
        "I need a book that combines career development with "
        "productivity by teaching professional goal setting, "
        "priority management, and consistent workplace performance."
    ),
    overlap_category="Productivity"
)

In [ ]:
# ============================================================
# PRODUCTIVITY: 10 NEEDS × 2 FORMULATIONS
# ============================================================

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR01",
    information_need="Mengatasi prokrastinasi",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about overcoming procrastination."
    ),
    long_query=(
        "I am looking for an English book that helps me stop "
        "procrastinating, begin important tasks, and complete work "
        "without waiting until the deadline."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR02",
    information_need="Manajemen waktu dan prioritas",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Time management and priority setting books."
    ),
    long_query=(
        "Recommend an English book about organizing time, choosing "
        "the most important tasks, and completing priorities "
        "consistently."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR03",
    information_need="Fokus dan deep work",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about deep work and concentration."
    ),
    long_query=(
        "I need an English book about reducing distractions, "
        "improving concentration, and performing focused work for "
        "long periods."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR04",
    information_need="Membangun sistem kerja efisien",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang sistem kerja yang efisien."
    ),
    long_query=(
        "Saya mencari buku berbahasa Inggris yang menjelaskan cara "
        "membangun alur kerja, mengatur tugas, dan menyelesaikan "
        "pekerjaan dengan lebih efisien."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR05",
    information_need="Mengelola beban pekerjaan",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku untuk mengelola banyak pekerjaan."
    ),
    long_query=(
        "Saya ingin menemukan buku berbahasa Inggris untuk mengatur "
        "banyak tugas, menghindari kewalahan, dan menjaga kemajuan "
        "pekerjaan secara teratur."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR06",
    information_need="Kebiasaan produktif",
    query_group="Secondary Language to English",
    query_language_code="ar",
    query_language="Arabic",
    target_language_code="en",
    target_language="English",
    short_query=(
        "كتب عن العادات المنتجة."
    ),
    long_query=(
        "أبحث عن كتاب باللغة الإنجليزية يشرح كيفية بناء عادات "
        "منتجة، وتنظيم اليوم، والاستمرار في إنجاز المهام المهمة."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR07",
    information_need="Efisiensi rapat dan email",
    query_group="Secondary Language to English",
    query_language_code="nl",
    query_language="Dutch",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Boeken over efficiënte vergaderingen en e-mailbeheer."
    ),
    long_query=(
        "Ik zoek een Engelstalig boek over het verminderen van "
        "onnodige vergaderingen, het beheren van e-mail en het "
        "efficiënter organiseren van kantoorwerk."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR08",
    information_need="Produktivitet dalam bahasa Denmark",
    query_group="English to Non-English",
    query_language_code="en",
    query_language="English",
    target_language_code="da",
    target_language="Danish",
    short_query=(
        "Danish books about productivity."
    ),
    long_query=(
        "Find a Danish-language book about time management, "
        "reducing distractions, and completing important work "
        "more efficiently."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR09",
    information_need="Gestão do tempo",
    query_group="Non-English Monolingual",
    query_language_code="pt",
    query_language="Portuguese",
    target_language_code="pt",
    target_language="Portuguese",
    short_query=(
        "Livros sobre gestão do tempo."
    ),
    long_query=(
        "Procuro um livro em português sobre como organizar tarefas, "
        "definir prioridades e usar o tempo de forma mais produtiva."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Productivity",
    information_need_id="PR10",
    information_need="Produktivitas dan kesehatan psikologis",
    query_group="Cross-Category",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang fokus, stres, dan produktivitas."
    ),
    long_query=(
        "Saya mencari buku yang menggabungkan produktivitas dan "
        "psikologi untuk meningkatkan fokus, mengelola stres, dan "
        "bekerja efektif tanpa mengalami kelelahan mental."
    ),
    overlap_category="Psychology"
)

In [ ]:
# ============================================================
# PSYCHOLOGY: 10 NEEDS × 2 FORMULATIONS
# ============================================================

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS01",
    information_need="Pikiran dan perilaku manusia",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about the human mind and behavior."
    ),
    long_query=(
        "I am looking for an English book that explains how human "
        "thoughts, emotions, social influences, and behavior are "
        "formed."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS02",
    information_need="Kecemasan dan depresi",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Books about anxiety and depression."
    ),
    long_query=(
        "Recommend an English psychology book that explains "
        "anxiety, depression, their causes, and evidence-based "
        "approaches to maintaining mental health."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS03",
    information_need="Memori dan pengambilan keputusan",
    query_group="English to English",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Cognitive psychology and memory books."
    ),
    long_query=(
        "I need an English book about cognitive psychology, memory, "
        "attention, reasoning, and how people make decisions."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS04",
    information_need="Emosi dan kepribadian",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang emosi dan kepribadian manusia."
    ),
    long_query=(
        "Saya mencari buku berbahasa Inggris yang menjelaskan "
        "bagaimana emosi, sifat kepribadian, dan pengalaman hidup "
        "memengaruhi perilaku manusia."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS05",
    information_need="Psikoterapi dan perubahan perilaku",
    query_group="Indonesian to English",
    query_language_code="id",
    query_language="Indonesian",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Buku tentang psikoterapi dan perubahan perilaku."
    ),
    long_query=(
        "Saya ingin menemukan buku berbahasa Inggris tentang "
        "psikoterapi, konseling, dan pendekatan psikologis untuk "
        "mengubah pola perilaku yang tidak sehat."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS06",
    information_need="Trauma dan attachment",
    query_group="Secondary Language to English",
    query_language_code="fr",
    query_language="French",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Livres sur le traumatisme et l'attachement."
    ),
    long_query=(
        "Je cherche un livre en anglais qui explique le traumatisme "
        "psychologique, les styles d'attachement et leur influence "
        "sur les relations adultes."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS07",
    information_need="Bias kognitif dan keputusan",
    query_group="Secondary Language to English",
    query_language_code="de",
    query_language="German",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Bücher über kognitive Verzerrungen."
    ),
    long_query=(
        "Ich suche ein englisches Buch darüber, wie kognitive "
        "Verzerrungen, Heuristiken und Emotionen menschliche "
        "Entscheidungen beeinflussen."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS08",
    information_need="Kesehatan mental dalam bahasa Spanyol",
    query_group="English to Non-English",
    query_language_code="en",
    query_language="English",
    target_language_code="es",
    target_language="Spanish",
    short_query=(
        "Spanish books about mental health."
    ),
    long_query=(
        "Find a Spanish-language book about emotional well-being, "
        "anxiety management, psychological resilience, and mental "
        "health care."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS09",
    information_need="Emosyon at ugnayan",
    query_group="Non-English Monolingual",
    query_language_code="tl",
    query_language="Tagalog",
    target_language_code="tl",
    target_language="Tagalog",
    short_query=(
        "Mga aklat tungkol sa emosyon at relasyon."
    ),
    long_query=(
        "Naghahanap ako ng aklat sa Tagalog tungkol sa pag-unawa "
        "sa emosyon, personalidad, at pagbuo ng mas malusog na "
        "ugnayan sa ibang tao."
    )
)

evaluation_query_pairs += create_query_pair(
    category="Psychology",
    information_need_id="PS10",
    information_need="Self-compassion dan identitas diri",
    query_group="Cross-Category",
    query_language_code="en",
    query_language="English",
    target_language_code="en",
    target_language="English",
    short_query=(
        "Psychology books about self-compassion and identity."
    ),
    long_query=(
        "I need a book that combines psychology and personal "
        "development to explain identity, self-acceptance, "
        "self-compassion, and healthier personal growth."
    ),
    overlap_category="Self Development"
)

In [ ]:
# ============================================================
# BUILD FINAL EVALUATION QUERY LIST
# ============================================================
# Menggabungkan seluruh pasangan query dan membuat query_id
# berurutan dari Q001 sampai Q100.

if not isinstance(evaluation_query_pairs, list):
    raise TypeError(
        "evaluation_query_pairs harus berupa list."
    )

if not all(
    isinstance(record, dict)
    for record in evaluation_query_pairs
):
    raise TypeError(
        "Setiap elemen evaluation_query_pairs harus berupa dictionary."
    )

# Membuat salinan agar data awal tidak berubah tanpa disengaja.
evaluation_queries = [
    record.copy()
    for record in evaluation_query_pairs
]

# Membuat query_id Q001 sampai Q100.
for index, query_record in enumerate(
    evaluation_queries,
    start=1
):
    query_record["query_id"] = f"Q{index:03d}"

print(
    "Jumlah query yang berhasil dibentuk:",
    len(evaluation_queries)
)

print(
    "Query ID pertama:",
    evaluation_queries[0]["query_id"]
)

print(
    "Query ID terakhir:",
    evaluation_queries[-1]["query_id"]
)

print(evaluation_query_pairs[0])

Jumlah query yang berhasil dibentuk: 100
Query ID pertama: Q001
Query ID terakhir: Q100
{'category': 'Self Development', 'information_need_id': 'SD01', 'information_need': 'Meningkatkan kepercayaan diri', 'formulation_id': 'F1', 'query_length': 'short', 'query_group': 'English to English', 'query_language_code': 'en', 'query_language': 'English', 'target_language_code': 'en', 'target_language': 'English', 'is_cross_lingual': False, 'is_cross_category': False, 'overlap_category': None, 'query': 'Books about building self-confidence and overcoming self-doubt.'}


In [ ]:
df_evaluation_queries.to_csv(
    "evaluation_queries.csv",
    index=False,
    encoding="utf-8"
)

print("Query evaluasi berhasil disimpan.")

Query evaluasi berhasil disimpan.


In [ ]:
df_evaluation_queries = pd.DataFrame(
    evaluation_queries
)

query_column_order = [
    "query_id",
    "category",
    "information_need_id",
    "information_need",
    "formulation_id",
    "query_length",
    "query_group",
    "query_language_code",
    "query_language",
    "target_language_code",
    "target_language",
    "is_cross_lingual",
    "is_cross_category",
    "overlap_category",
    "query"
]

df_evaluation_queries = df_evaluation_queries[
    query_column_order
]

In [ ]:
# ============================================================
# BASIC QUERY VALIDATION
# ============================================================

print(
    "Jumlah query evaluasi:",
    len(df_evaluation_queries)
)

print("\nDistribusi query per kategori:")
print(
    df_evaluation_queries["category"]
    .value_counts()
    .reindex(
        [
            "Self Development",
            "Technology",
            "Career Development",
            "Productivity",
            "Psychology"
        ]
    )
)

Jumlah query evaluasi: 100

Distribusi query per kategori:
category
Self Development      20
Technology            20
Career Development    20
Productivity          20
Psychology            20
Name: count, dtype: int64


In [ ]:
# ============================================================
# INFORMATION NEED VALIDATION
# ============================================================

information_need_counts = (
    df_evaluation_queries
    .groupby("category")[
        "information_need_id"
    ]
    .nunique()
)

print(
    "Jumlah information needs per kategori:"
)

print(information_need_counts)

Jumlah information needs per kategori:
category
Career Development    10
Productivity          10
Psychology            10
Self Development      10
Technology            10
Name: information_need_id, dtype: int64


In [ ]:
formulation_counts = (
    df_evaluation_queries
    .groupby(
        [
            "category",
            "information_need_id"
        ]
    )
    .size()
)

invalid_information_needs = (
    formulation_counts[
        formulation_counts != 2
    ]
)

print(
    "\nInformation need yang tidak mempunyai "
    "dua formulasi:"
)

print(invalid_information_needs)


Information need yang tidak mempunyai dua formulasi:
Series([], dtype: int64)


In [ ]:
query_length_validation = (
    df_evaluation_queries
    .groupby(
        [
            "category",
            "information_need_id"
        ]
    )["query_length"]
    .apply(set)
)

invalid_query_length_pairs = (
    query_length_validation[
        query_length_validation
        != {"short", "long"}
    ]
)

print(
    "\nPasangan tanpa short dan long:"
)

print(invalid_query_length_pairs)


Pasangan tanpa short dan long:
Series([], Name: query_length, dtype: object)


In [ ]:
# ============================================================
# QUERY GROUP DISTRIBUTION VALIDATION
# ============================================================

expected_query_group_distribution = {
    "English to English": 30,
    "Indonesian to English": 20,
    "Secondary Language to English": 20,
    "English to Non-English": 10,
    "Non-English Monolingual": 10,
    "Cross-Category": 10
}

actual_query_group_distribution = (
    df_evaluation_queries["query_group"]
    .value_counts()
    .reindex(
        expected_query_group_distribution.keys(),
        fill_value=0
    )
)

query_group_validation = pd.DataFrame({
    "expected": pd.Series(
        expected_query_group_distribution
    ),
    "actual": actual_query_group_distribution
})

query_group_validation["is_valid"] = (
    query_group_validation["expected"]
    == query_group_validation["actual"]
)

print(
    query_group_validation.to_string()
)

                               expected  actual  is_valid
English to English                   30      30      True
Indonesian to English                20      20      True
Secondary Language to English        20      20      True
English to Non-English               10      10      True
Non-English Monolingual              10      10      True
Cross-Category                       10      10      True


In [ ]:
# ============================================================
# PER-CATEGORY STRATIFICATION VALIDATION
# ============================================================

expected_per_category = {
    "English to English": 6,
    "Indonesian to English": 4,
    "Secondary Language to English": 4,
    "English to Non-English": 2,
    "Non-English Monolingual": 2,
    "Cross-Category": 2
}

category_group_distribution = pd.crosstab(
    df_evaluation_queries["category"],
    df_evaluation_queries["query_group"]
)

category_group_distribution = (
    category_group_distribution
    .reindex(
        columns=expected_per_category.keys(),
        fill_value=0
    )
)

print(
    category_group_distribution.to_string()
)

query_group         English to English  Indonesian to English  Secondary Language to English  English to Non-English  Non-English Monolingual  Cross-Category
category                                                                                                                                                     
Career Development                   6                      4                              4                       2                        2               2
Productivity                         6                      4                              4                       2                        2               2
Psychology                           6                      4                              4                       2                        2               2
Self Development                     6                      4                              4                       2                        2               2
Technology                           6              

In [ ]:
category_distribution_is_valid = True

for category in category_group_distribution.index:
    for query_group, expected_count in (
        expected_per_category.items()
    ):
        actual_count = (
            category_group_distribution.loc[
                category,
                query_group
            ]
        )

        if actual_count != expected_count:
            category_distribution_is_valid = False

            print(
                f"Tidak sesuai: {category} | "
                f"{query_group} | "
                f"expected={expected_count}, "
                f"actual={actual_count}"
            )

print(
    "\nKomposisi per kategori valid:",
    category_distribution_is_valid
)


Komposisi per kategori valid: True


In [ ]:
# ============================================================
# QUERY LANGUAGE DISTRIBUTION
# ============================================================

query_language_distribution = (
    df_evaluation_queries[
        [
            "query_language_code",
            "query_language"
        ]
    ]
    .value_counts()
    .rename("query_count")
    .reset_index()
)

print(
    query_language_distribution
    .to_string(index=False)
)

query_language_code query_language  query_count
                 en        English           46
                 id     Indonesian           26
                 de         German            6
                 ar         Arabic            6
                 fr         French            6
                 nl          Dutch            6
                 pt     Portuguese            2
                 tl        Tagalog            2


In [ ]:
target_language_distribution = (
    df_evaluation_queries[
        [
            "target_language_code",
            "target_language"
        ]
    ]
    .value_counts()
    .rename("query_count")
    .reset_index()
)

print(
    target_language_distribution
    .to_string(index=False)
)

target_language_code target_language  query_count
                  en         English           80
                  nl           Dutch            4
                  da          Danish            2
                  ar          Arabic            2
                  es         Spanish            2
                  de          German            2
                  fr          French            2
                  id      Indonesian            2
                  pt      Portuguese            2
                  tl         Tagalog            2


In [ ]:
# ============================================================
# FINAL QUERY QUALITY VALIDATION
# ============================================================

print(
    "Query ID duplikat:",
    df_evaluation_queries["query_id"]
    .duplicated()
    .sum()
)

print(
    "Query kosong:",
    df_evaluation_queries["query"]
    .isna()
    .sum()
    + df_evaluation_queries["query"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
)

print(
    "Information need ID duplikat lintas kategori:",
    df_evaluation_queries[
        [
            "category",
            "information_need_id"
        ]
    ]
    .drop_duplicates()
    .duplicated(
        subset=["information_need_id"]
    )
    .sum()
)

Query ID duplikat: 0
Query kosong: 0
Information need ID duplikat lintas kategori: 0


In [ ]:
# ============================================================
# BUILD STRATIFIED ANNOTATION CANDIDATE POOL
# ============================================================
# Fungsi ini:
# 1. melakukan retrieval Top-K dengan MiniLM dan MPNet;
# 2. menggabungkan kandidat dari kedua model;
# 3. menghapus kandidat buku yang sama;
# 4. mempertahankan metadata 100 query terstratifikasi;
# 5. menyiapkan kolom untuk proses anotasi manual.

from tqdm.auto import tqdm


def build_annotation_pool(
    query_dataframe,
    minilm_model,
    minilm_embeddings,
    mpnet_model,
    mpnet_embeddings,
    books_dataframe,
    top_k=50
):
    pooled_records = []

    # Metadata query yang wajib tersedia.
    required_query_columns = [
        "query_id",
        "category",
        "information_need_id",
        "information_need",
        "formulation_id",
        "query_length",
        "query_group",
        "query_language_code",
        "query_language",
        "target_language_code",
        "target_language",
        "is_cross_lingual",
        "is_cross_category",
        "overlap_category",
        "query"
    ]

    missing_query_columns = [
        column
        for column in required_query_columns
        if column not in query_dataframe.columns
    ]

    if missing_query_columns:
        raise ValueError(
            "Kolom query berikut belum tersedia: "
            f"{missing_query_columns}"
        )

    # Metadata buku yang diperlukan untuk anotasi.
    required_book_columns = [
        "book_id",
        "title",
        "author",
        "genre_text",
        "desc",
        "target_categories"
    ]

    missing_book_columns = [
        column
        for column in required_book_columns
        if column not in books_dataframe.columns
    ]

    if missing_book_columns:
        raise ValueError(
            "Kolom buku berikut belum tersedia: "
            f"{missing_book_columns}"
        )

    # Metadata buku dibuat unik berdasarkan book_id.
    book_metadata = (
        books_dataframe[
            required_book_columns
        ]
        .drop_duplicates(
            subset=["book_id"]
        )
        .copy()
    )

    # Memproses seluruh 100 query evaluasi.
    for row in tqdm(
        query_dataframe.itertuples(index=False),
        total=len(query_dataframe),
        desc="Membangun candidate pool"
    ):
        query_text = row.query

        # ====================================================
        # RETRIEVAL MENGGUNAKAN MINILM
        # ====================================================

        _, minilm_candidates = retrieve_candidates(
            query=query_text,
            embedding_model=minilm_model,
            corpus_embeddings=minilm_embeddings,
            dataframe=books_dataframe,
            top_k=top_k
        )

        minilm_candidates = (
            minilm_candidates[
                [
                    "book_id",
                    "retrieval_rank",
                    "retrieval_score"
                ]
            ]
            .rename(
                columns={
                    "retrieval_rank":
                        "minilm_retrieval_rank",
                    "retrieval_score":
                        "minilm_retrieval_score"
                }
            )
            .copy()
        )

        minilm_candidates[
            "retrieved_by_minilm"
        ] = True

        # ====================================================
        # RETRIEVAL MENGGUNAKAN MPNET
        # ====================================================

        _, mpnet_candidates = retrieve_candidates(
            query=query_text,
            embedding_model=mpnet_model,
            corpus_embeddings=mpnet_embeddings,
            dataframe=books_dataframe,
            top_k=top_k
        )

        mpnet_candidates = (
            mpnet_candidates[
                [
                    "book_id",
                    "retrieval_rank",
                    "retrieval_score"
                ]
            ]
            .rename(
                columns={
                    "retrieval_rank":
                        "mpnet_retrieval_rank",
                    "retrieval_score":
                        "mpnet_retrieval_score"
                }
            )
            .copy()
        )

        mpnet_candidates[
            "retrieved_by_mpnet"
        ] = True

        # ====================================================
        # MENGGABUNGKAN HASIL KEDUA RETRIEVER
        # ====================================================
        # Outer merge memastikan kandidat yang hanya ditemukan
        # oleh salah satu model tetap ikut dalam annotation pool.

        pooled_candidates = pd.merge(
            minilm_candidates,
            mpnet_candidates,
            on="book_id",
            how="outer",
            validate="one_to_one"
        )

        # Nilai kosong berarti buku tidak ditemukan oleh model.
        pooled_candidates["retrieved_by_minilm"] = (
            pooled_candidates["retrieved_by_minilm"]
            .fillna(False)
            .astype(bool)
        )

        pooled_candidates["retrieved_by_mpnet"] = (
            pooled_candidates["retrieved_by_mpnet"]
            .fillna(False)
            .astype(bool)
        )

        # Menambahkan metadata buku.
        pooled_candidates = pd.merge(
            pooled_candidates,
            book_metadata,
            on="book_id",
            how="left",
            validate="many_to_one"
        )

        # ====================================================
        # MENAMBAHKAN METADATA QUERY TERSTRATIFIKASI
        # ====================================================

        query_metadata = {
            "query_id": row.query_id,
            "query_category": row.category,
            "information_need_id":
                row.information_need_id,
            "information_need":
                row.information_need,
            "formulation_id":
                row.formulation_id,
            "query_length":
                row.query_length,
            "query_group":
                row.query_group,
            "query_language_code":
                row.query_language_code,
            "query_language":
                row.query_language,
            "target_language_code":
                row.target_language_code,
            "target_language":
                row.target_language,
            "is_cross_lingual":
                row.is_cross_lingual,
            "is_cross_category":
                row.is_cross_category,
            "overlap_category":
                row.overlap_category,
            "query":
                row.query
        }

        # Menambahkan metadata query pada seluruh kandidat.
        for column_name, column_value in (
            query_metadata.items()
        ):
            pooled_candidates[column_name] = (
                column_value
            )

        # Kolom yang akan diisi anotator.
        pooled_candidates["relevance_label"] = (
            np.nan
        )

        pooled_candidates["annotation_note"] = ""

        pooled_records.append(
            pooled_candidates
        )

    # Menggabungkan kandidat dari seluruh query.
    annotation_pool = pd.concat(
        pooled_records,
        ignore_index=True
    )

    # ========================================================
    # MEMBUAT ID ANOTASI UNIK
    # ========================================================
    # annotation_id digunakan untuk menggabungkan kembali hasil
    # anotasi buta dengan informasi ranking model.

    annotation_pool.insert(
        0,
        "annotation_id",
        [
            f"A{index:06d}"
            for index in range(
                1,
                len(annotation_pool) + 1
            )
        ]
    )

    return annotation_pool

In [ ]:
# ============================================================
# LOAD EMBEDDING MODELS FOR POOLING
# ============================================================

minilm_model = SentenceTransformer(
    embedding_model_names[
        "multilingual_minilm"
    ],
    device=device
)

mpnet_model = SentenceTransformer(
    embedding_model_names[
        "multilingual_mpnet"
    ],
    device=device
)

print("MiniLM dan MPNet berhasil dimuat.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MiniLM dan MPNet berhasil dimuat.


In [ ]:
# ============================================================
# CREATE STRATIFIED ANNOTATION POOL
# ============================================================

annotation_pool = build_annotation_pool(
    query_dataframe=df_evaluation_queries,
    minilm_model=minilm_model,
    minilm_embeddings=minilm_embeddings,
    mpnet_model=mpnet_model,
    mpnet_embeddings=mpnet_embeddings,
    books_dataframe=df_books_model,
    top_k=50
)

print(
    "Jumlah pasangan query-buku untuk dinilai:",
    len(annotation_pool)
)

print(
    "Jumlah query unik:",
    annotation_pool["query_id"].nunique()
)

print(
    "Rata-rata kandidat unik per query:",
    round(
        annotation_pool
        .groupby("query_id")
        .size()
        .mean(),
        2
    )
)

print(
    "Jumlah annotation_id duplikat:",
    annotation_pool["annotation_id"]
    .duplicated()
    .sum()
)

Membangun candidate pool:   0%|          | 0/100 [00:00<?, ?it/s]

Jumlah pasangan query-buku untuk dinilai: 7684
Jumlah query unik: 100
Rata-rata kandidat unik per query: 76.84
Jumlah annotation_id duplikat: 0


In [ ]:
# ============================================================
# CANDIDATE POOL SIZE VALIDATION
# ============================================================
# Dengan Top-50 dari dua model:
# - minimum kandidat unik teoretis = 50;
# - maksimum kandidat unik teoretis = 100.

candidate_count_per_query = (
    annotation_pool
    .groupby("query_id")
    .size()
)

print(
    candidate_count_per_query
    .describe()
    .round(2)
)

invalid_candidate_counts = (
    candidate_count_per_query[
        (candidate_count_per_query < 50)
        | (candidate_count_per_query > 100)
    ]
)

print(
    "\nQuery dengan jumlah kandidat tidak valid:"
)

print(invalid_candidate_counts)

count   100.00
mean     76.84
std       5.50
min      64.00
25%      73.00
50%      77.00
75%      81.00
max      92.00
dtype: float64

Query dengan jumlah kandidat tidak valid:
Series([], dtype: int64)


In [ ]:
# ============================================================
# STRATIFICATION VALIDATION IN ANNOTATION POOL
# ============================================================

query_metadata_check = (
    annotation_pool[
        [
            "query_id",
            "query_category",
            "information_need_id",
            "formulation_id",
            "query_length",
            "query_group",
            "query_language_code",
            "target_language_code",
            "is_cross_lingual",
            "is_cross_category"
        ]
    ]
    .drop_duplicates(
        subset=["query_id"]
    )
)

print(
    "Jumlah metadata query unik:",
    len(query_metadata_check)
)

print("\nDistribusi kategori:")
print(
    query_metadata_check[
        "query_category"
    ]
    .value_counts()
)

print("\nDistribusi kelompok query:")
print(
    query_metadata_check[
        "query_group"
    ]
    .value_counts()
)

print("\nDistribusi panjang query:")
print(
    query_metadata_check[
        "query_length"
    ]
    .value_counts()
)

Jumlah metadata query unik: 100

Distribusi kategori:
query_category
Self Development      20
Technology            20
Career Development    20
Productivity          20
Psychology            20
Name: count, dtype: int64

Distribusi kelompok query:
query_group
English to English               30
Indonesian to English            20
Secondary Language to English    20
English to Non-English           10
Non-English Monolingual          10
Cross-Category                   10
Name: count, dtype: int64

Distribusi panjang query:
query_length
short    50
long     50
Name: count, dtype: int64


In [ ]:
# ============================================================
# CREATE DESCRIPTION PREVIEW
# ============================================================
# Preview hanya digunakan dalam lembar anotasi.
# Deskripsi asli tetap disimpan di annotation_pool.

annotation_pool["desc_preview"] = (
    annotation_pool["desc"]
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
    .str.slice(0, 700)
)

In [ ]:
# ============================================================
# FULL ANNOTATION TEMPLATE
# ============================================================

annotation_columns = [
    # Identitas anotasi
    "annotation_id",

    # Metadata query
    "query_id",
    "query_category",
    "information_need_id",
    "information_need",
    "formulation_id",
    "query_length",
    "query_group",
    "query_language_code",
    "query_language",
    "target_language_code",
    "target_language",
    "is_cross_lingual",
    "is_cross_category",
    "overlap_category",
    "query",

    # Metadata buku
    "book_id",
    "title",
    "author",
    "genre_text",
    "target_categories",
    "desc_preview",

    # Informasi retrieval MiniLM
    "retrieved_by_minilm",
    "minilm_retrieval_rank",
    "minilm_retrieval_score",

    # Informasi retrieval MPNet
    "retrieved_by_mpnet",
    "mpnet_retrieval_rank",
    "mpnet_retrieval_score",

    # Kolom anotasi
    "relevance_label",
    "annotation_note"
]

annotation_template = annotation_pool[
    annotation_columns
].copy()

annotation_template.to_csv(
    "relevance_annotation_template_stratified.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Template anotasi lengkap tersimpan:",
    annotation_template.shape
)

Template anotasi lengkap tersimpan: (7684, 30)


In [ ]:
# ============================================================
# BLIND ANNOTATION TEMPLATE
# ============================================================

blind_annotation_columns = [
    # Identitas yang diperlukan untuk penggabungan kembali
    "annotation_id",

    # Informasi kebutuhan pengguna
    "query_id",
    "query_category",
    "information_need_id",
    "information_need",
    "formulation_id",
    "query_length",
    "query_language",
    "target_language",
    "is_cross_lingual",
    "is_cross_category",
    "overlap_category",
    "query",

    # Informasi buku yang dinilai
    "book_id",
    "title",
    "author",
    "genre_text",
    "desc_preview",

    # Kolom yang diisi anotator
    "relevance_label",
    "annotation_note"
]

blind_annotation_template = (
    annotation_pool[
        blind_annotation_columns
    ]
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(drop=True)
)

blind_annotation_template.to_csv(
    "blind_relevance_annotation_stratified.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Template anotasi buta tersimpan:",
    blind_annotation_template.shape
)

Template anotasi buta tersimpan: (7684, 20)


In [ ]:
# ============================================================
# GROUPED BLIND ANNOTATION TEMPLATE
# ============================================================
# Kandidat diacak dalam setiap query agar rank model tidak
# terlihat, tetapi seluruh kandidat untuk query yang sama tetap
# berada berdekatan.

grouped_blind_annotation = (
    annotation_pool
    .groupby(
        "query_id",
        group_keys=False
    )
    .sample(
        frac=1,
        random_state=42
    )
    .sort_values(
        "query_id",
        kind="stable"
    )
    .reset_index(drop=True)
)

grouped_blind_annotation = (
    grouped_blind_annotation[
        blind_annotation_columns
    ]
)

grouped_blind_annotation.to_csv(
    "grouped_blind_relevance_annotation.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Template anotasi buta terkelompok tersimpan:",
    grouped_blind_annotation.shape
)

Template anotasi buta terkelompok tersimpan: (7684, 20)


In [ ]:
# ============================================================
# ANNOTATION TEMPLATE VALIDATION
# ============================================================

print(
    "Annotation ID lengkap unik:",
    annotation_template[
        "annotation_id"
    ].is_unique
)

print(
    "Annotation ID blind unik:",
    grouped_blind_annotation[
        "annotation_id"
    ].is_unique
)

print(
    "Jumlah baris sama:",
    len(annotation_template)
    == len(grouped_blind_annotation)
)

print(
    "Jumlah query lengkap:",
    annotation_template[
        "query_id"
    ].nunique()
)

print(
    "Jumlah query blind:",
    grouped_blind_annotation[
        "query_id"
    ].nunique()
)

print(
    "Label awal masih kosong:",
    grouped_blind_annotation[
        "relevance_label"
    ].isna().sum()
)

Annotation ID lengkap unik: True
Annotation ID blind unik: True
Jumlah baris sama: True
Jumlah query lengkap: 100
Jumlah query blind: 100
Label awal masih kosong: 7684


In [ ]:
# ============================================================
# SAVE QUERY STRATIFICATION METADATA
# ============================================================

query_stratification_metadata = (
    df_evaluation_queries[
        [
            "query_id",
            "category",
            "information_need_id",
            "information_need",
            "formulation_id",
            "query_length",
            "query_group",
            "query_language_code",
            "query_language",
            "target_language_code",
            "target_language",
            "is_cross_lingual",
            "is_cross_category",
            "overlap_category"
        ]
    ]
    .copy()
)

query_stratification_metadata.to_csv(
    "query_stratification_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Metadata stratifikasi tersimpan:",
    query_stratification_metadata.shape
)

Metadata stratifikasi tersimpan: (100, 14)


In [ ]:
# ============================================================
# EVALUATION METRICS
# ============================================================

def precision_at_k(relevance_labels, k):
    labels = np.asarray(relevance_labels)[:k]

    if k == 0:
        return 0.0

    relevant_count = (
        labels >= 1
    ).sum()

    return relevant_count / k


def recall_at_k(
    relevance_labels,
    total_relevant,
    k
):
    if total_relevant == 0:
        return 0.0

    labels = np.asarray(relevance_labels)[:k]

    relevant_retrieved = (
        labels >= 1
    ).sum()

    return relevant_retrieved / total_relevant


def reciprocal_rank_at_k(
    relevance_labels,
    k
):
    labels = np.asarray(relevance_labels)[:k]

    relevant_positions = np.where(
        labels >= 1
    )[0]

    if len(relevant_positions) == 0:
        return 0.0

    first_relevant_rank = (
        relevant_positions[0] + 1
    )

    return 1.0 / first_relevant_rank


def dcg_at_k(relevance_labels, k):
    labels = np.asarray(
        relevance_labels,
        dtype=float
    )[:k]

    if len(labels) == 0:
        return 0.0

    positions = np.arange(
        2,
        len(labels) + 2
    )

    gains = (
        np.power(2, labels) - 1
    )

    discounts = np.log2(positions)

    return np.sum(
        gains / discounts
    )


def ndcg_at_k(relevance_labels, k):
    labels = np.asarray(
        relevance_labels,
        dtype=float
    )

    actual_dcg = dcg_at_k(
        labels,
        k
    )

    ideal_labels = np.sort(
        labels
    )[::-1]

    ideal_dcg = dcg_at_k(
        ideal_labels,
        k
    )

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg

In [ ]:
# ============================================================
# METRIC FUNCTION UNIT TEST
# ============================================================
# Hasil pada bagian ini hanya memvalidasi fungsi metrik.
# Angka yang dihasilkan bukan performa MiniLM atau MPNet.

sample_labels = [2, 0, 1, 0, 2]

print(
    "Precision@5:",
    precision_at_k(sample_labels, 5)
)

print(
    "Recall@5:",
    recall_at_k(
        sample_labels,
        total_relevant=3,
        k=5
    )
)

print(
    "MRR@5:",
    reciprocal_rank_at_k(
        sample_labels,
        5
    )
)

print(
    "NDCG@5:",
    ndcg_at_k(
        sample_labels,
        5
    )
)

Precision@5: 0.6
Recall@5: 1.0
MRR@5: 1.0
NDCG@5: 0.8642203869628404


In [ ]:
# ============================================================
# IMPORT LIBRARIES FOR ANNOTATION FILE EXPORT
# ============================================================

import os
import re
import shutil

import numpy as np
import pandas as pd

# ============================================================
# PREPARE MINIMAL BLIND ANNOTATION DATA
# ============================================================
# Hanya mempertahankan informasi yang benar-benar diperlukan
# untuk membandingkan kebutuhan pengguna dengan isi buku.
#
# Informasi model, rank, similarity score, rating, popularitas,
# dan nama penulis tidak disertakan agar anotasi tetap buta.

required_annotation_columns = [
    "annotation_id",
    "query_id",
    "information_need",
    "query",
    "target_language",
    "book_id",
    "title",
    "genre_text",
    "desc"
]

missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in annotation_pool.columns
]

if missing_annotation_columns:
    raise ValueError(
        "Kolom berikut belum tersedia pada annotation_pool: "
        f"{missing_annotation_columns}"
    )

blind_annotation_base = annotation_pool[
    required_annotation_columns
].copy()

# Mengganti nama desc agar jelas bahwa deskripsi yang digunakan
# adalah deskripsi lengkap, bukan preview.
blind_annotation_base = blind_annotation_base.rename(
    columns={
        "desc": "description_full"
    }
)

# Menjamin deskripsi disimpan sebagai teks.
blind_annotation_base["description_full"] = (
    blind_annotation_base["description_full"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Kolom yang nantinya diisi oleh anotator.
blind_annotation_base["relevance_label"] = np.nan
blind_annotation_base["annotation_note"] = ""

print(
    "Dimensi data dasar anotasi:",
    blind_annotation_base.shape
)

Dimensi data dasar anotasi: (7684, 11)


In [ ]:
# ============================================================
# VALIDATE BASE ANNOTATION DATA
# ============================================================

print(
    "Jumlah query unik:",
    blind_annotation_base["query_id"].nunique()
)

print(
    "Jumlah annotation_id duplikat:",
    blind_annotation_base["annotation_id"]
    .duplicated()
    .sum()
)

print(
    "Jumlah deskripsi kosong:",
    blind_annotation_base["description_full"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Jumlah label yang masih kosong:",
    blind_annotation_base["relevance_label"]
    .isna()
    .sum()
)

Jumlah query unik: 100
Jumlah annotation_id duplikat: 0
Jumlah deskripsi kosong: 0
Jumlah label yang masih kosong: 7684


In [ ]:
# ============================================================
# SAFE FILE NAME FUNCTION
# ============================================================
# Menghapus karakter yang berpotensi bermasalah pada nama file.

def sanitize_filename(value):
    filename = str(value).strip()

    filename = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        filename
    )

    filename = re.sub(
        r"\s+",
        "_",
        filename
    )

    return filename

In [ ]:
# ============================================================
# EXPORT ONE CSV FILE PER QUERY
# ============================================================
# Setiap anotator memperoleh 100 file CSV.
#
# Kandidat buku diacak di dalam setiap query agar urutan hasil
# retrieval asli tidak dapat diketahui anotator.
#
# random seed berbeda digunakan untuk kedua anotator, sehingga
# urutan kandidat pada file mereka juga berbeda.

def export_annotation_files_by_query(
    annotation_dataframe,
    annotator_id,
    output_directory,
    base_random_state
):
    # Menghapus folder lama agar tidak bercampur dengan hasil
    # dari proses sebelumnya.
    if os.path.exists(output_directory):
        shutil.rmtree(output_directory)

    os.makedirs(
        output_directory,
        exist_ok=True
    )

    exported_files = []

    query_ids = sorted(
        annotation_dataframe["query_id"]
        .unique()
    )

    for query_id in query_ids:
        query_data = annotation_dataframe[
            annotation_dataframe["query_id"]
            == query_id
        ].copy()

        # Mengambil angka dari Q001, Q002, dan seterusnya untuk
        # menghasilkan seed yang konsisten pada setiap query.
        query_number_match = re.search(
            r"\d+",
            str(query_id)
        )

        query_number = (
            int(query_number_match.group())
            if query_number_match
            else 0
        )

        query_random_state = (
            base_random_state
            + query_number
        )

        # Mengacak kandidat hanya di dalam query yang sama.
        query_data = (
            query_data
            .sample(
                frac=1,
                random_state=query_random_state
            )
            .reset_index(drop=True)
        )

        # Menambahkan identitas anotator.
        query_data.insert(
            1,
            "annotator_id",
            annotator_id
        )

        # Label dan catatan dipastikan kosong.
        query_data["relevance_label"] = np.nan
        query_data["annotation_note"] = ""

        # Urutan kolom yang mudah dibaca anotator.
        final_columns = [
            "annotation_id",
            "annotator_id",
            "query_id",
            "information_need",
            "query",
            "target_language",
            "book_id",
            "title",
            "genre_text",
            "description_full",
            "relevance_label",
            "annotation_note"
        ]

        query_data = query_data[
            final_columns
        ]

        safe_query_id = sanitize_filename(
            query_id
        )

        output_path = os.path.join(
            output_directory,
            f"{safe_query_id}.csv"
        )

        # utf-8-sig membantu kompatibilitas karakter multilingual
        # saat file dibuka menggunakan Microsoft Excel.
        query_data.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        exported_files.append(
            {
                "annotator_id": annotator_id,
                "query_id": query_id,
                "file_path": output_path,
                "candidate_count": len(query_data)
            }
        )

    export_summary = pd.DataFrame(
        exported_files
    )

    return export_summary

In [ ]:
# ============================================================
# CREATE ANNOTATOR 1 FILES
# ============================================================

annotator_1_summary = export_annotation_files_by_query(
    annotation_dataframe=blind_annotation_base,
    annotator_id="ANNOTATOR_1",
    output_directory=(
        "annotation_files/annotator_1"
    ),
    base_random_state=42
)

print(
    "Jumlah file Anotator 1:",
    len(annotator_1_summary)
)

Jumlah file Anotator 1: 100


In [ ]:
# ============================================================
# CREATE ANNOTATOR 2 FILES
# ============================================================

annotator_2_summary = export_annotation_files_by_query(
    annotation_dataframe=blind_annotation_base,
    annotator_id="ANNOTATOR_2",
    output_directory=(
        "annotation_files/annotator_2"
    ),
    base_random_state=84
)

print(
    "Jumlah file Anotator 2:",
    len(annotator_2_summary)
)

Jumlah file Anotator 2: 100


In [ ]:
# ============================================================
# SAVE EXPORT SUMMARIES
# ============================================================

annotator_1_summary.to_csv(
    "annotation_files/annotator_1/"
    "00_file_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

annotator_2_summary.to_csv(
    "annotation_files/annotator_2/"
    "00_file_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Ringkasan file anotasi berhasil disimpan.")

Ringkasan file anotasi berhasil disimpan.


In [ ]:
# ============================================================
# VALIDATE EXPORTED QUERY FILES
# ============================================================

def validate_exported_annotation_files(
    export_summary,
    expected_query_count=100
):
    print(
        "Jumlah file query:",
        len(export_summary)
    )

    print(
        "Jumlah query unik:",
        export_summary["query_id"]
        .nunique()
    )

    print(
        "Jumlah file yang tidak ditemukan:",
        (
            ~export_summary["file_path"]
            .apply(os.path.exists)
        ).sum()
    )

    print(
        "Jumlah kandidat minimum per query:",
        export_summary["candidate_count"]
        .min()
    )

    print(
        "Jumlah kandidat maksimum per query:",
        export_summary["candidate_count"]
        .max()
    )

    if len(export_summary) != expected_query_count:
        raise ValueError(
            "Jumlah file query tidak sesuai dengan "
            f"{expected_query_count}."
        )

    if (
        export_summary["query_id"].nunique()
        != expected_query_count
    ):
        raise ValueError(
            "Jumlah query unik pada file ekspor tidak sesuai."
        )

    if (
        ~export_summary["file_path"]
        .apply(os.path.exists)
    ).any():
        raise FileNotFoundError(
            "Terdapat file anotasi yang gagal dibuat."
        )

    print("Seluruh file anotasi valid.")

In [ ]:
print("VALIDASI ANOTATOR 1")

validate_exported_annotation_files(
    annotator_1_summary
)

print("\nVALIDASI ANOTATOR 2")

validate_exported_annotation_files(
    annotator_2_summary
)

VALIDASI ANOTATOR 1
Jumlah file query: 100
Jumlah query unik: 100
Jumlah file yang tidak ditemukan: 0
Jumlah kandidat minimum per query: 64
Jumlah kandidat maksimum per query: 92
Seluruh file anotasi valid.

VALIDASI ANOTATOR 2
Jumlah file query: 100
Jumlah query unik: 100
Jumlah file yang tidak ditemukan: 0
Jumlah kandidat minimum per query: 64
Jumlah kandidat maksimum per query: 92
Seluruh file anotasi valid.


In [ ]:
# ============================================================
# VALIDATE MATCHING CANDIDATES BETWEEN ANNOTATORS
# ============================================================

def load_all_annotation_ids(
    export_summary
):
    annotation_ids = set()

    for file_path in export_summary[
        "file_path"
    ]:
        query_file = pd.read_csv(
            file_path,
            encoding="utf-8-sig",
            usecols=["annotation_id"]
        )

        annotation_ids.update(
            query_file["annotation_id"]
            .astype(str)
            .tolist()
        )

    return annotation_ids

annotator_1_ids = load_all_annotation_ids(
    annotator_1_summary
)

annotator_2_ids = load_all_annotation_ids(
    annotator_2_summary
)

print(
    "Jumlah annotation_id Anotator 1:",
    len(annotator_1_ids)
)

print(
    "Jumlah annotation_id Anotator 2:",
    len(annotator_2_ids)
)

print(
    "Kandidat kedua anotator sama:",
    annotator_1_ids == annotator_2_ids
)

Jumlah annotation_id Anotator 1: 7684
Jumlah annotation_id Anotator 2: 7684
Kandidat kedua anotator sama: True


In [ ]:
# ============================================================
# BLINDNESS VALIDATION
# ============================================================

hidden_columns = {
    "retrieved_by_minilm",
    "minilm_retrieval_rank",
    "minilm_retrieval_score",
    "retrieved_by_mpnet",
    "mpnet_retrieval_rank",
    "mpnet_retrieval_score",
    "rating",
    "totalratings",
    "target_categories",
    "author"
}

sample_annotator_file = pd.read_csv(
    annotator_1_summary.iloc[0][
        "file_path"
    ],
    encoding="utf-8-sig"
)

leaked_columns = (
    hidden_columns
    .intersection(
        sample_annotator_file.columns
    )
)

print(
    "Kolom yang berpotensi membocorkan hasil model:",
    leaked_columns
)

Kolom yang berpotensi membocorkan hasil model: set()


In [ ]:
# ============================================================
# CREATE ZIP FILES
# ============================================================

annotator_1_zip = shutil.make_archive(
    base_name="blind_annotation_annotator_1",
    format="zip",
    root_dir="annotation_files/annotator_1"
)

annotator_2_zip = shutil.make_archive(
    base_name="blind_annotation_annotator_2",
    format="zip",
    root_dir="annotation_files/annotator_2"
)

print(
    "ZIP Anotator 1:",
    annotator_1_zip
)

print(
    "ZIP Anotator 2:",
    annotator_2_zip
)

ZIP Anotator 1: /content/blind_annotation_annotator_1.zip
ZIP Anotator 2: /content/blind_annotation_annotator_2.zip


In [ ]:
# ============================================================
# DOWNLOAD ZIP FILES FROM GOOGLE COLAB
# ============================================================

from google.colab import files

# files.download(
#     "blind_annotation_annotator_1.zip"
# )

In [ ]:
# files.download(
#     "blind_annotation_annotator_2.zip"
# )

## Pengecekan Hasil Kedua Anotator

In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
# Menghubungkan Google Colab dengan Google Drive agar file
# hasil anotasi dapat dibaca langsung dari folder Drive.

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# IMPORT LIBRARIES FOR GROUND TRUTH VALIDATION
# ============================================================

import os
import re
import glob

import numpy as np
import pandas as pd

from sklearn.metrics import (
    cohen_kappa_score,
    confusion_matrix
)

In [ ]:
# ============================================================
# ANNOTATION FOLDER CONFIGURATION
# ============================================================

annotator_1_folder = (
    "/content/drive/MyDrive/"
    "hasil_anotasi_anotator_1"
)

annotator_2_folder = (
    "/content/drive/MyDrive/"
    "hasil_anotasi_anotator_2"
)

print(
    "Folder Anotator 1 ditemukan:",
    os.path.isdir(annotator_1_folder)
)

print(
    "Folder Anotator 2 ditemukan:",
    os.path.isdir(annotator_2_folder)
)

Folder Anotator 1 ditemukan: True
Folder Anotator 2 ditemukan: True


In [ ]:
# ============================================================
# READ ONE ANNOTATION FILE
# ============================================================
# Fungsi ini membaca satu CSV, memvalidasi kolom utama, dan
# menambahkan identitas sumber file untuk keperluan audit.

def read_annotation_file(
    file_path,
    annotator_id
):
    dataframe = pd.read_csv(
        file_path,
        encoding="utf-8-sig"
    )

    required_columns = [
        "annotation_id",
        "annotator_id",
        "query_id",
        "book_id",
        "relevance_label"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Kolom tidak ditemukan pada {file_path}: "
            f"{missing_columns}"
        )

    # Menyamakan identitas anotator berdasarkan folder sumber.
    dataframe["annotator_id"] = annotator_id

    # Menyimpan nama file sumber untuk audit.
    dataframe["source_file"] = os.path.basename(
        file_path
    )

    return dataframe

In [ ]:
# ============================================================
# EXTRACT QUERY ID FROM FILE NAME
# ============================================================

def extract_query_id_from_filename(file_path):
    file_name = os.path.basename(file_path)

    query_match = re.search(
        r"Q\d{3}",
        file_name,
        flags=re.IGNORECASE
    )

    if query_match is None:
        return None

    return query_match.group().upper()

print(
    extract_query_id_from_filename(
        "Q001_labeled.csv"
    )
)

print(
    extract_query_id_from_filename(
        "labeled_Q002.csv"
    )
)

Q001
Q002


In [ ]:
# ============================================================
# LOAD ALL ANNOTATION FILES FROM ONE FOLDER
# ============================================================
# Fungsi membaca seluruh file CSV, mengurutkannya berdasarkan
# query_id, kemudian menggabungkannya menjadi satu DataFrame.

def load_annotator_files(
    folder_path,
    annotator_id,
    expected_query_count=100
):
    csv_files = glob.glob(
        os.path.join(
            folder_path,
            "*.csv"
        )
    )

    # Hanya mengambil file yang mengandung pola Q001-Q100.
    annotation_files = [
        file_path
        for file_path in csv_files
        if extract_query_id_from_filename(
            file_path
        ) is not None
    ]

    annotation_files = sorted(
        annotation_files,
        key=lambda file_path: (
            extract_query_id_from_filename(
                file_path
            )
        )
    )

    print(
        f"Jumlah file {annotator_id}:",
        len(annotation_files)
    )

    if len(annotation_files) != expected_query_count:
        print(
            f"Peringatan: jumlah file {annotator_id} "
            f"bukan {expected_query_count}."
        )

    annotation_dataframes = []

    for file_path in annotation_files:
        query_id_from_filename = (
            extract_query_id_from_filename(
                file_path
            )
        )

        query_dataframe = read_annotation_file(
            file_path=file_path,
            annotator_id=annotator_id
        )

        query_ids_inside_file = set(
            query_dataframe["query_id"]
            .astype(str)
            .str.upper()
            .unique()
        )

        # Satu file seharusnya hanya berisi satu query.
        if len(query_ids_inside_file) != 1:
            raise ValueError(
                f"File {file_path} berisi lebih dari "
                "satu query_id."
            )

        query_id_inside_file = next(
            iter(query_ids_inside_file)
        )

        # Memastikan nama file sesuai isi query.
        if (
            query_id_inside_file
            != query_id_from_filename
        ):
            raise ValueError(
                f"Query ID pada nama file "
                f"({query_id_from_filename}) berbeda dengan "
                f"isi file ({query_id_inside_file}): "
                f"{file_path}"
            )

        annotation_dataframes.append(
            query_dataframe
        )

    if not annotation_dataframes:
        raise ValueError(
            f"Tidak ada file anotasi yang berhasil dibaca "
            f"dari {folder_path}."
        )

    combined_annotations = pd.concat(
        annotation_dataframes,
        ignore_index=True
    )

    return combined_annotations

In [ ]:
# ============================================================
# LOAD ANNOTATOR 1 AND ANNOTATOR 2 RESULTS
# ============================================================

annotations_1 = load_annotator_files(
    folder_path=annotator_1_folder,
    annotator_id="ANNOTATOR_1",
    expected_query_count=100
)

annotations_2 = load_annotator_files(
    folder_path=annotator_2_folder,
    annotator_id="ANNOTATOR_2",
    expected_query_count=100
)

print(
    "\nJumlah baris Anotator 1:",
    len(annotations_1)
)

print(
    "Jumlah baris Anotator 2:",
    len(annotations_2)
)

print(
    "Jumlah query Anotator 1:",
    annotations_1["query_id"].nunique()
)

print(
    "Jumlah query Anotator 2:",
    annotations_2["query_id"].nunique()
)

Jumlah file ANNOTATOR_1: 100
Jumlah file ANNOTATOR_2: 100

Jumlah baris Anotator 1: 7684
Jumlah baris Anotator 2: 7684
Jumlah query Anotator 1: 100
Jumlah query Anotator 2: 100


In [ ]:
# ============================================================
# VALIDATE ANNOTATION LABELS
# ============================================================
# Validasi meliputi:
# - annotation_id tidak duplikat;
# - label tidak kosong;
# - label hanya terdiri dari 0, 1, dan 2;
# - satu annotation_id hanya mewakili satu pasangan query-buku.

def validate_annotations(
    dataframe,
    annotator_name
):
    print(
        f"\nVALIDASI {annotator_name}"
    )

    required_columns = [
        "annotation_id",
        "query_id",
        "book_id",
        "relevance_label"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{annotator_name}: kolom tidak ditemukan "
            f"{missing_columns}"
        )

    duplicate_annotation_ids = (
        dataframe["annotation_id"]
        .duplicated()
        .sum()
    )

    missing_labels = (
        dataframe["relevance_label"]
        .isna()
        .sum()
    )

    # Mengubah label menjadi numerik.
    numeric_labels = pd.to_numeric(
        dataframe["relevance_label"],
        errors="coerce"
    )

    non_numeric_labels = (
        numeric_labels.isna()
        & dataframe["relevance_label"].notna()
    ).sum()

    invalid_label_mask = (
        numeric_labels.notna()
        & ~numeric_labels.isin(
            [0, 1, 2]
        )
    )

    invalid_labels = dataframe.loc[
        invalid_label_mask,
        "relevance_label"
    ].unique()

    inconsistent_pairs = (
        dataframe
        .groupby("annotation_id")[
            ["query_id", "book_id"]
        ]
        .nunique()
        .gt(1)
        .any(axis=1)
        .sum()
    )

    print(
        "Jumlah baris:",
        len(dataframe)
    )

    print(
        "Jumlah query:",
        dataframe["query_id"].nunique()
    )

    print(
        "Annotation ID duplikat:",
        duplicate_annotation_ids
    )

    print(
        "Label kosong:",
        missing_labels
    )

    print(
        "Label bukan angka:",
        non_numeric_labels
    )

    print(
        "Label di luar 0, 1, 2:",
        invalid_labels
    )

    print(
        "Pasangan query-buku tidak konsisten:",
        inconsistent_pairs
    )

    print("\nDistribusi label:")
    print(
        numeric_labels
        .value_counts(dropna=False)
        .sort_index()
    )

    if duplicate_annotation_ids > 0:
        raise ValueError(
            f"{annotator_name}: terdapat annotation_id "
            "duplikat."
        )

    if missing_labels > 0:
        raise ValueError(
            f"{annotator_name}: masih terdapat label kosong."
        )

    if non_numeric_labels > 0:
        raise ValueError(
            f"{annotator_name}: terdapat label bukan angka."
        )

    if len(invalid_labels) > 0:
        raise ValueError(
            f"{annotator_name}: ditemukan label selain "
            "0, 1, dan 2."
        )

    if inconsistent_pairs > 0:
        raise ValueError(
            f"{annotator_name}: terdapat pasangan "
            "query-buku tidak konsisten."
        )

    dataframe["relevance_label"] = (
        numeric_labels.astype(int)
    )

    print(
        f"{annotator_name}: seluruh hasil anotasi valid."
    )

    return dataframe

In [ ]:
annotations_1 = validate_annotations(
    dataframe=annotations_1,
    annotator_name="Anotator 1"
)

annotations_2 = validate_annotations(
    dataframe=annotations_2,
    annotator_name="Anotator 2"
)


VALIDASI Anotator 1
Jumlah baris: 7684
Jumlah query: 100
Annotation ID duplikat: 0
Label kosong: 0
Label bukan angka: 0
Label di luar 0, 1, 2: []
Pasangan query-buku tidak konsisten: 0

Distribusi label:
relevance_label
0    4400
1    2690
2     594
Name: count, dtype: int64
Anotator 1: seluruh hasil anotasi valid.

VALIDASI Anotator 2
Jumlah baris: 7684
Jumlah query: 100
Annotation ID duplikat: 0
Label kosong: 0
Label bukan angka: 0
Label di luar 0, 1, 2: []
Pasangan query-buku tidak konsisten: 0

Distribusi label:
relevance_label
0    4181
1    1909
2    1594
Name: count, dtype: int64
Anotator 2: seluruh hasil anotasi valid.


In [ ]:
# ============================================================
# VALIDATE MATCHING ANNOTATION PAIRS
# ============================================================

annotation_ids_1 = set(
    annotations_1["annotation_id"]
)

annotation_ids_2 = set(
    annotations_2["annotation_id"]
)

only_in_annotator_1 = (
    annotation_ids_1
    - annotation_ids_2
)

only_in_annotator_2 = (
    annotation_ids_2
    - annotation_ids_1
)

print(
    "Annotation ID hanya di Anotator 1:",
    len(only_in_annotator_1)
)

print(
    "Annotation ID hanya di Anotator 2:",
    len(only_in_annotator_2)
)

print(
    "Pasangan anotasi sama:",
    annotation_ids_1
    == annotation_ids_2
)

Annotation ID hanya di Anotator 1: 0
Annotation ID hanya di Anotator 2: 0
Pasangan anotasi sama: True


In [ ]:
# ============================================================
# MERGE TWO ANNOTATORS
# ============================================================

metadata_columns = [
    column
    for column in [
        "annotation_id",
        "query_id",
        "information_need",
        "query",
        "target_language",
        "book_id",
        "title",
        "genre_text",
        "description_full"
    ]
    if column in annotations_1.columns
]

annotator_1_selected = annotations_1[
    metadata_columns
    + ["relevance_label"]
].rename(
    columns={
        "relevance_label":
            "label_annotator_1"
    }
)

annotator_2_selected = annotations_2[
    [
        "annotation_id",
        "query_id",
        "book_id",
        "relevance_label"
    ]
].rename(
    columns={
        "relevance_label":
            "label_annotator_2"
    }
)

merged_annotations = pd.merge(
    annotator_1_selected,
    annotator_2_selected,
    on=[
        "annotation_id",
        "query_id",
        "book_id"
    ],
    how="inner",
    validate="one_to_one"
)

print(
    "Jumlah hasil setelah merge:",
    len(merged_annotations)
)

print(
    "Sama dengan jumlah Anotator 1:",
    len(merged_annotations)
    == len(annotations_1)
)

print(
    "Sama dengan jumlah Anotator 2:",
    len(merged_annotations)
    == len(annotations_2)
)

Jumlah hasil setelah merge: 7684
Sama dengan jumlah Anotator 1: True
Sama dengan jumlah Anotator 2: True


In [ ]:
# ============================================================
# EXACT AGREEMENT ANALYSIS
# ============================================================

merged_annotations["is_agreement"] = (
    merged_annotations[
        "label_annotator_1"
    ]
    == merged_annotations[
        "label_annotator_2"
    ]
)

agreement_count = (
    merged_annotations["is_agreement"]
    .sum()
)

disagreement_count = (
    ~merged_annotations["is_agreement"]
).sum()

agreement_percentage = (
    agreement_count
    / len(merged_annotations)
    * 100
)

print(
    "Jumlah label sama:",
    agreement_count
)

print(
    "Jumlah label berbeda:",
    disagreement_count
)

print(
    "Persentase kesepakatan langsung:",
    round(
        agreement_percentage,
        2
    ),
    "%"
)

Jumlah label sama: 4145
Jumlah label berbeda: 3539
Persentase kesepakatan langsung: 53.94 %


In [ ]:
# ============================================================
# COHEN'S KAPPA
# ============================================================

unweighted_kappa = cohen_kappa_score(
    merged_annotations["label_annotator_1"],
    merged_annotations["label_annotator_2"]
)

linear_weighted_kappa = cohen_kappa_score(
    merged_annotations["label_annotator_1"],
    merged_annotations["label_annotator_2"],
    weights="linear"
)

quadratic_weighted_kappa = cohen_kappa_score(
    merged_annotations["label_annotator_1"],
    merged_annotations["label_annotator_2"],
    weights="quadratic"
)

print(
    "Cohen's Kappa tanpa bobot:",
    round(unweighted_kappa, 4)
)

print(
    "Cohen's Kappa linear:",
    round(linear_weighted_kappa, 4)
)

print(
    "Cohen's Kappa quadratic:",
    round(quadratic_weighted_kappa, 4)
)

Cohen's Kappa tanpa bobot: 0.2133
Cohen's Kappa linear: 0.2653
Cohen's Kappa quadratic: 0.3223


## Manual Calculation of Cohen's Kappa

In [ ]:
# ============================================================
# MANUAL COHEN'S KAPPA CALCULATION
# STEP 1 - PREPARE LABELS
# ============================================================

import numpy as np
import pandas as pd


judge_1_labels = (
    merged_annotations[
        "label_annotator_1"
    ]
    .astype(int)
    .to_numpy()
)

judge_2_labels = (
    merged_annotations[
        "label_annotator_2"
    ]
    .astype(int)
    .to_numpy()
)


# Label relevance yang digunakan
relevance_labels = np.array(
    [0, 1, 2],
    dtype=int
)


number_of_items = len(
    judge_1_labels
)


print("=" * 80)
print("MANUAL COHEN'S KAPPA CALCULATION")
print("=" * 80)

print(
    "Jumlah pasangan penilaian:",
    number_of_items
)

print(
    "Label relevansi:",
    relevance_labels.tolist()
)

print(
    "\nDistribusi awal label Judge 1:"
)

print(
    pd.Series(
        judge_1_labels
    )
    .value_counts()
    .sort_index()
)

print(
    "\nDistribusi awal label Judge 2:"
)

print(
    pd.Series(
        judge_2_labels
    )
    .value_counts()
    .sort_index()
)

MANUAL COHEN'S KAPPA CALCULATION
Jumlah pasangan penilaian: 7684
Label relevansi: [0, 1, 2]

Distribusi awal label Judge 1:
0    4400
1    2690
2     594
Name: count, dtype: int64

Distribusi awal label Judge 2:
0    4181
1    1909
2    1594
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 2 - BUILD OBSERVED CONFUSION MATRIX MANUALLY
# ============================================================

number_of_labels = len(
    relevance_labels
)

observed_matrix = np.zeros(
    (
        number_of_labels,
        number_of_labels
    ),
    dtype=int
)


for label_judge_1, label_judge_2 in zip(
    judge_1_labels,
    judge_2_labels
):
    row_index = np.where(
        relevance_labels
        == label_judge_1
    )[0][0]

    column_index = np.where(
        relevance_labels
        == label_judge_2
    )[0][0]

    observed_matrix[
        row_index,
        column_index
    ] += 1


observed_dataframe = pd.DataFrame(
    observed_matrix,
    index=[
        f"Judge 1 = {label}"
        for label in relevance_labels
    ],
    columns=[
        f"Judge 2 = {label}"
        for label in relevance_labels
    ]
)


print("\n" + "=" * 80)
print("STEP 2 - OBSERVED CONFUSION MATRIX")
print("=" * 80)

print(
    observed_dataframe
)


STEP 2 - OBSERVED CONFUSION MATRIX
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0         2983          881          536
Judge 1 = 1         1060          867          763
Judge 1 = 2          138          161          295


In [ ]:
# ============================================================
# STEP 3 - OBSERVED AGREEMENT
# ============================================================

observed_agreement_count = np.trace(
    observed_matrix
)

observed_agreement = (
    observed_agreement_count
    / number_of_items
)


print("\n" + "=" * 80)
print("STEP 3 - OBSERVED AGREEMENT")
print("=" * 80)

print(
    "Jumlah agreement diagonal:"
)

print(
    f"{observed_matrix[0, 0]} "
    f"+ {observed_matrix[1, 1]} "
    f"+ {observed_matrix[2, 2]} "
    f"= {observed_agreement_count}"
)

print()

print(
    "Observed Agreement (Po)"
)

print(
    f"Po = {observed_agreement_count} "
    f"/ {number_of_items}"
)

print(
    f"Po = {observed_agreement:.6f}"
)

print(
    f"Po = "
    f"{observed_agreement * 100:.2f}%"
)


STEP 3 - OBSERVED AGREEMENT
Jumlah agreement diagonal:
2983 + 867 + 295 = 4145

Observed Agreement (Po)
Po = 4145 / 7684
Po = 0.539433
Po = 53.94%


In [ ]:
# ============================================================
# STEP 4 - MARGINAL DISTRIBUTIONS
# ============================================================

judge_1_marginal = observed_matrix.sum(
    axis=1
)

judge_2_marginal = observed_matrix.sum(
    axis=0
)


marginal_dataframe = pd.DataFrame({
    "Label": relevance_labels,

    "Judge_1_Count":
        judge_1_marginal,

    "Judge_1_Probability":
        judge_1_marginal
        / number_of_items,

    "Judge_2_Count":
        judge_2_marginal,

    "Judge_2_Probability":
        judge_2_marginal
        / number_of_items
})


print("\n" + "=" * 80)
print("STEP 4 - MARGINAL DISTRIBUTIONS")
print("=" * 80)

print(
    marginal_dataframe
    .round(6)
    .to_string(index=False)
)


STEP 4 - MARGINAL DISTRIBUTIONS
 Label  Judge_1_Count  Judge_1_Probability  Judge_2_Count  Judge_2_Probability
     0           4400                 0.57           4181                 0.54
     1           2690                 0.35           1909                 0.25
     2            594                 0.08           1594                 0.21


In [ ]:
# ============================================================
# STEP 5 - EXPECTED AGREEMENT
# ============================================================

judge_1_probability = (
    judge_1_marginal
    / number_of_items
)

judge_2_probability = (
    judge_2_marginal
    / number_of_items
)


expected_agreement_per_label = (
    judge_1_probability
    * judge_2_probability
)

expected_agreement = (
    expected_agreement_per_label.sum()
)


expected_agreement_dataframe = (
    pd.DataFrame({
        "Label":
            relevance_labels,

        "P_Judge_1":
            judge_1_probability,

        "P_Judge_2":
            judge_2_probability,

        "P_Judge_1_x_P_Judge_2":
            expected_agreement_per_label
    })
)


print("\n" + "=" * 80)
print("STEP 5 - EXPECTED AGREEMENT")
print("=" * 80)

print(
    expected_agreement_dataframe
    .round(6)
    .to_string(index=False)
)

print()

print(
    "Expected Agreement (Pe)"
)

print(
    "Pe =",
    " + ".join(
        [
            f"{value:.6f}"
            for value
            in expected_agreement_per_label
        ]
    )
)

print(
    f"Pe = {expected_agreement:.6f}"
)


STEP 5 - EXPECTED AGREEMENT
 Label  P_Judge_1  P_Judge_2  P_Judge_1_x_P_Judge_2
     0       0.57       0.54                   0.31
     1       0.35       0.25                   0.09
     2       0.08       0.21                   0.02

Expected Agreement (Pe)
Pe = 0.311572 + 0.086973 + 0.016036
Pe = 0.414581


In [ ]:
# ============================================================
# STEP 6 - UNWEIGHTED COHEN'S KAPPA
# ============================================================

manual_unweighted_kappa = (
    (
        observed_agreement
        - expected_agreement
    )
    /
    (
        1
        - expected_agreement
    )
)


print("\n" + "=" * 80)
print("STEP 6 - UNWEIGHTED COHEN'S KAPPA")
print("=" * 80)

print(
    "Formula:"
)

print(
    "Kappa = (Po - Pe) / (1 - Pe)"
)

print()

print(
    f"Kappa = "
    f"({observed_agreement:.6f} "
    f"- {expected_agreement:.6f}) "
    f"/ "
    f"(1 - {expected_agreement:.6f})"
)

print(
    f"Kappa = "
    f"{manual_unweighted_kappa:.6f}"
)


STEP 6 - UNWEIGHTED COHEN'S KAPPA
Formula:
Kappa = (Po - Pe) / (1 - Pe)

Kappa = (0.539433 - 0.414581) / (1 - 0.414581)
Kappa = 0.213269


In [ ]:
# ============================================================
# STEP 7 - BUILD WEIGHT MATRICES
# ============================================================

linear_weight_matrix = np.zeros(
    (
        number_of_labels,
        number_of_labels
    ),
    dtype=float
)

quadratic_weight_matrix = np.zeros(
    (
        number_of_labels,
        number_of_labels
    ),
    dtype=float
)


for i in range(
    number_of_labels
):
    for j in range(
        number_of_labels
    ):
        distance = abs(i - j)

        linear_weight_matrix[
            i,
            j
        ] = (
            distance
            / (
                number_of_labels
                - 1
            )
        )

        quadratic_weight_matrix[
            i,
            j
        ] = (
            distance ** 2
            / (
                (
                    number_of_labels
                    - 1
                )
                ** 2
            )
        )


linear_weight_dataframe = pd.DataFrame(
    linear_weight_matrix,
    index=[
        f"Judge 1 = {label}"
        for label in relevance_labels
    ],
    columns=[
        f"Judge 2 = {label}"
        for label in relevance_labels
    ]
)


quadratic_weight_dataframe = pd.DataFrame(
    quadratic_weight_matrix,
    index=[
        f"Judge 1 = {label}"
        for label in relevance_labels
    ],
    columns=[
        f"Judge 2 = {label}"
        for label in relevance_labels
    ]
)


print("\n" + "=" * 80)
print("STEP 7A - LINEAR WEIGHT MATRIX")
print("=" * 80)

print(
    linear_weight_dataframe
)

print("\n" + "=" * 80)
print("STEP 7B - QUADRATIC WEIGHT MATRIX")
print("=" * 80)

print(
    quadratic_weight_dataframe
)


STEP 7A - LINEAR WEIGHT MATRIX
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0         0.00         0.50         1.00
Judge 1 = 1         0.50         0.00         0.50
Judge 1 = 2         1.00         0.50         0.00

STEP 7B - QUADRATIC WEIGHT MATRIX
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0         0.00         0.25         1.00
Judge 1 = 1         0.25         0.00         0.25
Judge 1 = 2         1.00         0.25         0.00


In [ ]:
# ============================================================
# STEP 8 - BUILD EXPECTED MATRIX
# ============================================================

expected_matrix = (
    np.outer(
        judge_1_marginal,
        judge_2_marginal
    )
    / number_of_items
)


expected_matrix_dataframe = pd.DataFrame(
    expected_matrix,
    index=[
        f"Judge 1 = {label}"
        for label in relevance_labels
    ],
    columns=[
        f"Judge 2 = {label}"
        for label in relevance_labels
    ]
)


print("\n" + "=" * 80)
print("STEP 8 - EXPECTED MATRIX")
print("=" * 80)

print(
    expected_matrix_dataframe
    .round(4)
)


STEP 8 - EXPECTED MATRIX
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0     2,394.12     1,093.13       912.75
Judge 1 = 1     1,463.68       668.30       558.02
Judge 1 = 2       323.21       147.57       123.22


In [ ]:
# ============================================================
# STEP 9 - LINEAR WEIGHTED KAPPA
# ============================================================

linear_observed_penalty_matrix = (
    linear_weight_matrix
    * observed_matrix
)

linear_expected_penalty_matrix = (
    linear_weight_matrix
    * expected_matrix
)


linear_observed_penalty = (
    linear_observed_penalty_matrix.sum()
)

linear_expected_penalty = (
    linear_expected_penalty_matrix.sum()
)


manual_linear_kappa = (
    1
    -
    (
        linear_observed_penalty
        / linear_expected_penalty
    )
)


print("\n" + "=" * 80)
print("STEP 9 - LINEAR WEIGHTED COHEN'S KAPPA")
print("=" * 80)

print(
    "\nObserved weighted disagreement:"
)

print(
    pd.DataFrame(
        linear_observed_penalty_matrix,
        index=[
            f"Judge 1 = {label}"
            for label in relevance_labels
        ],
        columns=[
            f"Judge 2 = {label}"
            for label in relevance_labels
        ]
    ).round(4)
)

print(
    "\nTotal observed weighted disagreement =",
    round(
        linear_observed_penalty,
        6
    )
)


print(
    "\nExpected weighted disagreement:"
)

print(
    pd.DataFrame(
        linear_expected_penalty_matrix,
        index=[
            f"Judge 1 = {label}"
            for label in relevance_labels
        ],
        columns=[
            f"Judge 2 = {label}"
            for label in relevance_labels
        ]
    ).round(4)
)

print(
    "\nTotal expected weighted disagreement =",
    round(
        linear_expected_penalty,
        6
    )
)


print("\nFormula:")

print(
    "Linear Kappa = "
    "1 - "
    "(Observed Weighted Disagreement / "
    "Expected Weighted Disagreement)"
)

print()

print(
    f"Linear Kappa = "
    f"1 - "
    f"({linear_observed_penalty:.6f} "
    f"/ {linear_expected_penalty:.6f})"
)

print(
    f"Linear Kappa = "
    f"{manual_linear_kappa:.6f}"
)


STEP 9 - LINEAR WEIGHTED COHEN'S KAPPA

Observed weighted disagreement:
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0         0.00       440.50       536.00
Judge 1 = 1       530.00         0.00       381.50
Judge 1 = 2       138.00        80.50         0.00

Total observed weighted disagreement = 2106.5

Expected weighted disagreement:
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0         0.00       546.56       912.75
Judge 1 = 1       731.84         0.00       279.01
Judge 1 = 2       323.21        73.79         0.00

Total expected weighted disagreement = 2867.160593

Formula:
Linear Kappa = 1 - (Observed Weighted Disagreement / Expected Weighted Disagreement)

Linear Kappa = 1 - (2106.500000 / 2867.160593)
Linear Kappa = 0.265301


In [ ]:
# ============================================================
# STEP 10 - QUADRATIC WEIGHTED KAPPA
# ============================================================

quadratic_observed_penalty_matrix = (
    quadratic_weight_matrix
    * observed_matrix
)

quadratic_expected_penalty_matrix = (
    quadratic_weight_matrix
    * expected_matrix
)


quadratic_observed_penalty = (
    quadratic_observed_penalty_matrix.sum()
)

quadratic_expected_penalty = (
    quadratic_expected_penalty_matrix.sum()
)


manual_quadratic_kappa = (
    1
    -
    (
        quadratic_observed_penalty
        / quadratic_expected_penalty
    )
)


print("\n" + "=" * 80)
print("STEP 10 - QUADRATIC WEIGHTED COHEN'S KAPPA")
print("=" * 80)

print(
    "\nObserved weighted disagreement:"
)

print(
    pd.DataFrame(
        quadratic_observed_penalty_matrix,
        index=[
            f"Judge 1 = {label}"
            for label in relevance_labels
        ],
        columns=[
            f"Judge 2 = {label}"
            for label in relevance_labels
        ]
    ).round(4)
)

print(
    "\nTotal observed weighted disagreement =",
    round(
        quadratic_observed_penalty,
        6
    )
)


print(
    "\nExpected weighted disagreement:"
)

print(
    pd.DataFrame(
        quadratic_expected_penalty_matrix,
        index=[
            f"Judge 1 = {label}"
            for label in relevance_labels
        ],
        columns=[
            f"Judge 2 = {label}"
            for label in relevance_labels
        ]
    ).round(4)
)

print(
    "\nTotal expected weighted disagreement =",
    round(
        quadratic_expected_penalty,
        6
    )
)


print("\nFormula:")

print(
    "Quadratic Kappa = "
    "1 - "
    "(Observed Weighted Disagreement / "
    "Expected Weighted Disagreement)"
)

print()

print(
    f"Quadratic Kappa = "
    f"1 - "
    f"({quadratic_observed_penalty:.6f} "
    f"/ {quadratic_expected_penalty:.6f})"
)

print(
    f"Quadratic Kappa = "
    f"{manual_quadratic_kappa:.6f}"
)


STEP 10 - QUADRATIC WEIGHTED COHEN'S KAPPA

Observed weighted disagreement:
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0         0.00       220.25       536.00
Judge 1 = 1       265.00         0.00       190.75
Judge 1 = 2       138.00        40.25         0.00

Total observed weighted disagreement = 1390.25

Expected weighted disagreement:
             Judge 2 = 0  Judge 2 = 1  Judge 2 = 2
Judge 1 = 0         0.00       273.28       912.75
Judge 1 = 1       365.92         0.00       139.51
Judge 1 = 2       323.21        36.89         0.00

Total expected weighted disagreement = 2051.560125

Formula:
Quadratic Kappa = 1 - (Observed Weighted Disagreement / Expected Weighted Disagreement)

Quadratic Kappa = 1 - (1390.250000 / 2051.560125)
Quadratic Kappa = 0.322345


In [ ]:
# ============================================================
# STEP 11 - COMPARE MANUAL VS SKLEARN
# ============================================================

kappa_comparison = pd.DataFrame({
    "Kappa_Type": [
        "Unweighted",
        "Linear Weighted",
        "Quadratic Weighted"
    ],

    "Manual": [
        manual_unweighted_kappa,
        manual_linear_kappa,
        manual_quadratic_kappa
    ],

    "Scikit_Learn": [
        unweighted_kappa,
        linear_weighted_kappa,
        quadratic_weighted_kappa
    ]
})


kappa_comparison[
    "Absolute_Difference"
] = np.abs(
    kappa_comparison["Manual"]
    - kappa_comparison["Scikit_Learn"]
)


print("\n" + "=" * 80)
print("VALIDATION: MANUAL VS SCIKIT-LEARN")
print("=" * 80)

print(
    kappa_comparison
    .round(8)
    .to_string(index=False)
)


VALIDATION: MANUAL VS SCIKIT-LEARN
        Kappa_Type  Manual  Scikit_Learn  Absolute_Difference
        Unweighted    0.21          0.21                 0.00
   Linear Weighted    0.27          0.27                 0.00
Quadratic Weighted    0.32          0.32                 0.00


In [ ]:
# ============================================================
# STEP 12 - AUTOMATIC VALIDATION
# ============================================================

tolerance = 1e-10


assert np.isclose(
    manual_unweighted_kappa,
    unweighted_kappa,
    atol=tolerance
), (
    "Manual unweighted Kappa "
    "tidak sama dengan sklearn."
)


assert np.isclose(
    manual_linear_kappa,
    linear_weighted_kappa,
    atol=tolerance
), (
    "Manual linear weighted Kappa "
    "tidak sama dengan sklearn."
)


assert np.isclose(
    manual_quadratic_kappa,
    quadratic_weighted_kappa,
    atol=tolerance
), (
    "Manual quadratic weighted Kappa "
    "tidak sama dengan sklearn."
)


print(
    "\nVALIDATION PASSED:"
)

print(
    "Semua perhitungan manual "
    "sesuai dengan scikit-learn."
)


VALIDATION PASSED:
Semua perhitungan manual sesuai dengan scikit-learn.


In [ ]:
# ============================================================
# FINAL MANUAL KAPPA SUMMARY
# ============================================================

manual_kappa_summary = pd.DataFrame({
    "Pengukuran": [
        "Jumlah Pasangan Penilaian",
        "Jumlah Exact Agreement",
        "Observed Agreement (Po)",
        "Expected Agreement (Pe)",
        "Unweighted Cohen's Kappa",
        "Linear Weighted Kappa",
        "Quadratic Weighted Kappa"
    ],

    "Nilai": [
        number_of_items,
        observed_agreement_count,
        observed_agreement,
        expected_agreement,
        manual_unweighted_kappa,
        manual_linear_kappa,
        manual_quadratic_kappa
    ]
})


print("=" * 80)
print("RINGKASAN PERHITUNGAN COHEN'S KAPPA")
print("=" * 80)

print(
    manual_kappa_summary
    .round(6)
    .to_string(index=False)
)

RINGKASAN PERHITUNGAN COHEN'S KAPPA
               Pengukuran    Nilai
Jumlah Pasangan Penilaian 7,684.00
   Jumlah Exact Agreement 4,145.00
  Observed Agreement (Po)     0.54
  Expected Agreement (Pe)     0.41
 Unweighted Cohen's Kappa     0.21
    Linear Weighted Kappa     0.27
 Quadratic Weighted Kappa     0.32


In [ ]:
import numpy as np
import pandas as pd

def manual_cohen_kappa(y1, y2, weights=None):
    """
    Fungsi manual untuk menghitung Cohen's Kappa
    (Unweighted, Linear, dan Quadratic)
    """
    # 1. Cari kategori unik dan urutkan
    # Pengurutan ini penting agar indeks (0, 1, 2, dst) konsisten
    # untuk menghitung jarak pada bobot linear/quadratic.
    labels = sorted(list(set(y1) | set(y2)))
    k = len(labels)
    label_to_idx = {label: i for i, label in enumerate(labels)}

    # 2. Buat matriks observasi (Observed Matrix / Confusion Matrix)
    # Menghitung frekuensi aktual dari jawaban Annotator 1 dan 2
    O = np.zeros((k, k))
    for a, b in zip(y1, y2):
        O[label_to_idx[a], label_to_idx[b]] += 1

    N = len(y1) # Total observasi

    # 3. Buat matriks ekspektasi kebetulan (Expected Matrix)
    # Peluang kedua anotator menjawab secara acak
    row_sums = O.sum(axis=1) # Total jawaban tiap kategori oleh Annotator 1
    col_sums = O.sum(axis=0) # Total jawaban tiap kategori oleh Annotator 2

    # Rumus ekspektasi: (Total Baris * Total Kolom) / Total Data
    E = np.outer(row_sums, col_sums) / N

    # 4. Buat matriks bobot ketidaksepakatan (Weight Matrix)
    W = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            jarak = abs(i - j)

            if weights is None or weights == 'none':
                # Tanpa bobot: Sepakat penalti 0, tidak sepakat penalti 1 (berapapun jaraknya)
                W[i, j] = 0 if i == j else 1
            elif weights == 'linear':
                # Bobot linear: Penalti sama dengan jarak selisih kategori
                W[i, j] = jarak
            elif weights == 'quadratic':
                # Bobot kuadratik: Penalti adalah kuadrat dari jarak
                # (Sangat menghukum kesalahan yang ekstrim)
                W[i, j] = jarak ** 2

    # 5. Hitung total ketidaksepakatan observasi dan ekspektasi
    obs_disagreement = np.sum(W * O)
    exp_disagreement = np.sum(W * E)

    # 6. Hitung Kappa
    if exp_disagreement == 0:
        return 1.0 # Hindari pembagian dengan nol

    kappa = 1 - (obs_disagreement / exp_disagreement)
    return kappa


# ============================================================
# PENERAPAN PADA DATA ANDA
# ============================================================
y1 = merged_annotations["label_annotator_1"]
y2 = merged_annotations["label_annotator_2"]

manual_unweighted = manual_cohen_kappa(y1, y2, weights='none')
manual_linear = manual_cohen_kappa(y1, y2, weights='linear')
manual_quadratic = manual_cohen_kappa(y1, y2, weights='quadratic')

print("--- HASIL PERHITUNGAN MANUAL ---")
print("Cohen's Kappa tanpa bobot :", round(manual_unweighted, 4))
print("Cohen's Kappa linear      :", round(manual_linear, 4))
print("Cohen's Kappa quadratic   :", round(manual_quadratic, 4))

--- HASIL PERHITUNGAN MANUAL ---
Cohen's Kappa tanpa bobot : 0.2133
Cohen's Kappa linear      : 0.2653
Cohen's Kappa quadratic   : 0.3223


In [ ]:
# ============================================================
# ANNOTATOR CONFUSION MATRIX
# ============================================================

annotator_confusion_matrix = pd.crosstab(
    merged_annotations[
        "label_annotator_1"
    ],
    merged_annotations[
        "label_annotator_2"
    ],
    rownames=["Anotator 1"],
    colnames=["Anotator 2"],
    dropna=False
).reindex(
    index=[0, 1, 2],
    columns=[0, 1, 2],
    fill_value=0
)

print(
    annotator_confusion_matrix
)

Anotator 2     0    1    2
Anotator 1                
0           2983  881  536
1           1060  867  763
2            138  161  295


In [ ]:
# ============================================================
# DISAGREEMENT TYPE ANALYSIS
# ============================================================

disagreement_data = merged_annotations[
    ~merged_annotations["is_agreement"]
].copy()

disagreement_data["label_difference"] = (
    disagreement_data[
        "label_annotator_1"
    ]
    - disagreement_data[
        "label_annotator_2"
    ]
).abs()

print(
    "Distribusi besar selisih label:"
)

print(
    disagreement_data[
        "label_difference"
    ]
    .value_counts()
    .sort_index()
)

Distribusi besar selisih label:
label_difference
1    2865
2     674
Name: count, dtype: int64


In [ ]:
# ============================================================
# AGREED GROUND TRUTH
# ============================================================

agreed_annotations = merged_annotations[
    merged_annotations["is_agreement"]
].copy()

agreed_annotations["ground_truth_label"] = (
    agreed_annotations[
        "label_annotator_1"
    ]
)

print(
    "Jumlah ground truth langsung:",
    len(agreed_annotations)
)

Jumlah ground truth langsung: 4145


## Pembuatan File Adjudikasi

In [ ]:
# ============================================================
# CREATE ADJUDICATION TEMPLATE
# ============================================================

adjudication_columns = [
    column
    for column in [
        "annotation_id",
        "query_id",
        "information_need",
        "query",
        "target_language",
        "book_id",
        "title",
        "genre_text",
        "description_full",
        "label_annotator_1",
        "label_annotator_2",
        "label_difference"
    ]
    if column in disagreement_data.columns
]

adjudication_template = disagreement_data[
    adjudication_columns
].copy()

adjudication_template[
    "adjudicated_label"
] = np.nan

adjudication_template[
    "adjudication_note"
] = ""

adjudication_template.to_csv(
    "/content/drive/MyDrive/"
    "ground_truth_adjudication_template.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Jumlah pasangan yang perlu adjudikasi:",
    len(adjudication_template)
)

print(
    "Template adjudikasi berhasil disimpan."
)

Jumlah pasangan yang perlu adjudikasi: 3539
Template adjudikasi berhasil disimpan.


In [ ]:
# ============================================================
# SAVE AGREED ANNOTATIONS
# ============================================================

agreed_annotations.to_csv(
    "/content/drive/MyDrive/"
    "ground_truth_agreed_annotations.csv",
    index=False,
    encoding="utf-8-sig"
)

merged_annotations.to_csv(
    "/content/drive/MyDrive/"
    "merged_two_annotators.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Hasil merge dan label yang sepakat berhasil disimpan."
)

Hasil merge dan label yang sepakat berhasil disimpan.


## Pengecekan Hasil Adjudikasi

In [ ]:
# ============================================================
# LOAD COMPLETED ADJUDICATION
# ============================================================

adjudicated_annotations = pd.read_csv(
    "/content/drive/MyDrive/"
    "ground_truth_adjudication_filled.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# VALIDATE ADJUDICATED LABELS
# ============================================================

adjudicated_annotations[
    "adjudicated_label"
] = pd.to_numeric(
    adjudicated_annotations[
        "adjudicated_label"
    ],
    errors="coerce"
)

missing_adjudicated_labels = (
    adjudicated_annotations[
        "adjudicated_label"
    ]
    .isna()
    .sum()
)

invalid_adjudicated_labels = (
    adjudicated_annotations.loc[
        adjudicated_annotations[
            "adjudicated_label"
        ].notna()
        & ~adjudicated_annotations[
            "adjudicated_label"
        ].isin([0, 1, 2]),
        "adjudicated_label"
    ]
    .unique()
)

print(
    "Label adjudikasi kosong:",
    missing_adjudicated_labels
)

print(
    "Label adjudikasi tidak valid:",
    invalid_adjudicated_labels
)

Label adjudikasi kosong: 0
Label adjudikasi tidak valid: []


In [ ]:
# ============================================================
# BUILD FINAL GROUND TRUTH
# ============================================================

ground_truth_agreed = agreed_annotations.copy()

ground_truth_agreed[
    "ground_truth_source"
] = "annotator_agreement"

ground_truth_agreed = (
    ground_truth_agreed.rename(
        columns={
            "ground_truth_label":
                "relevance_label"
        }
    )
)

ground_truth_adjudicated = (
    adjudicated_annotations.copy()
)

ground_truth_adjudicated[
    "relevance_label"
] = (
    ground_truth_adjudicated[
        "adjudicated_label"
    ]
    .astype(int)
)

ground_truth_adjudicated[
    "ground_truth_source"
] = "adjudication"

In [ ]:
final_ground_truth_columns = [
    "annotation_id",
    "query_id",
    "book_id",
    "relevance_label",
    "ground_truth_source"
]

final_ground_truth = pd.concat(
    [
        ground_truth_agreed[
            final_ground_truth_columns
        ],
        ground_truth_adjudicated[
            final_ground_truth_columns
        ]
    ],
    ignore_index=True
)

final_ground_truth = (
    final_ground_truth
    .sort_values(
        [
            "query_id",
            "annotation_id"
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# FINAL GROUND TRUTH VALIDATION
# ============================================================

print(
    "Jumlah ground truth final:",
    len(final_ground_truth)
)

print(
    "Jumlah annotation_id unik:",
    final_ground_truth[
        "annotation_id"
    ]
    .nunique()
)

print(
    "Annotation ID duplikat:",
    final_ground_truth[
        "annotation_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Label kosong:",
    final_ground_truth[
        "relevance_label"
    ]
    .isna()
    .sum()
)

print(
    "Jumlah query:",
    final_ground_truth[
        "query_id"
    ]
    .nunique()
)

print("\nDistribusi label ground truth:")
print(
    final_ground_truth[
        "relevance_label"
    ]
    .value_counts()
    .sort_index()
)

print("\nSumber ground truth:")
print(
    final_ground_truth[
        "ground_truth_source"
    ]
    .value_counts()
)

Jumlah ground truth final: 7684
Jumlah annotation_id unik: 7684
Annotation ID duplikat: 0
Label kosong: 0
Jumlah query: 100

Distribusi label ground truth:
relevance_label
0    4231
1    1820
2    1633
Name: count, dtype: int64

Sumber ground truth:
ground_truth_source
annotator_agreement    4145
adjudication           3539
Name: count, dtype: int64


In [ ]:
# ============================================================
# SAVE FINAL GROUND TRUTH
# ============================================================

final_ground_truth.to_csv(
    "/content/drive/MyDrive/"
    "final_relevance_ground_truth.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Ground truth final berhasil disimpan."
)

Ground truth final berhasil disimpan.


In [ ]:
# ============================================================
# FINAL GROUND TRUTH VALIDATION
# ============================================================
# Ground truth harus memuat satu label final untuk setiap
# pasangan query dan buku dalam annotation pool.

required_ground_truth_columns = [
    "query_id",
    "book_id",
    "relevance_label"
]

missing_ground_truth_columns = [
    column
    for column in required_ground_truth_columns
    if column not in final_ground_truth.columns
]

if missing_ground_truth_columns:
    raise ValueError(
        "Kolom ground truth tidak ditemukan: "
        f"{missing_ground_truth_columns}"
    )

final_ground_truth = final_ground_truth.copy()

final_ground_truth["query_id"] = (
    final_ground_truth["query_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

final_ground_truth["book_id"] = (
    final_ground_truth["book_id"]
    .astype(str)
    .str.strip()
)

final_ground_truth["relevance_label"] = pd.to_numeric(
    final_ground_truth["relevance_label"],
    errors="coerce"
)

print(
    "Jumlah ground truth:",
    len(final_ground_truth)
)

print(
    "Jumlah query:",
    final_ground_truth["query_id"].nunique()
)

print(
    "Pasangan query-buku duplikat:",
    final_ground_truth.duplicated(
        subset=["query_id", "book_id"]
    ).sum()
)

print(
    "Label kosong:",
    final_ground_truth["relevance_label"]
    .isna()
    .sum()
)

print(
    "Label tidak valid:",
    final_ground_truth.loc[
        ~final_ground_truth["relevance_label"].isin(
            [0, 1, 2]
        ),
        "relevance_label"
    ].unique()
)

Jumlah ground truth: 7684
Jumlah query: 100
Pasangan query-buku duplikat: 0
Label kosong: 0
Label tidak valid: []


In [ ]:
if final_ground_truth.duplicated(
    subset=["query_id", "book_id"]
).any():
    raise ValueError(
        "Ground truth memiliki pasangan query-buku duplikat."
    )

if final_ground_truth["relevance_label"].isna().any():
    raise ValueError(
        "Ground truth masih memiliki label kosong."
    )

if not final_ground_truth[
    "relevance_label"
].isin([0, 1, 2]).all():
    raise ValueError(
        "Ground truth hanya boleh berisi label 0, 1, atau 2."
    )

final_ground_truth["relevance_label"] = (
    final_ground_truth["relevance_label"]
    .astype(int)
)

print("Ground truth valid.")

Ground truth valid.


In [ ]:
# ============================================================
# EVALUATION DATA VALIDATION
# ============================================================

required_query_columns = [
    "query_id",
    "category",
    "query"
]

required_book_columns = [
    "book_id",
    "model_text"
]

missing_query_columns = [
    column
    for column in required_query_columns
    if column not in df_evaluation_queries.columns
]

missing_book_columns = [
    column
    for column in required_book_columns
    if column not in df_books_model.columns
]

if missing_query_columns:
    raise ValueError(
        f"Kolom query tidak ditemukan: {missing_query_columns}"
    )

if missing_book_columns:
    raise ValueError(
        f"Kolom buku tidak ditemukan: {missing_book_columns}"
    )

df_evaluation_queries = (
    df_evaluation_queries.copy()
)

df_evaluation_queries["query_id"] = (
    df_evaluation_queries["query_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_books_model = df_books_model.copy()

df_books_model["book_id"] = (
    df_books_model["book_id"]
    .astype(str)
    .str.strip()
)

print(
    "Jumlah query evaluasi:",
    len(df_evaluation_queries)
)

print(
    "Jumlah buku:",
    len(df_books_model)
)

print(
    "Query ID duplikat:",
    df_evaluation_queries["query_id"]
    .duplicated()
    .sum()
)

print(
    "Book ID duplikat:",
    df_books_model["book_id"]
    .duplicated()
    .sum()
)

Jumlah query evaluasi: 100
Jumlah buku: 4083
Query ID duplikat: 0
Book ID duplikat: 0


In [ ]:
# ============================================================
# LOAD EVALUATION MODELS
# ============================================================

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

minilm_model = SentenceTransformer(
    embedding_model_names["multilingual_minilm"],
    device=device
)

mpnet_model = SentenceTransformer(
    embedding_model_names["multilingual_mpnet"],
    device=device
)

reranker_model = CrossEncoder(
    reranker_model_name,
    device=device
)

print("Semua model evaluasi berhasil dimuat.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Semua model evaluasi berhasil dimuat.


In [ ]:
# ============================================================
# LOAD SAVED CORPUS EMBEDDINGS
# ============================================================

minilm_embeddings = np.load(
    "book_embeddings_multilingual_minilm.npy"
)

mpnet_embeddings = np.load(
    "book_embeddings_multilingual_mpnet.npy"
)

print(
    "Shape MiniLM:",
    minilm_embeddings.shape
)

print(
    "Shape MPNet:",
    mpnet_embeddings.shape
)

Shape MiniLM: (4083, 384)
Shape MPNet: (4083, 768)


In [ ]:
if len(minilm_embeddings) != len(df_books_model):
    raise ValueError(
        "Jumlah embedding MiniLM tidak sama dengan jumlah buku."
    )

if len(mpnet_embeddings) != len(df_books_model):
    raise ValueError(
        "Jumlah embedding MPNet tidak sama dengan jumlah buku."
    )

print("Embedding sesuai dengan dataset buku.")

Embedding sesuai dengan dataset buku.


In [ ]:
# ============================================================
# GENERATE RETRIEVAL AND RERANKING RESULTS
# ============================================================
# Untuk setiap query:
# 1. mengambil Top-50 melalui cosine similarity;
# 2. menyimpan ranking retrieval;
# 3. melakukan reranking seluruh Top-50;
# 4. menyimpan ranking Cross-Encoder.

from tqdm.auto import tqdm
import time


def generate_system_rankings(
    query_dataframe,
    embedding_model,
    corpus_embeddings,
    books_dataframe,
    reranker,
    model_key,
    top_k=50,
    reranker_batch_size=16
):
    retrieval_records = []
    reranking_records = []
    timing_records = []

    for row in tqdm(
        query_dataframe.itertuples(index=False),
        total=len(query_dataframe),
        desc=f"Evaluasi {model_key}"
    ):
        query_id = row.query_id
        query_text = row.query

        # ====================================================
        # RETRIEVAL
        # ====================================================

        retrieval_start = time.perf_counter()

        clean_query, candidates = retrieve_candidates(
            query=query_text,
            embedding_model=embedding_model,
            corpus_embeddings=corpus_embeddings,
            dataframe=books_dataframe,
            top_k=top_k
        )

        retrieval_time = (
            time.perf_counter()
            - retrieval_start
        )

        candidates = candidates.copy()

        candidates["query_id"] = query_id
        candidates["system_name"] = (
            f"{model_key}_RETRIEVAL"
        )

        retrieval_selected = candidates[
            [
                "query_id",
                "system_name",
                "book_id",
                "retrieval_rank",
                "retrieval_score"
            ]
        ].copy()

        retrieval_selected = (
            retrieval_selected.rename(
                columns={
                    "retrieval_rank": "rank",
                    "retrieval_score": "system_score"
                }
            )
        )

        retrieval_records.append(
            retrieval_selected
        )

        # ====================================================
        # CROSS-ENCODER RERANKING
        # ====================================================

        reranking_start = time.perf_counter()

        reranked = rerank_candidates(
            query=clean_query,
            candidates=candidates,
            reranker=reranker,
            top_n=top_k,
            batch_size=reranker_batch_size
        )

        reranking_time = (
            time.perf_counter()
            - reranking_start
        )

        reranked["query_id"] = query_id
        reranked["system_name"] = (
            f"{model_key}_RERANK"
        )

        reranking_selected = reranked[
            [
                "query_id",
                "system_name",
                "book_id",
                "final_rank",
                "reranker_score",
                "retrieval_rank",
                "retrieval_score"
            ]
        ].copy()

        reranking_selected = (
            reranking_selected.rename(
                columns={
                    "final_rank": "rank",
                    "reranker_score": "system_score"
                }
            )
        )

        reranking_records.append(
            reranking_selected
        )

        timing_records.append({
            "query_id": query_id,
            "model_key": model_key,
            "retrieval_time_seconds": retrieval_time,
            "reranking_time_seconds": reranking_time,
            "total_time_seconds": (
                retrieval_time + reranking_time
            )
        })

    retrieval_results = pd.concat(
        retrieval_records,
        ignore_index=True
    )

    reranking_results = pd.concat(
        reranking_records,
        ignore_index=True
    )

    timing_results = pd.DataFrame(
        timing_records
    )

    return (
        retrieval_results,
        reranking_results,
        timing_results
    )

In [ ]:
# ============================================================
# EVALUATE MINILM PIPELINE
# ============================================================

(
    minilm_retrieval_rankings,
    minilm_rerank_rankings,
    minilm_evaluation_timing
) = generate_system_rankings(
    query_dataframe=df_evaluation_queries,
    embedding_model=minilm_model,
    corpus_embeddings=minilm_embeddings,
    books_dataframe=df_books_model,
    reranker=reranker_model,
    model_key="MINILM",
    top_k=50,
    reranker_batch_size=16
)

print(
    "MiniLM retrieval:",
    minilm_retrieval_rankings.shape
)

print(
    "MiniLM reranking:",
    minilm_rerank_rankings.shape
)

Evaluasi MINILM:   0%|          | 0/100 [00:00<?, ?it/s]

MiniLM retrieval: (5000, 5)
MiniLM reranking: (5000, 7)


In [ ]:
# ============================================================
# EVALUATE MPNET PIPELINE
# ============================================================

(
    mpnet_retrieval_rankings,
    mpnet_rerank_rankings,
    mpnet_evaluation_timing
) = generate_system_rankings(
    query_dataframe=df_evaluation_queries,
    embedding_model=mpnet_model,
    corpus_embeddings=mpnet_embeddings,
    books_dataframe=df_books_model,
    reranker=reranker_model,
    model_key="MPNET",
    top_k=50,
    reranker_batch_size=16
)

print(
    "MPNet retrieval:",
    mpnet_retrieval_rankings.shape
)

print(
    "MPNet reranking:",
    mpnet_rerank_rankings.shape
)

Evaluasi MPNET:   0%|          | 0/100 [00:00<?, ?it/s]

MPNet retrieval: (5000, 5)
MPNet reranking: (5000, 7)


In [ ]:
# ============================================================
# COMBINE FOUR SYSTEM CONFIGURATIONS
# ============================================================

all_system_rankings = pd.concat(
    [
        minilm_retrieval_rankings,
        minilm_rerank_rankings,
        mpnet_retrieval_rankings,
        mpnet_rerank_rankings
    ],
    ignore_index=True
)

print(
    "Jumlah ranking keseluruhan:",
    len(all_system_rankings)
)

print("\nJumlah hasil per sistem:")
print(
    all_system_rankings["system_name"]
    .value_counts()
)

Jumlah ranking keseluruhan: 20000

Jumlah hasil per sistem:
system_name
MINILM_RETRIEVAL    5000
MINILM_RERANK       5000
MPNET_RETRIEVAL     5000
MPNET_RERANK        5000
Name: count, dtype: int64


In [ ]:
duplicate_ranking_pairs = (
    all_system_rankings
    .duplicated(
        subset=[
            "system_name",
            "query_id",
            "book_id"
        ]
    )
    .sum()
)

print(
    "Pasangan ranking duplikat:",
    duplicate_ranking_pairs
)

Pasangan ranking duplikat: 0


In [ ]:
# ============================================================
# MATCH RANKINGS WITH GROUND TRUTH
# ============================================================

evaluated_rankings = pd.merge(
    all_system_rankings,
    final_ground_truth[
        [
            "query_id",
            "book_id",
            "relevance_label"
        ]
    ],
    on=[
        "query_id",
        "book_id"
    ],
    how="left",
    validate="many_to_one"
)

missing_ground_truth_count = (
    evaluated_rankings["relevance_label"]
    .isna()
    .sum()
)

print(
    "Ranking tanpa ground truth:",
    missing_ground_truth_count
)

Ranking tanpa ground truth: 0


In [ ]:
if missing_ground_truth_count > 0:
    unmatched_rankings = evaluated_rankings[
        evaluated_rankings["relevance_label"]
        .isna()
    ]

    print(
        unmatched_rankings[
            [
                "system_name",
                "query_id",
                "book_id",
                "rank"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

    raise ValueError(
        "Terdapat ranking yang belum memiliki ground truth."
    )

evaluated_rankings["relevance_label"] = (
    evaluated_rankings["relevance_label"]
    .astype(int)
)

In [ ]:
# ============================================================
# TOTAL RELEVANT DOCUMENTS PER QUERY
# ============================================================
# relevance_label >= 1 dianggap relevan.
# Label asli 0–2 tetap digunakan untuk NDCG.

ground_truth_statistics = (
    final_ground_truth
    .groupby("query_id")
    .agg(
        total_judged=(
            "book_id",
            "count"
        ),
        total_relevant=(
            "relevance_label",
            lambda values: (
                values >= 1
            ).sum()
        ),
        total_highly_relevant=(
            "relevance_label",
            lambda values: (
                values == 2
            ).sum()
        )
    )
    .reset_index()
)

print(
    ground_truth_statistics
    .describe()
    .round(2)
)

       total_judged  total_relevant  total_highly_relevant
count        100.00          100.00                 100.00
mean          76.84           34.53                  16.33
std            5.50           15.46                  10.91
min           64.00            6.00                   3.00
25%           73.00           24.00                   8.00
50%           77.00           31.00                  15.00
75%           81.00           41.75                  20.00
max           92.00           75.00                  58.00


In [ ]:
# ============================================================
# FINAL EVALUATION METRICS
# Precision, Recall, AP/MAP, MRR, NDCG
# ============================================================

import numpy as np


def precision_at_k(
    ranked_labels,
    k,
    relevance_threshold=1
):
    """
    Precision@K:
    proporsi dokumen relevan pada K hasil teratas.

    Label 1 dan 2 dianggap relevan ketika
    relevance_threshold=1.
    """

    labels = np.asarray(
        ranked_labels,
        dtype=int
    )[:k]

    if k <= 0:
        return 0.0

    relevant_count = (
        labels >= relevance_threshold
    ).sum()

    return (
        relevant_count / k
    )


def recall_at_k(
    ranked_labels,
    total_relevant,
    k,
    relevance_threshold=1
):
    """
    Recall@K:
    proporsi seluruh dokumen relevan pada ground truth
    yang berhasil ditemukan pada Top-K.
    """

    if total_relevant == 0:
        return np.nan

    labels = np.asarray(
        ranked_labels,
        dtype=int
    )[:k]

    relevant_retrieved = (
        labels >= relevance_threshold
    ).sum()

    return (
        relevant_retrieved
        / total_relevant
    )


def average_precision_at_k(
    ranked_labels,
    total_relevant,
    k,
    relevance_threshold=1
):
    """
    Average Precision@K (AP@K).

    AP dihitung untuk satu query.
    MAP@K adalah rata-rata AP@K seluruh query.

    Binary relevance:
    label 1 atau 2 dianggap relevan.
    """

    if total_relevant == 0:
        return np.nan

    labels = np.asarray(
        ranked_labels,
        dtype=int
    )[:k]

    binary_relevance = (
        labels >= relevance_threshold
    ).astype(int)

    if len(binary_relevance) == 0:
        return 0.0

    cumulative_relevant = np.cumsum(
        binary_relevance
    )

    ranks = np.arange(
        1,
        len(binary_relevance) + 1
    )

    precision_each_rank = (
        cumulative_relevant / ranks
    )

    ap_contributions = (
        precision_each_rank
        * binary_relevance
    )

    # Truncated AP@K:
    # jumlah relevan maksimal yang mungkin ditemukan
    # pada K posisi.
    denominator = min(
        total_relevant,
        k
    )

    if denominator == 0:
        return np.nan

    return (
        ap_contributions.sum()
        / denominator
    )


def reciprocal_rank_at_k(
    ranked_labels,
    k,
    relevance_threshold=1
):
    """
    Reciprocal Rank@K:
    kebalikan posisi dokumen relevan pertama.
    """

    labels = np.asarray(
        ranked_labels,
        dtype=int
    )[:k]

    relevant_positions = np.where(
        labels >= relevance_threshold
    )[0]

    if len(relevant_positions) == 0:
        return 0.0

    first_relevant_rank = (
        relevant_positions[0] + 1
    )

    return (
        1.0 / first_relevant_rank
    )


def dcg_at_k(
    relevance_labels,
    k
):
    """
    Discounted Cumulative Gain.

    Menggunakan graded relevance 0, 1, 2.
    """

    labels = np.asarray(
        relevance_labels,
        dtype=float
    )[:k]

    if len(labels) == 0:
        return 0.0

    gains = (
        np.power(2, labels)
        - 1
    )

    discounts = np.log2(
        np.arange(
            2,
            len(labels) + 2
        )
    )

    return np.sum(
        gains / discounts
    )


def ndcg_at_k(
    ranked_labels,
    all_ground_truth_labels,
    k
):
    """
    NDCG@K:
    membandingkan DCG ranking sistem dengan
    ranking ideal dari seluruh pooled ground truth.
    """

    actual_dcg = dcg_at_k(
        relevance_labels=ranked_labels,
        k=k
    )

    ideal_labels = np.sort(
        np.asarray(
            all_ground_truth_labels,
            dtype=float
        )
    )[::-1]

    ideal_dcg = dcg_at_k(
        relevance_labels=ideal_labels,
        k=k
    )

    if ideal_dcg == 0:
        return np.nan

    return (
        actual_dcg / ideal_dcg
    )

In [ ]:
# ============================================================
# EVALUATE ONE SYSTEM PER QUERY
# Precision, Recall, AP, MRR, NDCG @5 AND @10
# ============================================================

def evaluate_system_per_query(
    system_rankings,
    ground_truth,
    query_metadata,
    system_name
):
    system_data = system_rankings[
        system_rankings["system_name"]
        == system_name
    ].copy()

    evaluation_records = []

    for query_id, query_ranking in (
        system_data.groupby("query_id")
    ):
        # ----------------------------------------------------
        # Urutkan ranking sistem
        # ----------------------------------------------------

        query_ranking = (
            query_ranking
            .sort_values("rank")
            .reset_index(drop=True)
        )

        ranked_labels = (
            query_ranking[
                "relevance_label"
            ]
            .astype(int)
            .tolist()
        )

        # ----------------------------------------------------
        # Ground truth untuk query
        # ----------------------------------------------------

        query_ground_truth = (
            ground_truth[
                ground_truth["query_id"]
                == query_id
            ]
        )

        all_ground_truth_labels = (
            query_ground_truth[
                "relevance_label"
            ]
            .astype(int)
            .tolist()
        )

        # Label 1 dan 2 dianggap relevant
        total_relevant = (
            query_ground_truth[
                "relevance_label"
            ]
            .ge(1)
            .sum()
        )

        evaluation_record = {
            "system_name": system_name,
            "query_id": query_id,
            "total_relevant_in_pool":
                total_relevant
        }

        # ====================================================
        # HITUNG METRIK @5 DAN @10
        # ====================================================

        for k in [5, 10]:

            evaluation_record[
                f"precision_at_{k}"
            ] = precision_at_k(
                ranked_labels=ranked_labels,
                k=k,
                relevance_threshold=1
            )

            evaluation_record[
                f"recall_at_{k}"
            ] = recall_at_k(
                ranked_labels=ranked_labels,
                total_relevant=total_relevant,
                k=k,
                relevance_threshold=1
            )

            # Ini AP per query.
            # Setelah dirata-rata seluruh query menjadi MAP.
            evaluation_record[
                f"average_precision_at_{k}"
            ] = average_precision_at_k(
                ranked_labels=ranked_labels,
                total_relevant=total_relevant,
                k=k,
                relevance_threshold=1
            )

            evaluation_record[
                f"mrr_at_{k}"
            ] = reciprocal_rank_at_k(
                ranked_labels=ranked_labels,
                k=k,
                relevance_threshold=1
            )

            evaluation_record[
                f"ndcg_at_{k}"
            ] = ndcg_at_k(
                ranked_labels=ranked_labels,
                all_ground_truth_labels=(
                    all_ground_truth_labels
                ),
                k=k
            )

        # Recall@50 tetap dipertahankan sebagai
        # diagnostic candidate-retrieval metric.
        evaluation_record[
            "pooled_recall_at_50"
        ] = recall_at_k(
            ranked_labels=ranked_labels,
            total_relevant=total_relevant,
            k=50,
            relevance_threshold=1
        )

        evaluation_records.append(
            evaluation_record
        )

    evaluation_dataframe = pd.DataFrame(
        evaluation_records
    )

    # ========================================================
    # ADD QUERY METADATA
    # ========================================================

    metadata_columns = [
        column
        for column in [
            "query_id",
            "category",
            "information_need_id",
            "formulation_id",
            "query_length",
            "query_group",
            "query_language",
            "target_language",
            "is_cross_lingual",
            "is_cross_category"
        ]
        if column in query_metadata.columns
    ]

    evaluation_dataframe = pd.merge(
        evaluation_dataframe,
        query_metadata[
            metadata_columns
        ],
        on="query_id",
        how="left",
        validate="one_to_one"
    )

    return evaluation_dataframe

In [ ]:
# ============================================================
# EVALUATE ALL FOUR CONFIGURATIONS
# ============================================================

system_names = [
    "MINILM_RETRIEVAL",
    "MINILM_RERANK",
    "MPNET_RETRIEVAL",
    "MPNET_RERANK"
]

per_query_evaluation_records = []

for system_name in system_names:
    system_evaluation = (
        evaluate_system_per_query(
            system_rankings=evaluated_rankings,
            ground_truth=final_ground_truth,
            query_metadata=df_evaluation_queries,
            system_name=system_name
        )
    )

    per_query_evaluation_records.append(
        system_evaluation
    )

per_query_evaluation = pd.concat(
    per_query_evaluation_records,
    ignore_index=True
)

print(
    "Jumlah evaluasi per query:",
    len(per_query_evaluation)
)

print(
    per_query_evaluation["system_name"]
    .value_counts()
)

Jumlah evaluasi per query: 400
system_name
MINILM_RETRIEVAL    100
MINILM_RERANK       100
MPNET_RETRIEVAL     100
MPNET_RERANK        100
Name: count, dtype: int64


In [ ]:
# ============================================================
# OVERALL EVALUATION SUMMARY
# ============================================================

metric_columns = [
    # Precision
    "precision_at_5",
    "precision_at_10",

    # Recall
    "recall_at_5",
    "recall_at_10",

    # AP → rata-rata menjadi MAP
    "average_precision_at_5",
    "average_precision_at_10",

    # MRR
    "mrr_at_5",
    "mrr_at_10",

    # NDCG
    "ndcg_at_5",
    "ndcg_at_10"
]

overall_evaluation_summary = (
    per_query_evaluation
    .groupby("system_name")[
        metric_columns
    ]
    .mean()
    .reset_index()
)

# Setelah rata-rata seluruh query,
# average_precision berubah makna menjadi MAP.
overall_evaluation_summary = (
    overall_evaluation_summary.rename(
        columns={
            "average_precision_at_5":
                "map_at_5",
            "average_precision_at_10":
                "map_at_10"
        }
    )
)

# Tambahkan pooled Recall@50 sebagai diagnostik terpisah.
pooled_recall_summary = (
    per_query_evaluation
    .groupby("system_name")[
        "pooled_recall_at_50"
    ]
    .mean()
    .reset_index()
)

overall_evaluation_summary = pd.merge(
    overall_evaluation_summary,
    pooled_recall_summary,
    on="system_name",
    how="left",
    validate="one_to_one"
)

numeric_columns = [
    column
    for column in (
        overall_evaluation_summary.columns
    )
    if column != "system_name"
]

overall_evaluation_summary[
    numeric_columns
] = (
    overall_evaluation_summary[
        numeric_columns
    ].round(4)
)

print(
    overall_evaluation_summary
    .to_string(index=False)
)

     system_name  precision_at_5  precision_at_10  recall_at_5  recall_at_10  map_at_5  map_at_10  mrr_at_5  mrr_at_10  ndcg_at_5  ndcg_at_10  pooled_recall_at_50
   MINILM_RERANK            0.72             0.65         0.11          0.20      0.65       0.55      0.83       0.83       0.56        0.53                 0.64
MINILM_RETRIEVAL            0.68             0.63         0.11          0.20      0.60       0.53      0.83       0.83       0.56        0.53                 0.64
    MPNET_RERANK            0.77             0.73         0.12          0.23      0.71       0.65      0.88       0.88       0.60        0.58                 0.78
 MPNET_RETRIEVAL            0.77             0.72         0.12          0.23      0.72       0.64      0.86       0.86       0.62        0.59                 0.78


In [ ]:
# ============================================================
# MANUAL METRIC CALCULATION FOR ONE QUERY
# ============================================================

manual_query_id = "Q001"
manual_system_name = "MPNET_RETRIEVAL"


# ============================================================
# 1. GET SYSTEM RANKING
# ============================================================

manual_ranking = (
    evaluated_rankings[
        (
            evaluated_rankings["query_id"]
            == manual_query_id
        )
        &
        (
            evaluated_rankings["system_name"]
            == manual_system_name
        )
    ]
    .sort_values("rank")
    .copy()
)


# ============================================================
# 2. ADD BOOK TITLE
# ============================================================

book_title_lookup = (
    df_books_model[
        [
            "book_id",
            "title"
        ]
    ]
    .copy()
)

book_title_lookup[
    "book_id"
] = (
    book_title_lookup[
        "book_id"
    ].astype(str)
)

manual_ranking[
    "book_id"
] = (
    manual_ranking[
        "book_id"
    ].astype(str)
)

manual_ranking = pd.merge(
    manual_ranking,
    book_title_lookup,
    on="book_id",
    how="left",
    validate="many_to_one"
)


# ============================================================
# 3. QUERY INFORMATION
# ============================================================

query_information = (
    df_evaluation_queries[
        df_evaluation_queries[
            "query_id"
        ]
        == manual_query_id
    ]
)

print("=" * 100)
print("MANUAL METRIC CALCULATION")
print("=" * 100)

print(
    "Query ID:",
    manual_query_id
)

print(
    "System:",
    manual_system_name
)

if not query_information.empty:
    print(
        "Query:",
        query_information.iloc[0][
            "query"
        ]
    )


# ============================================================
# 4. GROUND TRUTH INFORMATION
# ============================================================

query_ground_truth = (
    final_ground_truth[
        final_ground_truth["query_id"]
        == manual_query_id
    ]
    .copy()
)

all_ground_truth_labels = (
    query_ground_truth[
        "relevance_label"
    ]
    .astype(int)
    .to_numpy()
)

total_relevant = (
    all_ground_truth_labels >= 1
).sum()

print(
    "Total relevant dalam pooled ground truth:",
    total_relevant
)

print(
    "Aturan binary relevance: "
    "label 1 dan 2 = relevant"
)

print(
    "NDCG tetap menggunakan graded relevance 0/1/2."
)

MANUAL METRIC CALCULATION
Query ID: Q001
System: MPNET_RETRIEVAL
Query: Books about building self-confidence and overcoming self-doubt.
Total relevant dalam pooled ground truth: 24
Aturan binary relevance: label 1 dan 2 = relevant
NDCG tetap menggunakan graded relevance 0/1/2.


In [ ]:
# ============================================================
# EXPLAIN MANUAL METRIC CALCULATION
# ============================================================

def explain_manual_metrics(
    ranking_dataframe,
    all_ground_truth_labels,
    total_relevant,
    k
):
    top_k = (
        ranking_dataframe
        .head(k)
        .copy()
        .reset_index(drop=True)
    )

    labels = (
        top_k[
            "relevance_label"
        ]
        .astype(int)
        .to_numpy()
    )

    binary_relevance = (
        labels >= 1
    ).astype(int)

    ranks = np.arange(
        1,
        len(labels) + 1
    )

    cumulative_relevant = np.cumsum(
        binary_relevance
    )

    precision_each_rank = (
        cumulative_relevant / ranks
    )

    ap_contribution = (
        precision_each_rank
        * binary_relevance
    )

    # ========================================================
    # PRECISION
    # ========================================================

    relevant_at_k = (
        binary_relevance.sum()
    )

    precision_value = (
        relevant_at_k / k
    )

    # ========================================================
    # RECALL
    # ========================================================

    recall_value = (
        relevant_at_k / total_relevant
        if total_relevant > 0
        else np.nan
    )

    # ========================================================
    # AP@K
    # ========================================================

    ap_denominator = min(
        total_relevant,
        k
    )

    ap_value = (
        ap_contribution.sum()
        / ap_denominator
        if ap_denominator > 0
        else np.nan
    )

    # ========================================================
    # MRR
    # ========================================================

    relevant_positions = np.where(
        binary_relevance == 1
    )[0]

    if len(relevant_positions) > 0:
        first_relevant_rank = (
            relevant_positions[0] + 1
        )

        rr_value = (
            1.0 / first_relevant_rank
        )
    else:
        first_relevant_rank = None
        rr_value = 0.0

    # ========================================================
    # DCG
    # ========================================================

    gains = (
        np.power(2, labels)
        - 1
    )

    discounts = np.log2(
        ranks + 1
    )

    dcg_contribution = (
        gains / discounts
    )

    dcg_value = (
        dcg_contribution.sum()
    )

    # ========================================================
    # IDCG
    # ========================================================

    ideal_labels = np.sort(
        np.asarray(
            all_ground_truth_labels,
            dtype=int
        )
    )[::-1][:k]

    ideal_ranks = np.arange(
        1,
        len(ideal_labels) + 1
    )

    ideal_gains = (
        np.power(
            2,
            ideal_labels
        )
        - 1
    )

    ideal_discounts = np.log2(
        ideal_ranks + 1
    )

    ideal_contribution = (
        ideal_gains
        / ideal_discounts
    )

    idcg_value = (
        ideal_contribution.sum()
    )

    ndcg_value = (
        dcg_value / idcg_value
        if idcg_value > 0
        else np.nan
    )

    # ========================================================
    # DETAIL TABLE
    # ========================================================

    detail = pd.DataFrame({
        "Rank": ranks,
        "Title": (
            top_k["title"]
            .fillna("-")
            .tolist()
        ),
        "Label": labels,
        "Relevant(>=1)": (
            binary_relevance
        ),
        "Cumulative Relevant": (
            cumulative_relevant
        ),
        "Precision@Rank": (
            precision_each_rank
        ),
        "AP Contribution": (
            ap_contribution
        ),
        "Gain": gains,
        "Discount": discounts,
        "DCG Contribution": (
            dcg_contribution
        )
    })

    print("\n")
    print("=" * 110)
    print(
        f"PERHITUNGAN MANUAL METRIK @ {k}"
    )
    print("=" * 110)

    print(
        detail.round(4)
        .to_string(index=False)
    )

    # ========================================================
    # PRECISION EXPLANATION
    # ========================================================

    print("\n[1] PRECISION@" + str(k))

    print(
        f"Relevant pada Top-{k} "
        f"= {relevant_at_k}"
    )

    print(
        f"Precision@{k} "
        f"= {relevant_at_k} / {k}"
    )

    print(
        f"= {precision_value:.4f}"
    )

    # ========================================================
    # RECALL EXPLANATION
    # ========================================================

    print("\n[2] RECALL@" + str(k))

    print(
        f"Total relevant ground truth "
        f"= {total_relevant}"
    )

    print(
        f"Recall@{k} "
        f"= {relevant_at_k} / "
        f"{total_relevant}"
    )

    print(
        f"= {recall_value:.4f}"
    )

    # ========================================================
    # AP EXPLANATION
    # ========================================================

    print("\n[3] AVERAGE PRECISION@" + str(k))

    relevant_precision_values = (
        precision_each_rank[
            binary_relevance == 1
        ]
    )

    print(
        "Precision pada posisi relevan =",
        np.round(
            relevant_precision_values,
            4
        ).tolist()
    )

    print(
        "Jumlah kontribusi AP =",
        round(
            float(
                ap_contribution.sum()
            ),
            4
        )
    )

    print(
        f"Denominator "
        f"= min({total_relevant}, {k}) "
        f"= {ap_denominator}"
    )

    print(
        f"AP@{k} "
        f"= {ap_contribution.sum():.4f} "
        f"/ {ap_denominator}"
    )

    print(
        f"= {ap_value:.4f}"
    )

    # ========================================================
    # MRR EXPLANATION
    # ========================================================

    print("\n[4] RECIPROCAL RANK@" + str(k))

    if first_relevant_rank is not None:
        print(
            "Dokumen relevan pertama berada "
            f"pada rank {first_relevant_rank}"
        )

        print(
            f"RR@{k} "
            f"= 1 / {first_relevant_rank}"
        )

        print(
            f"= {rr_value:.4f}"
        )

    else:
        print(
            "Tidak ada dokumen relevan "
            f"pada Top-{k}."
        )

        print(
            f"RR@{k} = 0"
        )

    # ========================================================
    # NDCG EXPLANATION
    # ========================================================

    print("\n[5] NDCG@" + str(k))

    print(
        "Label aktual:",
        labels.tolist()
    )

    print(
        "Ideal label Top-K:",
        ideal_labels.tolist()
    )

    print(
        f"DCG@{k} "
        f"= {dcg_value:.4f}"
    )

    print(
        f"IDCG@{k} "
        f"= {idcg_value:.4f}"
    )

    print(
        f"NDCG@{k} "
        f"= {dcg_value:.4f} "
        f"/ {idcg_value:.4f}"
    )

    print(
        f"= {ndcg_value:.4f}"
    )

    return {
        "precision": precision_value,
        "recall": recall_value,
        "ap": ap_value,
        "rr": rr_value,
        "ndcg": ndcg_value
    }

In [ ]:
# ============================================================
# MANUAL CALCULATION @5 AND @10
# ============================================================

manual_metrics_at_5 = (
    explain_manual_metrics(
        ranking_dataframe=manual_ranking,
        all_ground_truth_labels=(
            all_ground_truth_labels
        ),
        total_relevant=total_relevant,
        k=5
    )
)

manual_metrics_at_10 = (
    explain_manual_metrics(
        ranking_dataframe=manual_ranking,
        all_ground_truth_labels=(
            all_ground_truth_labels
        ),
        total_relevant=total_relevant,
        k=10
    )
)



PERHITUNGAN MANUAL METRIK @ 5
 Rank                                                                                                   Title  Label  Relevant(>=1)  Cumulative Relevant  Precision@Rank  AP Contribution  Gain  Discount  DCG Contribution
    1 The Power of Self-Confidence: Become Unstoppable, Irresistible, and Unafraid in Every Area of Your Life      1              1                    1            1.00             1.00     1      1.00              1.00
    2      The Self-Esteem Workbook for Teens: Activities to Help You Build Confidence and Achieve Your Goals      1              1                    2            1.00             1.00     1      1.58              0.63
    3                                                                    Confidence: Finding It and Living It      1              1                    3            1.00             1.00     1      2.00              0.50
    4                                                                     الأسرار الكامل

In [ ]:
# ============================================================
# MAP = MEAN AP ACROSS ALL QUERIES
# ============================================================

system_query_results = (
    per_query_evaluation[
        per_query_evaluation[
            "system_name"
        ]
        == manual_system_name
    ]
)

map_at_5 = (
    system_query_results[
        "average_precision_at_5"
    ].mean()
)

map_at_10 = (
    system_query_results[
        "average_precision_at_10"
    ].mean()
)

print("\n" + "=" * 100)
print("MEAN AVERAGE PRECISION")
print("=" * 100)

print(
    f"MAP@5 = mean AP@5 "
    f"dari {len(system_query_results)} query"
)

print(
    f"MAP@5 = {map_at_5:.4f}"
)

print()

print(
    f"MAP@10 = mean AP@10 "
    f"dari {len(system_query_results)} query"
)

print(
    f"MAP@10 = {map_at_10:.4f}"
)


MEAN AVERAGE PRECISION
MAP@5 = mean AP@5 dari 100 query
MAP@5 = 0.7152

MAP@10 = mean AP@10 dari 100 query
MAP@10 = 0.6382


In [ ]:
# ============================================================
# ADD LATENCY SUMMARY
# ============================================================

all_timing_results = pd.concat(
    [
        minilm_evaluation_timing,
        mpnet_evaluation_timing
    ],
    ignore_index=True
)

timing_summary = (
    all_timing_results
    .groupby("model_key")
    .agg(
        mean_retrieval_time=(
            "retrieval_time_seconds",
            "mean"
        ),
        mean_reranking_time=(
            "reranking_time_seconds",
            "mean"
        ),
        mean_total_time=(
            "total_time_seconds",
            "mean"
        ),
        median_total_time=(
            "total_time_seconds",
            "median"
        )
    )
    .reset_index()
)

timing_summary[
    [
        "mean_retrieval_time",
        "mean_reranking_time",
        "mean_total_time",
        "median_total_time"
    ]
] = timing_summary[
    [
        "mean_retrieval_time",
        "mean_reranking_time",
        "mean_total_time",
        "median_total_time"
    ]
].round(4)

print(
    timing_summary.to_string(index=False)
)

model_key  mean_retrieval_time  mean_reranking_time  mean_total_time  median_total_time
   MINILM                 0.03                 0.35             0.38               0.37
    MPNET                 0.03                 0.35             0.38               0.38


In [ ]:
# ============================================================
# EVALUATION BY CATEGORY
# ============================================================

evaluation_by_category = (
    per_query_evaluation
    .groupby(
        [
            "system_name",
            "category"
        ]
    )[metric_columns]
    .mean()
    .reset_index()
)

evaluation_by_category[
    metric_columns
] = evaluation_by_category[
    metric_columns
].round(4)

print(
    evaluation_by_category
    .to_string(index=False)
)

     system_name           category  precision_at_5  precision_at_10  recall_at_5  recall_at_10  average_precision_at_5  average_precision_at_10  mrr_at_5  mrr_at_10  ndcg_at_5  ndcg_at_10
   MINILM_RERANK Career Development            0.60             0.58         0.10          0.19                    0.53                     0.48      0.74       0.75       0.49        0.47
   MINILM_RERANK       Productivity            0.68             0.57         0.11          0.18                    0.59                     0.46      0.86       0.86       0.56        0.48
   MINILM_RERANK         Psychology            0.81             0.72         0.11          0.19                    0.76                     0.65      0.95       0.95       0.66        0.61
   MINILM_RERANK   Self Development            0.68             0.67         0.10          0.19                    0.56                     0.53      0.71       0.72       0.45        0.47
   MINILM_RERANK         Technology            0.81    

In [ ]:
# ============================================================
# EVALUATION BY QUERY LENGTH
# ============================================================

evaluation_by_query_length = (
    per_query_evaluation
    .groupby(
        [
            "system_name",
            "query_length"
        ]
    )[metric_columns]
    .mean()
    .reset_index()
)

evaluation_by_query_length[
    metric_columns
] = evaluation_by_query_length[
    metric_columns
].round(4)

print(
    evaluation_by_query_length
    .to_string(index=False)
)

     system_name query_length  precision_at_5  precision_at_10  recall_at_5  recall_at_10  average_precision_at_5  average_precision_at_10  mrr_at_5  mrr_at_10  ndcg_at_5  ndcg_at_10
   MINILM_RERANK         long            0.68             0.61         0.11          0.19                    0.60                     0.51      0.77       0.78       0.53        0.48
   MINILM_RERANK        short            0.76             0.69         0.12          0.21                    0.69                     0.59      0.88       0.89       0.59        0.57
MINILM_RETRIEVAL         long            0.63             0.57         0.10          0.19                    0.56                     0.48      0.80       0.80       0.49        0.46
MINILM_RETRIEVAL        short            0.72             0.69         0.11          0.21                    0.65                     0.58      0.85       0.86       0.62        0.59
    MPNET_RERANK         long            0.74             0.73         0.12          

In [ ]:
# ============================================================
# EVALUATION BY CROSS-LINGUAL STATUS
# ============================================================

evaluation_by_cross_lingual = (
    per_query_evaluation
    .groupby(
        [
            "system_name",
            "is_cross_lingual"
        ]
    )[metric_columns]
    .mean()
    .reset_index()
)

evaluation_by_cross_lingual[
    metric_columns
] = evaluation_by_cross_lingual[
    metric_columns
].round(4)

print(
    evaluation_by_cross_lingual
    .to_string(index=False)
)

     system_name  is_cross_lingual  precision_at_5  precision_at_10  recall_at_5  recall_at_10  average_precision_at_5  average_precision_at_10  mrr_at_5  mrr_at_10  ndcg_at_5  ndcg_at_10
   MINILM_RERANK             False            0.73             0.66         0.12          0.21                    0.66                     0.57      0.84       0.85       0.58        0.55
   MINILM_RERANK              True            0.71             0.64         0.11          0.20                    0.64                     0.53      0.81       0.82       0.54        0.51
MINILM_RETRIEVAL             False            0.68             0.65         0.11          0.20                    0.61                     0.55      0.82       0.82       0.56        0.53
MINILM_RETRIEVAL              True            0.67             0.61         0.11          0.19                    0.60                     0.51      0.83       0.83       0.55        0.52
    MPNET_RERANK             False            0.78          

In [ ]:
# ============================================================
# EVALUATION BY QUERY GROUP
# ============================================================

evaluation_by_query_group = (
    per_query_evaluation
    .groupby(
        [
            "system_name",
            "query_group"
        ]
    )[metric_columns]
    .mean()
    .reset_index()
)

evaluation_by_query_group[
    metric_columns
] = evaluation_by_query_group[
    metric_columns
].round(4)

print(
    evaluation_by_query_group
    .to_string(index=False)
)

     system_name                   query_group  precision_at_5  precision_at_10  recall_at_5  recall_at_10  average_precision_at_5  average_precision_at_10  mrr_at_5  mrr_at_10  ndcg_at_5  ndcg_at_10
   MINILM_RERANK                Cross-Category            0.66             0.59         0.09          0.16                    0.59                     0.49      0.85       0.85       0.47        0.43
   MINILM_RERANK            English to English            0.77             0.67         0.12          0.21                    0.71                     0.60      0.87       0.87       0.62        0.58
   MINILM_RERANK        English to Non-English            0.90             0.75         0.10          0.17                    0.89                     0.70      1.00       1.00       0.65        0.59
   MINILM_RERANK         Indonesian to English            0.70             0.65         0.10          0.19                    0.62                     0.54      0.81       0.81       0.56        0.52


In [ ]:
# ============================================================
# RERANKING IMPROVEMENT ANALYSIS
# ============================================================

# Kolom yang benar-benar tersedia pada
# overall_evaluation_summary.
summary_metric_columns = [
    "precision_at_5",
    "precision_at_10",
    "recall_at_5",
    "recall_at_10",
    "map_at_5",
    "map_at_10",
    "mrr_at_5",
    "mrr_at_10",
    "ndcg_at_5",
    "ndcg_at_10",
    "pooled_recall_at_50"
]

# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

missing_summary_metrics = [
    metric
    for metric in summary_metric_columns
    if metric not in overall_evaluation_summary.columns
]

if missing_summary_metrics:
    raise ValueError(
        "Kolom metrik berikut tidak tersedia pada "
        "overall_evaluation_summary: "
        f"{missing_summary_metrics}"
    )

# ============================================================
# PREPARE SUMMARY TABLE
# ============================================================

retrieval_vs_rerank = (
    overall_evaluation_summary
    .set_index("system_name")
)

required_systems = [
    "MINILM_RETRIEVAL",
    "MINILM_RERANK",
    "MPNET_RETRIEVAL",
    "MPNET_RERANK"
]

missing_systems = [
    system_name
    for system_name in required_systems
    if system_name not in retrieval_vs_rerank.index
]

if missing_systems:
    raise ValueError(
        "Sistem berikut tidak tersedia pada summary: "
        f"{missing_systems}"
    )

# ============================================================
# CALCULATE RERANKING IMPROVEMENT
# ============================================================

reranking_improvement_records = []

for model_key in [
    "MINILM",
    "MPNET"
]:
    retrieval_name = (
        f"{model_key}_RETRIEVAL"
    )

    rerank_name = (
        f"{model_key}_RERANK"
    )

    improvement_record = {
        "model": model_key
    }

    for metric in summary_metric_columns:
        retrieval_score = float(
            retrieval_vs_rerank.loc[
                retrieval_name,
                metric
            ]
        )

        rerank_score = float(
            retrieval_vs_rerank.loc[
                rerank_name,
                metric
            ]
        )

        absolute_change = (
            rerank_score
            - retrieval_score
        )

        improvement_record[
            f"{metric}_retrieval"
        ] = retrieval_score

        improvement_record[
            f"{metric}_rerank"
        ] = rerank_score

        improvement_record[
            f"{metric}_absolute_change"
        ] = absolute_change

        if retrieval_score != 0:
            relative_change = (
                absolute_change
                / retrieval_score
                * 100
            )
        else:
            relative_change = np.nan

        improvement_record[
            f"{metric}_relative_change_percent"
        ] = relative_change

    reranking_improvement_records.append(
        improvement_record
    )

# ============================================================
# CREATE RESULT DATAFRAME
# ============================================================

reranking_improvement = pd.DataFrame(
    reranking_improvement_records
)

print(
    reranking_improvement
    .round(4)
    .to_string(index=False)
)

 model  precision_at_5_retrieval  precision_at_5_rerank  precision_at_5_absolute_change  precision_at_5_relative_change_percent  precision_at_10_retrieval  precision_at_10_rerank  precision_at_10_absolute_change  precision_at_10_relative_change_percent  recall_at_5_retrieval  recall_at_5_rerank  recall_at_5_absolute_change  recall_at_5_relative_change_percent  recall_at_10_retrieval  recall_at_10_rerank  recall_at_10_absolute_change  recall_at_10_relative_change_percent  map_at_5_retrieval  map_at_5_rerank  map_at_5_absolute_change  map_at_5_relative_change_percent  map_at_10_retrieval  map_at_10_rerank  map_at_10_absolute_change  map_at_10_relative_change_percent  mrr_at_5_retrieval  mrr_at_5_rerank  mrr_at_5_absolute_change  mrr_at_5_relative_change_percent  mrr_at_10_retrieval  mrr_at_10_rerank  mrr_at_10_absolute_change  mrr_at_10_relative_change_percent  ndcg_at_5_retrieval  ndcg_at_5_rerank  ndcg_at_5_absolute_change  ndcg_at_5_relative_change_percent  ndcg_at_10_retrieval  ndcg_

In [ ]:
# ============================================================
# SAVE EVALUATION RESULTS
# ============================================================

evaluation_output_folder = (
    "/content/drive/MyDrive/"
    "book_recommendation_evaluation"
)

os.makedirs(
    evaluation_output_folder,
    exist_ok=True
)

all_system_rankings.to_csv(
    os.path.join(
        evaluation_output_folder,
        "all_system_rankings.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

evaluated_rankings.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluated_rankings_with_ground_truth.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

per_query_evaluation.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluation_per_query.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

overall_evaluation_summary.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluation_overall_summary.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

evaluation_by_category.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluation_by_category.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

evaluation_by_query_length.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluation_by_query_length.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

evaluation_by_cross_lingual.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluation_by_cross_lingual.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

evaluation_by_query_group.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluation_by_query_group.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

timing_summary.to_csv(
    os.path.join(
        evaluation_output_folder,
        "evaluation_timing_summary.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

reranking_improvement.to_csv(
    os.path.join(
        evaluation_output_folder,
        "reranking_improvement.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

print(
    "Seluruh hasil evaluasi berhasil disimpan di:"
)

print(evaluation_output_folder)

Seluruh hasil evaluasi berhasil disimpan di:
/content/drive/MyDrive/book_recommendation_evaluation


In [ ]:
# ============================================================
# LOAD BEST MODEL FOR INFERENCE
# ============================================================

import time
import numpy as np
import pandas as pd
import torch

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
    util
)

device = "cuda" if torch.cuda.is_available() else "cpu"

best_embedding_model_name = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

optional_reranker_name = (
    "cross-encoder/"
    "mmarco-mMiniLMv2-L12-H384-v1"
)

best_embedding_model = SentenceTransformer(
    best_embedding_model_name,
    device=device
)

optional_reranker = CrossEncoder(
    optional_reranker_name,
    device=device
)

print("Device:", device)
print("Model utama:", best_embedding_model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: cuda
Model utama: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


In [ ]:
# Dataset harus memiliki urutan baris yang sama dengan saat
# corpus embedding dibuat.

df_books_model = pd.read_csv(
    "books_prepared.csv",
    low_memory=False
)

mpnet_embeddings = np.load(
    "book_embeddings_multilingual_mpnet.npy"
)

if len(df_books_model) != len(mpnet_embeddings):
    raise ValueError(
        "Jumlah buku tidak sesuai dengan jumlah embedding."
    )

print("Jumlah buku:", len(df_books_model))
print("Shape embedding:", mpnet_embeddings.shape)

Jumlah buku: 4083
Shape embedding: (4083, 768)


In [ ]:
# ============================================================
# AUTOMATIC BENCHMARK METRICS OF SELECTED MODEL
# ============================================================

best_model_name = (
    "MPNET_RETRIEVAL"
)

best_model_row = (
    overall_evaluation_summary[
        overall_evaluation_summary[
            "system_name"
        ]
        == best_model_name
    ]
)

if best_model_row.empty:
    raise ValueError(
        f"Benchmark {best_model_name} "
        "tidak ditemukan."
    )

best_model_metrics = (
    best_model_row
    .iloc[0]
    .to_dict()
)


print("=" * 80)
print("METRIK MODEL UTAMA")
print("=" * 80)

for metric_name, metric_value in (
    best_model_metrics.items()
):
    if metric_name == "system_name":
        print(
            "System:",
            metric_value
        )

    elif isinstance(
        metric_value,
        (int, float, np.number)
    ):
        print(
            f"{metric_name}: "
            f"{metric_value:.4f}"
        )

METRIK MODEL UTAMA
System: MPNET_RETRIEVAL
precision_at_5: 0.7660
precision_at_10: 0.7160
recall_at_5: 0.1223
recall_at_10: 0.2268
map_at_5: 0.7152
map_at_10: 0.6382
mrr_at_5: 0.8590
mrr_at_10: 0.8640
ndcg_at_5: 0.6207
ndcg_at_10: 0.5912
pooled_recall_at_50: 0.7771


In [ ]:
best_model_name = "MPNET_RETRIEVAL"

best_model_metrics = (
    overall_evaluation_summary[
        overall_evaluation_summary["system_name"]
        == best_model_name
    ]
    .iloc[0]
    .to_dict()
)

print(best_model_metrics)

{'system_name': 'MPNET_RETRIEVAL', 'precision_at_5': 0.766, 'precision_at_10': 0.716, 'recall_at_5': 0.1223, 'recall_at_10': 0.2268, 'map_at_5': 0.7152, 'map_at_10': 0.6382, 'mrr_at_5': 0.859, 'mrr_at_10': 0.864, 'ndcg_at_5': 0.6207, 'ndcg_at_10': 0.5912, 'pooled_recall_at_50': 0.7771}


In [ ]:
# ============================================================
# CLEAN USER QUERY
# ============================================================

def prepare_inference_query(query):
    query = clean_user_query(query)

    if not query:
        raise ValueError(
            "Query tidak boleh kosong."
        )

    return query

## Manual Sentence-BERT Inference Process

In [ ]:
# ============================================================
# MANUAL SENTENCE-BERT INFERENCE
# Tokenization → Transformer → Mean Pooling
# → Normalization → Cosine Similarity
# ============================================================

import torch
import numpy as np
import pandas as pd


manual_query = input(
    "Masukkan query untuk melihat alur Sentence-BERT: "
)

manual_query = prepare_inference_query(
    manual_query
)


print("=" * 100)
print("ALUR MANUAL SENTENCE-BERT")
print("=" * 100)

print(
    "\nTeks Input:"
)

print(
    manual_query
)


# ============================================================
# 1. ACCESS TRANSFORMER MODULE
# ============================================================

transformer_module = (
    best_embedding_model[0]
)

tokenizer = (
    transformer_module.tokenizer
)

transformer_model = (
    transformer_module.auto_model
)


# ============================================================
# 2. TOKENIZATION
# ============================================================

encoded_query = tokenizer(
    manual_query,
    padding=False,
    truncation=True,
    max_length=(
        best_embedding_model.max_seq_length
    ),
    return_tensors="pt"
)

encoded_query = {
    key: value.to(device)
    for key, value
    in encoded_query.items()
}

input_ids = (
    encoded_query["input_ids"][0]
)

attention_mask = (
    encoded_query[
        "attention_mask"
    ][0]
)

tokens = (
    tokenizer.convert_ids_to_tokens(
        input_ids.detach().cpu().tolist()
    )
)


print("\n")
print("=" * 100)
print("[ALUR A] TOKENISASI")
print("=" * 100)

print(
    f"Jumlah token: {len(tokens)}"
)

print()

for position, (
    token,
    token_id,
    mask_value
) in enumerate(
    zip(
        tokens,
        input_ids.detach().cpu().tolist(),
        attention_mask.detach().cpu().tolist()
    ),
    start=1
):
    print(
        f"{position:>3}. "
        f"Token: {token:<20} "
        f"→ ID: {token_id:<8} "
        f"Attention Mask: {mask_value}"
    )

print(
    "\nCatatan: tokenizer dapat memecah satu kata "
    "menjadi beberapa subword token."
)

Masukkan query untuk melihat alur Sentence-BERT: I am a complete beginner who wants to transition into data science. I need a practical book that teaches Python, data analysis, and the basic skills required to start a career in this field.
ALUR MANUAL SENTENCE-BERT

Teks Input:
I am a complete beginner who wants to transition into data science. I need a practical book that teaches Python, data analysis, and the basic skills required to start a career in this field.


[ALUR A] TOKENISASI
Jumlah token: 42

  1. Token: <s>                  → ID: 0        Attention Mask: 1
  2. Token: ▁I                   → ID: 87       Attention Mask: 1
  3. Token: ▁am                  → ID: 444      Attention Mask: 1
  4. Token: ▁a                   → ID: 10       Attention Mask: 1
  5. Token: ▁complete            → ID: 28484    Attention Mask: 1
  6. Token: ▁begin               → ID: 9842     Attention Mask: 1
  7. Token: ner                  → ID: 1679     Attention Mask: 1
  8. Token: ▁who            

In [ ]:
# ============================================================
# 3. TRANSFORMER FORWARD PASS
# ============================================================

transformer_model.eval()

with torch.no_grad():
    transformer_output = (
        transformer_model(
            **encoded_query
        )
    )


token_embeddings = (
    transformer_output
    .last_hidden_state[0]
)


print("\n")
print("=" * 100)
print(
    "[ALUR B] CONTEXTUAL TOKEN EMBEDDINGS"
)
print("=" * 100)

print(
    "Ukuran matriks token embedding:",
    tuple(
        token_embeddings.shape
    )
)

print(
    "Format: "
    "(jumlah token × dimensi embedding)"
)

print()


# Tampilkan hanya 5 angka pertama setiap token.
for index, token in enumerate(tokens):

    vector_preview = (
        token_embeddings[index]
        [:5]
        .detach()
        .cpu()
        .numpy()
    )

    vector_string = (
        np.array2string(
            vector_preview,
            precision=4,
            suppress_small=True
        )
    )

    print(
        f"Token: {token:<20} "
        f"→ {vector_string}"
    )

print(
    "\nSetiap token sebenarnya memiliki "
    f"{token_embeddings.shape[1]} nilai."
)

print(
    "Hanya lima nilai pertama ditampilkan."
)



[ALUR B] CONTEXTUAL TOKEN EMBEDDINGS
Ukuran matriks token embedding: (42, 768)
Format: (jumlah token × dimensi embedding)

Token: <s>                  → [-0.0192  0.3464 -0.0011  0.0182  0.0596]
Token: ▁I                   → [-0.0487  0.4013 -0.0048 -0.0373  0.063 ]
Token: ▁am                  → [-0.0127  0.294  -0.0064 -0.0098  0.0873]
Token: ▁a                   → [ 0.0009  0.3027 -0.006  -0.0103  0.0364]
Token: ▁complete            → [-0.0489  0.4157 -0.0064  0.0252  0.02  ]
Token: ▁begin               → [-0.0696  0.275  -0.0101  0.0022  0.0326]
Token: ner                  → [ 0.0008  0.2715 -0.0082  0.0125  0.0203]
Token: ▁who                 → [-0.0396  0.4033 -0.0036 -0.0241  0.0389]
Token: ▁wants               → [-0.0578  0.3755 -0.0056 -0.0743  0.0713]
Token: ▁to                  → [-0.0893  0.4056 -0.0036 -0.115   0.1715]
Token: ▁transition          → [-0.0015  0.4278  0.0005 -0.026   0.1002]
Token: ▁into                → [-0.1444  0.42   -0.0025 -0.15    0.1059]
Token: ▁dat

In [ ]:
# ============================================================
# 4. MANUAL MEAN POOLING
# ============================================================

expanded_attention_mask = (
    attention_mask
    .unsqueeze(-1)
    .expand(
        token_embeddings.size()
    )
    .float()
)

sum_token_embeddings = (
    torch.sum(
        token_embeddings
        * expanded_attention_mask,
        dim=0
    )
)

sum_attention_mask = (
    torch.clamp(
        expanded_attention_mask.sum(
            dim=0
        ),
        min=1e-9
    )
)

pooled_embedding = (
    sum_token_embeddings
    / sum_attention_mask
)


pooled_embedding_np = (
    pooled_embedding
    .detach()
    .cpu()
    .numpy()
)


print("\n")
print("=" * 100)
print("[ALUR C] MEAN POOLING")
print("=" * 100)

print(
    "Pooling strategy: Mean Pooling"
)

print(
    "Dimensi sentence embedding:",
    pooled_embedding_np.shape
)

print(
    "5 nilai awal SEBELUM normalization:"
)

print(
    np.round(
        pooled_embedding_np[:5],
        6
    )
)

print(
    "L2 Norm sebelum normalization:",
    round(
        float(
            np.linalg.norm(
                pooled_embedding_np
            )
        ),
        6
    )
)



[ALUR C] MEAN POOLING
Pooling strategy: Mean Pooling
Dimensi sentence embedding: (768,)
5 nilai awal SEBELUM normalization:
[-0.058491  0.412038 -0.003188 -0.035751  0.039132]
L2 Norm sebelum normalization: 2.890816


In [ ]:
# ============================================================
# 5. MANUAL L2 NORMALIZATION
# ============================================================

embedding_norm = np.linalg.norm(
    pooled_embedding_np
)

if embedding_norm == 0:
    raise ValueError(
        "Norm embedding bernilai 0."
    )

normalized_query_embedding = (
    pooled_embedding_np
    / embedding_norm
)


print("\n")
print("=" * 100)
print("[ALUR D] L2 NORMALIZATION")
print("=" * 100)

print(
    "5 nilai awal SEBELUM normalization:"
)

print(
    np.round(
        pooled_embedding_np[:5],
        6
    )
)

print(
    "\n5 nilai awal SETELAH normalization:"
)

print(
    np.round(
        normalized_query_embedding[:5],
        6
    )
)

print(
    "\nL2 Norm sebelum:",
    round(
        float(
            np.linalg.norm(
                pooled_embedding_np
            )
        ),
        6
    )
)

print(
    "L2 Norm setelah:",
    round(
        float(
            np.linalg.norm(
                normalized_query_embedding
            )
        ),
        6
    )
)



[ALUR D] L2 NORMALIZATION
5 nilai awal SEBELUM normalization:
[-0.058491  0.412038 -0.003188 -0.035751  0.039132]

5 nilai awal SETELAH normalization:
[-0.020233  0.142533 -0.001103 -0.012367  0.013537]

L2 Norm sebelum: 2.890816
L2 Norm setelah: 1.0


In [ ]:
# ============================================================
# 6. COSINE SIMILARITY WITH ALL CORPUS EMBEDDINGS
# ============================================================

corpus_matrix = np.asarray(
    mpnet_embeddings,
    dtype=np.float32
)


# Defensive normalization:
# file corpus Anda sebenarnya sudah dibuat menggunakan
# normalize_embeddings=True.
corpus_norms = np.linalg.norm(
    corpus_matrix,
    axis=1,
    keepdims=True
)

normalized_corpus_matrix = (
    corpus_matrix
    / np.clip(
        corpus_norms,
        1e-12,
        None
    )
)


# Karena query dan corpus sudah normalized,
# dot product ekuivalen dengan cosine similarity.
all_similarity_scores = (
    normalized_corpus_matrix
    @ normalized_query_embedding
)


sorted_indices = np.argsort(
    -all_similarity_scores
)

highest_index = int(
    sorted_indices[0]
)

lowest_index = int(
    sorted_indices[-1]
)


highest_score = float(
    all_similarity_scores[
        highest_index
    ]
)

lowest_score = float(
    all_similarity_scores[
        lowest_index
    ]
)


print("\n")
print("=" * 100)
print("[ALUR E] SORTING SEMANTIC SIMILARITY")
print("=" * 100)

print(
    "Jumlah corpus yang dibandingkan:",
    len(corpus_matrix)
)

print(
    "Seluruh skor diurutkan dari "
    "similarity tertinggi ke terendah."
)

print(
    "\nCorpus terpilih untuk demonstrasi:"
)

print(
    "1. Buku dengan similarity tertinggi"
)

print(
    "2. Buku dengan similarity terendah"
)



[ALUR E] SORTING SEMANTIC SIMILARITY
Jumlah corpus yang dibandingkan: 4083
Seluruh skor diurutkan dari similarity tertinggi ke terendah.

Corpus terpilih untuk demonstrasi:
1. Buku dengan similarity tertinggi
2. Buku dengan similarity terendah


In [ ]:
# ============================================================
# 7. MANUAL COSINE SIMILARITY FOR EXTREME RESULTS
# ============================================================

def explain_manual_cosine(
    query_vector,
    corpus_vector,
    book_row,
    label
):
    query_vector = np.asarray(
        query_vector,
        dtype=float
    )

    corpus_vector = np.asarray(
        corpus_vector,
        dtype=float
    )

    dot_product = np.dot(
        query_vector,
        corpus_vector
    )

    query_magnitude = np.linalg.norm(
        query_vector
    )

    corpus_magnitude = np.linalg.norm(
        corpus_vector
    )

    cosine_score = (
        dot_product
        / (
            query_magnitude
            * corpus_magnitude
        )
    )

    print("\n")
    print("=" * 100)
    print(label)
    print("=" * 100)

    print(
        "Judul:",
        book_row.get(
            "title",
            "-"
        )
    )

    print(
        "Penulis:",
        book_row.get(
            "author",
            "-"
        )
    )

    print(
        "Book ID:",
        book_row.get(
            "book_id",
            "-"
        )
    )

    print(
        "\n5 nilai awal Query Vector:"
    )

    print(
        np.round(
            query_vector[:5],
            6
        )
    )

    print(
        "5 nilai awal Corpus Vector:"
    )

    print(
        np.round(
            corpus_vector[:5],
            6
        )
    )

    print("\nStep 1 - Dot Product")

    print(
        "Query · Corpus =",
        round(
            float(dot_product),
            6
        )
    )

    print(
        "\nStep 2 - Magnitude Query"
    )

    print(
        "||Query|| =",
        round(
            float(query_magnitude),
            6
        )
    )

    print(
        "\nStep 3 - Magnitude Corpus"
    )

    print(
        "||Corpus|| =",
        round(
            float(corpus_magnitude),
            6
        )
    )

    print(
        "\nStep 4 - Cosine Similarity"
    )

    print(
        "cosine = "
        "dot_product / "
        "(||Query|| × ||Corpus||)"
    )

    print(
        f"= {dot_product:.6f} / "
        f"({query_magnitude:.6f} × "
        f"{corpus_magnitude:.6f})"
    )

    print(
        f"= {cosine_score:.6f}"
    )

    return cosine_score

In [ ]:
# ============================================================
# HIGHEST SIMILARITY
# ============================================================

highest_book = (
    df_books_model.iloc[
        highest_index
    ]
)

highest_manual_score = (
    explain_manual_cosine(
        query_vector=(
            normalized_query_embedding
        ),
        corpus_vector=(
            normalized_corpus_matrix[
                highest_index
            ]
        ),
        book_row=highest_book,
        label=(
            "[BUKU DENGAN SIMILARITY TERTINGGI]"
        )
    )
)


# ============================================================
# LOWEST SIMILARITY
# ============================================================

lowest_book = (
    df_books_model.iloc[
        lowest_index
    ]
)

lowest_manual_score = (
    explain_manual_cosine(
        query_vector=(
            normalized_query_embedding
        ),
        corpus_vector=(
            normalized_corpus_matrix[
                lowest_index
            ]
        ),
        book_row=lowest_book,
        label=(
            "[BUKU DENGAN SIMILARITY TERENDAH]"
        )
    )
)



[BUKU DENGAN SIMILARITY TERTINGGI]
Judul: Python for Data Analysis
Penulis: Wes McKinney
Book ID: BOOK_02847

5 nilai awal Query Vector:
[-0.020233  0.142533 -0.001103 -0.012367  0.013537]
5 nilai awal Corpus Vector:
[-0.005835  0.084334 -0.001117  0.003782  0.017508]

Step 1 - Dot Product
Query · Corpus = 0.667596

Step 2 - Magnitude Query
||Query|| = 1.0

Step 3 - Magnitude Corpus
||Corpus|| = 1.0

Step 4 - Cosine Similarity
cosine = dot_product / (||Query|| × ||Corpus||)
= 0.667596 / (1.000000 × 1.000000)
= 0.667596


[BUKU DENGAN SIMILARITY TERENDAH]
Judul: Rape is Rape: How Denial, Distortion, and Victim Blaming are Fueling a Hidden Acquaintance Rape Crisis
Penulis: Jody Raphael
Book ID: BOOK_03457

5 nilai awal Query Vector:
[-0.020233  0.142533 -0.001103 -0.012367  0.013537]
5 nilai awal Corpus Vector:
[ 0.042881  0.024191 -0.002783 -0.026434  0.005004]

Step 1 - Dot Product
Query · Corpus = -0.048042

Step 2 - Magnitude Query
||Query|| = 1.0

Step 3 - Magnitude Corpus
||Corpu

In [ ]:
# ============================================================
# MANUAL INFERENCE SUMMARY
# ============================================================

manual_inference_summary = pd.DataFrame({
    "Type": [
        "Highest Similarity",
        "Lowest Similarity"
    ],
    "Corpus Index": [
        highest_index,
        lowest_index
    ],
    "Book ID": [
        highest_book.get(
            "book_id",
            "-"
        ),
        lowest_book.get(
            "book_id",
            "-"
        )
    ],
    "Title": [
        highest_book.get(
            "title",
            "-"
        ),
        lowest_book.get(
            "title",
            "-"
        )
    ],
    "Cosine Similarity": [
        highest_manual_score,
        lowest_manual_score
    ]
})


print("\n")
print("=" * 100)
print("RINGKASAN MANUAL INFERENCE")
print("=" * 100)

print(
    manual_inference_summary
    .round(6)
    .to_string(index=False)
)



RINGKASAN MANUAL INFERENCE
              Type  Corpus Index    Book ID                                                                                                  Title  Cosine Similarity
Highest Similarity          2846 BOOK_02847                                                                               Python for Data Analysis               0.67
 Lowest Similarity          3456 BOOK_03457 Rape is Rape: How Denial, Distortion, and Victim Blaming are Fueling a Hidden Acquaintance Rape Crisis              -0.05


## Hybrid Title and Semantic Search

In [ ]:
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.1 MB/s eta 0:00:00


In [ ]:
import re
import unicodedata

from rapidfuzz import fuzz, process

In [ ]:
# ============================================================
# TITLE NORMALIZATION
# ============================================================
# Menyamakan kapitalisasi, tanda baca, dan spasi agar judul
# yang secara makna sama tetap dapat dicocokkan.
#
# Contoh:
# "The Lean Startup!" → "the lean startup"

def normalize_title(title):
    if pd.isna(title):
        return ""

    title = unicodedata.normalize(
        "NFKC",
        str(title)
    )

    title = title.casefold()

    # Mengganti tanda baca dengan spasi.
    title = re.sub(
        r"[^\w\s]",
        " ",
        title,
        flags=re.UNICODE
    )

    # Menyamakan spasi berlebih.
    title = re.sub(
        r"\s+",
        " ",
        title
    )

    return title.strip()

In [ ]:
# ============================================================
# CREATE TITLE SEARCH INDEX
# ============================================================

df_books_model["title_normalized"] = (
    df_books_model["title"]
    .apply(normalize_title)
)

print(
    "Judul kosong setelah normalisasi:",
    df_books_model["title_normalized"]
    .eq("")
    .sum()
)

Judul kosong setelah normalisasi: 0


In [ ]:
# ============================================================
# FUZZY TITLE CHOICES
# ============================================================
# Dictionary memakai index DataFrame sebagai key agar hasil
# RapidFuzz dapat diarahkan kembali ke buku yang benar.

title_choices = (
    df_books_model["title_normalized"]
    .to_dict()
)

In [ ]:
# ============================================================
# EXACT TITLE MATCH
# ============================================================

def find_exact_title_matches(
    query,
    dataframe
):
    normalized_query = normalize_title(query)

    if not normalized_query:
        return dataframe.iloc[0:0].copy()

    exact_matches = dataframe[
        dataframe["title_normalized"]
        == normalized_query
    ].copy()

    exact_matches["title_match_score"] = 100.0
    exact_matches["match_type"] = "exact_title"

    return exact_matches

In [ ]:
# ============================================================
# FUZZY TITLE MATCH
# ============================================================
# WRatio cukup fleksibel untuk:
# - typo;
# - kata hilang;
# - urutan kata sedikit berbeda;
# - variasi tanda baca.
#
# score_cutoff mencegah saran judul yang terlalu lemah.

def find_fuzzy_title_matches(
    query,
    dataframe,
    choices,
    score_cutoff=90,
    limit=5,
    minimum_length_ratio=0.55
):
    normalized_query = normalize_title(query)

    if not normalized_query:
        return dataframe.iloc[0:0].copy()

    # Jangan menjalankan fuzzy title untuk query kebutuhan.
    if not is_title_like_query(normalized_query):
        return dataframe.iloc[0:0].copy()

    fuzzy_results = process.extract(
        normalized_query,
        choices,
        scorer=fuzz.WRatio,
        processor=None,
        score_cutoff=score_cutoff,
        limit=limit
    )

    if not fuzzy_results:
        return dataframe.iloc[0:0].copy()

    matched_records = []

    query_length = len(normalized_query)

    for matched_title, score, dataframe_index in fuzzy_results:
        matched_title_length = len(matched_title)

        length_ratio = (
            min(query_length, matched_title_length)
            / max(query_length, matched_title_length)
        )

        # Menghindari query sangat panjang dicocokkan
        # dengan judul yang jauh lebih pendek.
        if length_ratio < minimum_length_ratio:
            continue

        record = dataframe.loc[
            [dataframe_index]
        ].copy()

        record["title_match_score"] = float(score)
        record["title_length_ratio"] = float(
            length_ratio
        )
        record["match_type"] = "fuzzy_title"

        matched_records.append(record)

    if not matched_records:
        return dataframe.iloc[0:0].copy()

    fuzzy_matches = pd.concat(
        matched_records,
        ignore_index=False
    )

    fuzzy_matches = fuzzy_matches.drop_duplicates(
        subset=["book_id"]
    )

    fuzzy_matches = fuzzy_matches.sort_values(
        "title_match_score",
        ascending=False
    )

    return fuzzy_matches

In [ ]:
# ============================================================
# DETECT WHETHER QUERY LOOKS LIKE A BOOK TITLE
# ============================================================

def is_title_like_query(
    query,
    max_words=12,
    max_characters=120
):
    normalized_query = normalize_title(query)

    if not normalized_query:
        return False

    words = normalized_query.split()

    # Query sangat panjang biasanya merupakan kebutuhan,
    # bukan judul buku.
    if len(words) > max_words:
        return False

    if len(normalized_query) > max_characters:
        return False

    # Pola yang biasanya menunjukkan kebutuhan pengguna.
    semantic_need_patterns = [
        r"\bi need\b",
        r"\bi want\b",
        r"\bi am looking for\b",
        r"\blooking for\b",
        r"\brecommend\b",
        r"\brecommendation\b",
        r"\bbook about\b",
        r"\bbook that\b",
        r"\bbooks about\b",
        r"\bhelp me\b",
        r"\bhow to\b",
        r"\bhow can i\b",
        r"\bwhat book\b",
        r"\bwhat should i read\b",

        # Bahasa Indonesia
        r"\bsaya ingin\b",
        r"\bsaya butuh\b",
        r"\bsaya mencari\b",
        r"\bcarikan\b",
        r"\brekomendasikan\b",
        r"\brekomendasi buku\b",
        r"\bbuku tentang\b",
        r"\bbuku yang\b",
        r"\bbagaimana cara\b",
        r"\bcara untuk\b"
    ]

    for pattern in semantic_need_patterns:
        if re.search(
            pattern,
            normalized_query,
            flags=re.IGNORECASE
        ):
            return False

    return True

In [ ]:
# ============================================================
# DETERMINE SEARCH INTENT
# ============================================================

def determine_search_intent(
    query,
    dataframe
):
    exact_matches = find_exact_title_matches(
        query=query,
        dataframe=dataframe
    )

    if not exact_matches.empty:
        return {
            "intent": "exact_title",
            "exact_matches": exact_matches,
            "is_title_like": True
        }

    title_like = is_title_like_query(
        query
    )

    if title_like:
        return {
            "intent": "possible_title",
            "exact_matches": exact_matches,
            "is_title_like": True
        }

    return {
        "intent": "semantic_need",
        "exact_matches": exact_matches,
        "is_title_like": False
    }

In [ ]:
test_queries = [
    "The Lean Startup",
    "The Len Startap",
    (
        "I am a complete beginner who wants to "
        "transition into data science. I need a practical book"
    ),
    (
        "Saya ingin buku tentang mengatasi "
        "prokrastinasi"
    )
]

for test_query in test_queries:
    intent_result = determine_search_intent(
        query=test_query,
        dataframe=df_books_model
    )

    print(
        test_query,
        "→",
        intent_result["intent"]
    )

The Lean Startup → possible_title
The Len Startap → possible_title
I am a complete beginner who wants to transition into data science. I need a practical book → semantic_need
Saya ingin buku tentang mengatasi prokrastinasi → semantic_need


In [ ]:
# ============================================================
# SEMANTIC SIMILARITY FROM ANCHOR BOOKS
# ============================================================
# Bila terdapat beberapa edisi dengan judul sama, embedding
# anchor dirata-ratakan.

def retrieve_similar_to_anchor_books(
    anchor_indices,
    dataframe,
    corpus_embeddings,
    top_k=50
):
    anchor_indices = np.asarray(
        anchor_indices,
        dtype=int
    )

    if len(anchor_indices) == 0:
        raise ValueError(
            "Anchor book tidak tersedia."
        )

    anchor_embedding = corpus_embeddings[
        anchor_indices
    ].mean(axis=0, keepdims=True)

    # Normalisasi setelah averaging.
    anchor_norm = np.linalg.norm(
        anchor_embedding,
        axis=1,
        keepdims=True
    )

    anchor_embedding = (
        anchor_embedding
        / np.clip(anchor_norm, 1e-12, None)
    )

    similarity_scores = util.cos_sim(
        anchor_embedding,
        corpus_embeddings
    )[0].cpu().numpy()

    # Buku anchor dikeluarkan dari semantic tail.
    similarity_scores[anchor_indices] = -np.inf

    top_k = min(
        top_k,
        len(dataframe) - len(anchor_indices)
    )

    candidate_indices = np.argpartition(
        -similarity_scores,
        top_k - 1
    )[:top_k]

    candidate_indices = candidate_indices[
        np.argsort(
            -similarity_scores[candidate_indices]
        )
    ]

    candidates = dataframe.iloc[
        candidate_indices
    ].copy()

    candidates["corpus_index"] = candidate_indices
    candidates["retrieval_score"] = (
        similarity_scores[candidate_indices]
    )
    candidates["retrieval_rank"] = np.arange(
        1,
        len(candidates) + 1
    )
    candidates["match_type"] = "semantic_similar"
    candidates["title_match_score"] = np.nan

    return candidates

In [ ]:
# ============================================================
# OPTIONAL RERANKING FOR ALL SEMANTIC CANDIDATES
# ============================================================

def rerank_semantic_results(
    query,
    candidates,
    reranker,
    batch_size=16
):
    """
    Mengurutkan ulang seluruh semantic candidates menggunakan
    Cross-Encoder tanpa memotongnya menjadi Top-10.

    Pemotongan halaman dilakukan setelah ranking final selesai.
    """

    if candidates is None or candidates.empty:
        empty_results = candidates.copy()

        if "reranker_score" not in empty_results.columns:
            empty_results["reranker_score"] = np.nan

        return empty_results

    if reranker is None:
        raise ValueError(
            "Objek reranker belum diberikan."
        )

    if "model_text" not in candidates.columns:
        raise ValueError(
            "Kolom model_text tidak ditemukan pada candidates."
        )

    query_document_pairs = [
        (
            query,
            document_text
        )
        for document_text in candidates[
            "model_text"
        ].fillna("").astype(str)
    ]

    reranker_scores = reranker.predict(
        query_document_pairs,
        batch_size=batch_size,
        show_progress_bar=False
    )

    reranked_results = candidates.copy()

    reranked_results["reranker_score"] = (
        np.asarray(
            reranker_scores,
            dtype=float
        ).reshape(-1)
    )

    reranked_results = (
        reranked_results
        .sort_values(
            by="reranker_score",
            ascending=False,
            kind="stable"
        )
        .reset_index(drop=True)
    )

    return reranked_results

In [ ]:
# ============================================================
# HYBRID TITLE AND SEMANTIC BOOK SEARCH WITH FULL TOP-K RESULTS
# ============================================================

def search_books_hybrid(
    query,
    dataframe,
    embedding_model,
    corpus_embeddings,
    title_choice_index,
    reranker=None,
    top_k_candidates=50,
    fuzzy_score_cutoff=85,
    fuzzy_suggestion_limit=5,
    fuzzy_auto_accept_score=92,
    fuzzy_min_length_ratio=0.65,
    use_reranker=False,
    auto_accept_fuzzy=False,
    reranker_batch_size=16
):
    """
    Menghasilkan ranking lengkap hingga Top-K kandidat.

    Menjalankan pencarian buku dengan tiga jalur:

    1. exact_title
       Query sama persis dengan judul buku.

    2. possible_title
       Query tampak seperti judul dan diperiksa menggunakan
       fuzzy matching.

       - Jika fuzzy sangat kuat dan auto_accept_fuzzy=True,
         buku fuzzy dipasang di posisi atas.
       - Jika fuzzy belum cukup kuat, fuzzy hanya menjadi
         suggestion dan hasil utama tetap semantic retrieval.

    3. semantic_need
       Query berupa kebutuhan informasi pengguna dan langsung
       diproses menggunakan semantic retrieval.

    Pagination dilakukan setelah fungsi ini selesai agar urutan
    ranking 1 sampai Top-K tetap konsisten.
    """

    # ========================================================
    # 1. INPUT VALIDATION
    # ========================================================

    if dataframe is None or dataframe.empty:
        raise ValueError(
            "DataFrame buku kosong atau belum tersedia."
        )

    if embedding_model is None:
        raise ValueError(
            "Embedding model belum tersedia."
        )

    if corpus_embeddings is None:
        raise ValueError(
            "Corpus embeddings belum tersedia."
        )

    if len(dataframe) != len(corpus_embeddings):
        raise ValueError(
            "Jumlah baris dataframe tidak sama dengan "
            "jumlah corpus embeddings."
        )

    if top_k_candidates < 1:
        raise ValueError(
            "top_k_candidates minimal 1."
        )

    top_k_candidates = min(
        int(top_k_candidates),
        len(dataframe)
    )

    clean_query = prepare_inference_query(
        query
    )

    # ========================================================
    # 2. INITIALIZE RESULT CONTAINERS
    # ========================================================

    empty_dataframe = dataframe.iloc[0:0].copy()

    fuzzy_suggestions = empty_dataframe.copy()
    pinned_results = empty_dataframe.copy()
    semantic_candidates = empty_dataframe.copy()

    search_mode = "semantic_query"
    suggested_title = None

    best_fuzzy_score = None
    best_fuzzy_length_ratio = None
    fuzzy_auto_accepted = False

    # ========================================================
    # 3. DETERMINE SEARCH INTENT
    # ========================================================

    intent_result = determine_search_intent(
        query=clean_query,
        dataframe=dataframe
    )

    search_intent = intent_result[
        "intent"
    ]

    exact_matches = intent_result[
        "exact_matches"
    ]

    title_like = intent_result[
        "is_title_like"
    ]

    # ========================================================
    # HELPER: NORMAL SEMANTIC RETRIEVAL
    # ========================================================

    def run_semantic_retrieval():
        _, candidates = retrieve_candidates(
            query=clean_query,
            embedding_model=embedding_model,
            corpus_embeddings=corpus_embeddings,
            dataframe=dataframe,
            top_k=top_k_candidates
        )

        candidates = candidates.copy()

        candidates["match_type"] = (
            "semantic_query"
        )

        candidates["title_match_score"] = np.nan
        candidates["title_length_ratio"] = np.nan

        return candidates

    # ========================================================
    # 4. ROUTE A: EXACT TITLE
    # ========================================================

    if search_intent == "exact_title":
        search_mode = "exact_title"

        pinned_results = exact_matches.copy()

        pinned_results["title_match_score"] = 100.0
        pinned_results["title_length_ratio"] = 1.0
        pinned_results["match_type"] = "exact_title"

        anchor_indices = (
            pinned_results.index.to_numpy()
        )

        semantic_candidates = (
            retrieve_similar_to_anchor_books(
                anchor_indices=anchor_indices,
                dataframe=dataframe,
                corpus_embeddings=corpus_embeddings,
                top_k=top_k_candidates
            )
        )

    # ========================================================
    # 5. ROUTE B: POSSIBLE TITLE / TYPO
    # ========================================================

    elif search_intent == "possible_title":
        fuzzy_suggestions = (
            find_fuzzy_title_matches(
                query=clean_query,
                dataframe=dataframe,
                choices=title_choice_index,
                score_cutoff=fuzzy_score_cutoff,
                limit=fuzzy_suggestion_limit
            )
        )

        if not fuzzy_suggestions.empty:
            best_fuzzy_row = (
                fuzzy_suggestions.iloc[0]
            )

            best_fuzzy_score = float(
                best_fuzzy_row[
                    "title_match_score"
                ]
            )

            best_fuzzy_length_ratio = float(
                best_fuzzy_row.get(
                    "title_length_ratio",
                    0.0
                )
            )

            suggested_title = (
                best_fuzzy_row["title"]
            )

            fuzzy_auto_accepted = (
                auto_accept_fuzzy
                and best_fuzzy_score
                >= fuzzy_auto_accept_score
                and best_fuzzy_length_ratio
                >= fuzzy_min_length_ratio
            )

            # ------------------------------------------------
            # FUZZY TITLE SANGAT KUAT
            # ------------------------------------------------

            if fuzzy_auto_accepted:
                search_mode = "fuzzy_title"

                best_fuzzy_title = (
                    best_fuzzy_row[
                        "title_normalized"
                    ]
                )

                pinned_results = dataframe[
                    dataframe["title_normalized"]
                    == best_fuzzy_title
                ].copy()

                pinned_results[
                    "title_match_score"
                ] = best_fuzzy_score

                pinned_results[
                    "title_length_ratio"
                ] = best_fuzzy_length_ratio

                pinned_results[
                    "match_type"
                ] = "fuzzy_title_suggestion"

                anchor_indices = (
                    pinned_results.index.to_numpy()
                )

                semantic_candidates = (
                    retrieve_similar_to_anchor_books(
                        anchor_indices=anchor_indices,
                        dataframe=dataframe,
                        corpus_embeddings=corpus_embeddings,
                        top_k=top_k_candidates
                    )
                )

            # ------------------------------------------------
            # FUZZY HANYA MENJADI SARAN
            # ------------------------------------------------

            else:
                search_mode = (
                    "semantic_query_with_suggestion"
                )

                semantic_candidates = (
                    run_semantic_retrieval()
                )

        else:
            search_mode = "semantic_query"

            semantic_candidates = (
                run_semantic_retrieval()
            )

    # ========================================================
    # 6. ROUTE C: SEMANTIC INFORMATION NEED
    # ========================================================

    else:
        search_mode = "semantic_query"

        semantic_candidates = (
            run_semantic_retrieval()
        )

    # ========================================================
    # 7. REMOVE PINNED BOOKS FROM SEMANTIC CANDIDATES
    # ========================================================

    pinned_results = (
        pinned_results
        .drop_duplicates(
            subset=["book_id"],
            keep="first"
        )
        .copy()
    )

    if (
        not pinned_results.empty
        and not semantic_candidates.empty
    ):
        pinned_book_ids = set(
            pinned_results[
                "book_id"
            ].astype(str)
        )

        semantic_candidates = (
            semantic_candidates[
                ~semantic_candidates[
                    "book_id"
                ]
                .astype(str)
                .isin(pinned_book_ids)
            ]
            .copy()
        )

    # Jangan biarkan pinned results melebihi Top-K.
    pinned_results = (
        pinned_results
        .head(top_k_candidates)
        .copy()
    )

    # Jumlah posisi tersisa untuk hasil semantik.
    semantic_result_limit = max(
        top_k_candidates
        - len(pinned_results),
        0
    )

    # ========================================================
    # 8. OPTIONAL CROSS-ENCODER RERANKING
    # ========================================================

    reranking_used = False

    if (
        use_reranker
        and semantic_result_limit > 0
        and not semantic_candidates.empty
    ):
        if reranker is None:
            raise ValueError(
                "use_reranker=True, tetapi objek "
                "reranker belum diberikan."
            )

        semantic_results = (
            rerank_semantic_results(
                query=clean_query,
                candidates=semantic_candidates,
                reranker=reranker,
                batch_size=reranker_batch_size
            )
            .head(semantic_result_limit)
            .copy()
        )

        reranking_used = True

    else:
        semantic_results = (
            semantic_candidates
            .head(semantic_result_limit)
            .copy()
        )

        if (
            "reranker_score"
            not in semantic_results.columns
        ):
            semantic_results[
                "reranker_score"
            ] = np.nan

    # ========================================================
    # 9. PREPARE PINNED RESULT COLUMNS
    # ========================================================

    if not pinned_results.empty:
        pinned_results = pinned_results.copy()

        pinned_results["retrieval_score"] = np.nan
        pinned_results["retrieval_rank"] = np.nan
        pinned_results["reranker_score"] = np.nan

    # ========================================================
    # 10. COMBINE FULL TOP-K RESULTS
    # ========================================================

    combined_results = pd.concat(
        [
            pinned_results,
            semantic_results
        ],
        ignore_index=True,
        sort=False
    )

    combined_results = (
        combined_results
        .drop_duplicates(
            subset=["book_id"],
            keep="first"
        )
        .head(top_k_candidates)
        .reset_index(drop=True)
    )

    if "final_rank" in combined_results.columns:
        combined_results = (
            combined_results.drop(
                columns=["final_rank"]
            )
        )

    combined_results.insert(
        0,
        "final_rank",
        np.arange(
            1,
            len(combined_results) + 1
        )
    )

    # ========================================================
    # 11. SYSTEM METADATA
    # ========================================================

    system_used = (
        "MPNET_RERANK"
        if reranking_used
        else "MPNET_RETRIEVAL"
    )

    metadata = {
        "original_query": query,
        "clean_query": clean_query,

        "search_intent": search_intent,
        "search_mode": search_mode,
        "is_title_like_query": title_like,

        "suggested_title": suggested_title,
        "exact_match_count": len(
            exact_matches
        ),
        "fuzzy_suggestion_count": len(
            fuzzy_suggestions
        ),

        "best_fuzzy_score":
            best_fuzzy_score,

        "best_fuzzy_length_ratio":
            best_fuzzy_length_ratio,

        "fuzzy_auto_accept_score":
            fuzzy_auto_accept_score,

        "fuzzy_min_length_ratio":
            fuzzy_min_length_ratio,

        "auto_accept_fuzzy":
            auto_accept_fuzzy,

        "fuzzy_auto_accepted":
            fuzzy_auto_accepted,

        "use_reranker_requested":
            use_reranker,

        "reranking_used":
            reranking_used,

        "system_used":
            system_used,

        "top_k_candidates":
            top_k_candidates,

        "total_results":
            len(combined_results),

        "default_page_size":
            10,

        "pinned_result_count":
            len(pinned_results),

        "semantic_result_count":
            len(semantic_results)
    }

    return (
        combined_results,
        fuzzy_suggestions,
        metadata
    )

In [ ]:
# ============================================================
# PAGINATE RANKED RECOMMENDATION RESULTS
# ============================================================

def paginate_results(
    results,
    page=1,
    page_size=10
):
    """
    Mengambil satu halaman dari ranking lengkap tanpa
    mengubah final_rank asli.
    """

    if results is None:
        raise ValueError(
            "Results belum tersedia."
        )

    if page < 1:
        raise ValueError(
            "Nomor halaman minimal 1."
        )

    if page_size < 1:
        raise ValueError(
            "page_size minimal 1."
        )

    total_results = len(results)

    total_pages = (
        int(
            np.ceil(
                total_results / page_size
            )
        )
        if total_results > 0
        else 0
    )

    start_index = (
        page - 1
    ) * page_size

    end_index = (
        start_index
        + page_size
    )

    if (
        total_pages == 0
        or page > total_pages
    ):
        page_results = (
            results.iloc[0:0]
            .copy()
            .reset_index(drop=True)
        )
    else:
        page_results = (
            results
            .iloc[start_index:end_index]
            .copy()
            .reset_index(drop=True)
        )

    pagination_metadata = {
        "page": int(page),
        "page_size": int(page_size),
        "total_results": int(
            total_results
        ),
        "total_pages": int(
            total_pages
        ),
        "has_previous": (
            page > 1
            and total_pages > 0
        ),
        "has_next": (
            page < total_pages
        ),
        "previous_page": (
            page - 1
            if page > 1
            and total_pages > 0
            else None
        ),
        "next_page": (
            page + 1
            if page < total_pages
            else None
        ),
        "start_rank": (
            start_index + 1
            if not page_results.empty
            else None
        ),
        "end_rank": (
            int(
                page_results[
                    "final_rank"
                ].max()
            )
            if (
                not page_results.empty
                and "final_rank"
                in page_results.columns
            )
            else None
        )
    }

    return (
        page_results,
        pagination_metadata
    )

## Contoh Penggunaan

In [ ]:
# results, suggestions, search_metadata = (
#     search_books_hybrid(
#         query="The Lean Startup",
#         dataframe=df_books_model,
#         embedding_model=best_embedding_model,
#         corpus_embeddings=mpnet_embeddings,
#         title_choice_index=title_choices,
#         reranker=optional_reranker,
#         top_k_candidates=50,
#         top_n=10,
#         use_reranker=False
#     )
# )

# print(search_metadata)

# print(
#     results[
#         [
#             "final_rank",
#             "title",
#             "author",
#             "match_type",
#             "retrieval_score"
#         ]
#     ].to_string(index=False)
# )

In [ ]:
# results, suggestions, search_metadata = (
#     search_books_hybrid(
#         query=(
#             "Buku tentang membangun startup teknologi "
#             "dengan tim kecil dan modal terbatas"
#         ),
#         dataframe=df_books_model,
#         embedding_model=best_embedding_model,
#         corpus_embeddings=mpnet_embeddings,
#         title_choice_index=title_choices,
#         reranker=optional_reranker,
#         top_k_candidates=50,
#         top_n=10,
#         use_reranker=False
#     )
# )

# print(search_metadata["search_mode"])

In [ ]:
# results, suggestions, search_metadata = (
#     search_books_hybrid(
#         query="The Len Startap",
#         dataframe=df_books_model,
#         embedding_model=best_embedding_model,
#         corpus_embeddings=mpnet_embeddings,
#         title_choice_index=title_choices,
#         reranker=optional_reranker,
#         top_k_candidates=50,
#         top_n=10,
#         fuzzy_score_cutoff=85,
#         use_reranker=True,
#         auto_accept_fuzzy=False
#     )
# )

# print(
#     "Cross-Encoder aktif:",
#     search_metadata["reranking_used"]
# )

## Recommendation Output

In [ ]:
def export_complete_recommendations_to_markdown(
    metadata,
    results,
    timing,
    benchmark_metrics,
    file_name="output_complete_recommendations.md",
    description_limit=700
):

    # ========================================================
    # VALIDATE SYSTEM METADATA
    # ========================================================

    reranking_used = metadata.get(
        "reranking_used",
        False
    )

    system_used = metadata.get(
        "system_used"
    )

    if system_used is None:
        system_used = (
            "MPNET_RERANK"
            if reranking_used
            else "MPNET_RETRIEVAL"
        )

    expected_system = (
        "MPNET_RERANK"
        if reranking_used
        else "MPNET_RETRIEVAL"
    )

    if system_used != expected_system:
        raise ValueError(
            "Metadata sistem tidak konsisten. "
            f"reranking_used={reranking_used}, "
            f"system_used={system_used}"
        )

    print("Metadata sistem konsisten.")

    # Buka file dalam mode write
    with open(file_name, "w", encoding="utf-8") as f:

        # ============================================================
        # HEADER UTAMA
        # ============================================================
        f.write("# Laporan Rekomendasi Buku (Hybrid Search)\n\n")

        # ============================================================
        # 1. INFORMASI PENCARIAN & METADATA
        # ============================================================
        f.write("## 🔍 Informasi Pencarian\n")

        # Mengambil dari kamus metadata hasil search_books_hybrid
        query_asli = metadata.get("original_query", "-")
        query_bersih = metadata.get("clean_query", "-")
        search_mode = metadata.get("search_mode", "-")
        exact_count = metadata.get("exact_match_count", 0)
        suggested_title = metadata.get("suggested_title", None)
        is_reranked = metadata.get("reranking_used", False)

        f.write(f"* **Query Asli:** `{query_asli}`\n")
        f.write(f"* **Query Bersih:** `{query_bersih}`\n")
        f.write(f"* **Mode Pencarian:** `{search_mode}`\n")

        # Logika teks Exact Match
        status_exact = "Ya" if exact_count > 0 else "Tidak"
        f.write(f"* **Exact Match Ditemukan:** {status_exact} ({exact_count} buku)\n")

        # Hanya tampilkan judul saran jika ada (fuzzy match aktif)
        if suggested_title:
            f.write(f"* **Judul yang Disarankan (Fuzzy):** *{suggested_title}*\n")

        f.write(f"* **Reranker Digunakan:** {'Ya' if is_reranked else 'Tidak'}\n\n")
        f.write("---\n\n")

        # ============================================================
        # 2. SISTEM & BENCHMARK
        # ============================================================
        f.write("## ⚙️ Sistem & Performa\n")
        f.write(f"**Sistem:** {system_used}\n\n")

        f.write("**Metrik Benchmark 100 Query:**\n")
        metric_labels = {
            "precision_at_5":
                "Precision@5",

            "precision_at_10":
                "Precision@10",

            "recall_at_5":
                "Recall@5",

            "recall_at_10":
                "Recall@10",

            "map_at_5":
                "MAP@5",

            "map_at_10":
                "MAP@10",

            "mrr_at_5":
                "MRR@5",

            "mrr_at_10":
                "MRR@10",

            "ndcg_at_5":
                "NDCG@5",

            "ndcg_at_10":
                "NDCG@10",

            "pooled_recall_at_50":
                "Pooled Recall@50"
        }

        for column_name, display_name in metric_labels.items():
            if column_name in benchmark_metrics:
                f.write(f"* {display_name}: **{benchmark_metrics[column_name]:.4f}**\n")
        f.write("\n---\n\n")

        # ============================================================
        # 3. WAKTU INFERENSI
        # ============================================================
        f.write("## ⏱️ Waktu Inferensi Query Ini\n")
        for timing_name, timing_value in timing.items():
            f.write(f"* **{timing_name}**: {timing_value:.4f} detik\n")
        f.write("\n---\n\n")

        # ============================================================
        # 4. HASIL REKOMENDASI BUKU
        # ============================================================
        f.write("## 📚 Hasil Rekomendasi\n\n")

        for _, book in results.iterrows():
            f.write(f"### Peringkat {int(book.get('final_rank', 0))}\n\n")

            # --- BLOK 1: Identitas Utama Buku ---
            f.write(f"* **Judul:** **{book.get('title', '-')}**\n")
            f.write(f"* **Penulis:** {book.get('author', '-')}\n")
            f.write(f"* **Book ID:** {book.get('book_id', '-')} | **ISBN-13:** {book.get('isbn13_clean', '-')}\n")
            f.write(f"* **Genre/Kategori:** {book.get('genre_text', '-')} ({book.get('target_categories', '-')})\n")

            # --- BLOK 2: Metrik Pencarian (Hybrid Search Analytics) ---
            f.write("* **Analitik Pencarian:**\n")
            f.write(f"  * Match Type: `{book.get('match_type', '-')}`\n")

            # Skor Title/Fuzzy (Hanya tampil jika bukan NaN)
            title_score = book.get("title_match_score", np.nan)
            if pd.notna(title_score):
                f.write(f"  * Title Match Score: **{title_score:.2f}**\n")

            # Skor Semantic / Cosine Similarity (Hanya tampil jika bukan NaN)
            retrieval_score = book.get("retrieval_score", np.nan)
            if pd.notna(retrieval_score):
                f.write(f"  * Cosine Similarity: **{retrieval_score:.4f}**\n")

            # Skor Reranker (Hanya tampil jika bukan NaN)
            reranker_score = book.get("reranker_score", np.nan)
            if pd.notna(reranker_score):
                f.write(f"  * Reranker Score: **{reranker_score:.4f}**\n")

            # --- BLOK 3: Metadata Tambahan Buku ---
            rating = book.get("rating", np.nan)
            rating_str = f"{rating:.2f}" if pd.notna(rating) else "Tidak tersedia"

            totalratings = book.get("totalratings", np.nan)
            totalratings_str = str(int(totalratings)) if pd.notna(totalratings) else "0"

            f.write(f"* **Rating:** {rating_str} (dari {totalratings_str} ulasan)\n")

            # Format Deskripsi
            description = str(book.get("desc", ""))
            if description_limit is not None and len(description) > description_limit:
                description = description[:description_limit] + "..."

            f.write(f"\n**Deskripsi:**\n> {description}\n\n")
            f.write(f"**Tautan Buku:** [Klik di sini]({book.get('link', '#')})\n\n")

            # Garis pemisah antar buku
            f.write("---\n\n")

    print(f"Laporan berhasil diekspor ke: {file_name}")

In [ ]:
# print(
#     "Isi search_metadata:",
#     search_metadata
# )

# print(
#     "Daftar key:",
#     list(search_metadata.keys())
# )

In [ ]:
import time

# ============================================================
# INTERACTIVE HYBRID INFERENCE WITH PAGINATION
# ============================================================

user_query = input(
    "Masukkan kebutuhan atau judul buku: "
)

start_time = time.perf_counter()

(
    all_recommendation_results,
    fuzzy_suggestions,
    search_metadata
) = search_books_hybrid(
    query=user_query,
    dataframe=df_books_model,
    embedding_model=best_embedding_model,
    corpus_embeddings=mpnet_embeddings,
    title_choice_index=title_choices,
    reranker=optional_reranker,

    # Menghasilkan ranking lengkap maksimal 50 buku.
    top_k_candidates=50,

    fuzzy_score_cutoff=85,
    fuzzy_suggestion_limit=5,
    fuzzy_auto_accept_score=92,
    fuzzy_min_length_ratio=0.65,

    use_reranker=False,
    auto_accept_fuzzy=False
)

total_time = (
    time.perf_counter()
    - start_time
)

inference_timing = {
    "total_time_seconds":
        total_time
}

system_used = search_metadata.get(
    "system_used"
)

if system_used is None:
    raise KeyError(
        "system_used tidak ditemukan pada search_metadata."
    )

print(
    "Sistem yang digunakan:",
    system_used
)

print(
    "Mode pencarian:",
    search_metadata.get(
        "search_mode"
    )
)

print(
    "Jumlah ranking lengkap:",
    len(all_recommendation_results)
)

print(
    "Cross-Encoder aktif:",
    search_metadata.get(
        "reranking_used",
        False
    )
)

Masukkan kebutuhan atau judul buku: I am a complete beginner who wants to transition into data science. I need a practical book that teaches Python, data analysis, and the basic skills required to start a career in this field.
Sistem yang digunakan: MPNET_RETRIEVAL
Mode pencarian: semantic_query
Jumlah ranking lengkap: 50
Cross-Encoder aktif: False


In [ ]:
print(
    "Query menyerupai judul:",
    search_metadata[
        "is_title_like_query"
    ]
)

print(
    "Mode pencarian:",
    search_metadata["search_mode"]
)

Query menyerupai judul: False
Mode pencarian: semantic_query


In [ ]:
# ============================================================
# SHOW DEFAULT FIRST PAGE: RANK 1–10
# ============================================================

current_page = 1
page_size = 10

(
    recommendation_results,
    pagination_metadata
) = paginate_results(
    results=all_recommendation_results,
    page=current_page,
    page_size=page_size
)

print("=" * 80)
print(
    f"Halaman {pagination_metadata['page']} "
    f"dari {pagination_metadata['total_pages']}"
)

print(
    f"Menampilkan peringkat "
    f"{pagination_metadata['start_rank']}–"
    f"{pagination_metadata['end_rank']}"
)

print(
    f"Total hasil: "
    f"{pagination_metadata['total_results']}"
)

print("=" * 80)

display_columns = [
    column
    for column in [
        "final_rank",
        "title",
        "author",
        "match_type",
        "title_match_score",
        "retrieval_score",
        "reranker_score"
    ]
    if column in recommendation_results.columns
]

print(
    recommendation_results[
        display_columns
    ].to_string(index=False)
)

Halaman 1 dari 5
Menampilkan peringkat 1–10
Total hasil: 50
 final_rank                                                                        title                                                     author     match_type  title_match_score  retrieval_score  reranker_score
          1                                                     Python for Data Analysis                                               Wes McKinney semantic_query                NaN             0.67             NaN
          2                     Introduction to Computation and Programming Using Python                                             John V. Guttag semantic_query                NaN             0.64             NaN
          3                                                              Python Cookbook                               David Beazley,Brian K. Jones semantic_query                NaN             0.62             NaN
          4                                     Data Structures and Algorithms i

In [ ]:
# ------------------------------------------------------------
# MENAMPILKAN HASIL SINGKAT DI LAYAR (TERMINAL/NOTEBOOK)
# ------------------------------------------------------------
print("=" * 60)
print(f"Mode pencarian: {search_metadata['search_mode']}")

if search_metadata.get("suggested_title") is not None:
    print(f"Mungkin maksud Anda: {search_metadata['suggested_title']}")
print("=" * 60)

print(
    recommendation_results[
        [
            "final_rank",
            "title",
            "author",
            "match_type",
            "title_match_score",
            "retrieval_score",
            "reranker_score"
        ]
    ].to_string(index=False)
)
print("=" * 60)

Mode pencarian: semantic_query
 final_rank                                                                        title                                                     author     match_type  title_match_score  retrieval_score  reranker_score
          1                                                     Python for Data Analysis                                               Wes McKinney semantic_query                NaN             0.67             NaN
          2                     Introduction to Computation and Programming Using Python                                             John V. Guttag semantic_query                NaN             0.64             NaN
          3                                                              Python Cookbook                               David Beazley,Brian K. Jones semantic_query                NaN             0.62             NaN
          4                                     Data Structures and Algorithms in Python Michael T. Goodrich,

In [ ]:
# ============================================================
# SELECT BENCHMARK METRICS
# ============================================================

system_used = search_metadata.get(
    "system_used"
)

if system_used is None:
    system_used = (
        "MPNET_RERANK"
        if search_metadata.get(
            "reranking_used",
            False
        )
        else "MPNET_RETRIEVAL"
    )

benchmark_rows = (
    overall_evaluation_summary[
        overall_evaluation_summary[
            "system_name"
        ]
        == system_used
    ]
)

if benchmark_rows.empty:
    available_systems = (
        overall_evaluation_summary[
            "system_name"
        ]
        .astype(str)
        .tolist()
    )

    raise ValueError(
        f"Metrik benchmark untuk '{system_used}' "
        "tidak ditemukan. Sistem yang tersedia: "
        f"{available_systems}"
    )

selected_benchmark_metrics = (
    benchmark_rows
    .iloc[0]
    .to_dict()
)

print(
    "Benchmark yang digunakan:",
    system_used
)

Benchmark yang digunakan: MPNET_RETRIEVAL


In [ ]:
# ============================================================
# EXPORT TO MARKDOWN
# ============================================================

export_complete_recommendations_to_markdown(
    metadata=search_metadata,
    results=recommendation_results,
    timing=inference_timing,
    benchmark_metrics=(
        selected_benchmark_metrics
    ),
    file_name="output_interactive_hybrid.md",
    description_limit=700
)

print(
    "Hasil berhasil diekspor ke "
    "output_interactive_hybrid.md"
)

Metadata sistem konsisten.
Laporan berhasil diekspor ke: output_interactive_hybrid.md
Hasil berhasil diekspor ke output_interactive_hybrid.md


# Deployment

In [ ]:
# ============================================================
# CONVERT SEARCH RESULTS TO API RESPONSE
# ============================================================

def safe_value(value):
    if isinstance(value, np.generic):
        value = value.item()

    if pd.isna(value):
        return None

    return value


def results_to_api_response(
    results,
    fuzzy_suggestions,
    search_metadata
):
    recommendation_items = []

    for _, row in results.iterrows():
        recommendation_items.append({
            "rank": int(row["final_rank"]),
            "book_id": safe_value(
                row.get("book_id")
            ),
            "title": safe_value(
                row.get("title")
            ),
            "author": safe_value(
                row.get("author")
            ),
            "format": safe_value(
                row.get("bookformat")
            ),
            "genre": safe_value(
                row.get("genre_text")
            ),
            "categories": safe_value(
                row.get("target_categories")
            ),
            "description": safe_value(
                row.get("desc")
            ),
            "image_url": safe_value(
                row.get("img")
            ),
            "source_url": safe_value(
                row.get("link")
            ),
            "pages": safe_value(
                row.get("pages")
            ),
            "rating": safe_value(
                row.get("rating")
            ),
            "total_ratings": safe_value(
                row.get("totalratings")
            ),
            "isbn10": safe_value(
                row.get("isbn10_clean")
            ),
            "isbn13": safe_value(
                row.get("isbn13_clean")
            ),
            "match_type": safe_value(
                row.get("match_type")
            ),
            "title_match_score": safe_value(
                row.get("title_match_score")
            ),
            "cosine_similarity": safe_value(
                row.get("retrieval_score")
            ),
            "reranker_score": safe_value(
                row.get("reranker_score")
            )
        })

    suggestion_items = []

    for _, row in fuzzy_suggestions.iterrows():
        suggestion_items.append({
            "book_id": safe_value(
                row.get("book_id")
            ),
            "title": safe_value(
                row.get("title")
            ),
            "author": safe_value(
                row.get("author")
            ),
            "score": safe_value(
                row.get("title_match_score")
            )
        })

    return {
        "query": search_metadata["original_query"],
        "search_mode": search_metadata["search_mode"],
        "suggested_title": (
            search_metadata["suggested_title"]
        ),
        "reranking_used": (
            search_metadata["reranking_used"]
        ),
        "suggestions": suggestion_items,
        "results": recommendation_items
    }

In [ ]:
# ============================================================
# CONVERT PAGINATED RESULTS TO API RESPONSE
# ============================================================

def paginated_results_to_api_response(
    all_results,
    fuzzy_suggestions,
    search_metadata,
    page=1,
    page_size=10
):
    (
        page_results,
        pagination_metadata
    ) = paginate_results(
        results=all_results,
        page=page,
        page_size=page_size
    )

    api_response = results_to_api_response(
        results=page_results,
        fuzzy_suggestions=fuzzy_suggestions,
        search_metadata=search_metadata
    )

    api_response[
        "pagination"
    ] = pagination_metadata

    return api_response

In [ ]:
# ============================================================
# TEST PAGINATED API RESPONSE
# ============================================================

import json

api_page = 1
api_page_size = 10

api_response = (
    paginated_results_to_api_response(
        all_results=(
            all_recommendation_results
        ),
        fuzzy_suggestions=(
            fuzzy_suggestions
        ),
        search_metadata=(
            search_metadata
        ),
        page=api_page,
        page_size=api_page_size
    )
)

print(
    json.dumps(
        api_response,
        ensure_ascii=False,
        indent=2
    )
)

{
  "query": "I am a complete beginner who wants to transition into data science. I need a practical book that teaches Python, data analysis, and the basic skills required to start a career in this field.",
  "search_mode": "semantic_query",
  "suggested_title": null,
  "reranking_used": false,
  "suggestions": [],
  "results": [
    {
      "rank": 1,
      "book_id": "BOOK_02847",
      "title": "Python for Data Analysis",
      "author": "Wes McKinney",
      "format": "Paperback",
      "genre": "Computer Science, Programming, Science, Technology, Nonfiction, Technical, Reference, Coding, Computers, Textbooks",
      "categories": "['Technology']",
      "description": "Python for Data Analysis, is concerned with the nuts and bolts of manipulating, processing, cleaning, and crunching data in Python. It is also a practical, modern introduction to scientific computing in Python, tailored for data-intensive applications. This is a book about the parts of the Python language and librar

In [ ]:
# ============================================================
# CREATE DEPLOYMENT DIRECTORY
# ============================================================
from pathlib import Path

deployment_root = Path(
    "book_recommendation_deployment"
)

data_directory = deployment_root / "data"
model_directory = deployment_root / "models"
config_directory = deployment_root / "config"

for directory in [
    data_directory,
    model_directory,
    config_directory
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

In [ ]:
# ============================================================
# PREPARE WEBSITE BOOK CATALOG
# ============================================================

website_catalog_columns = [
    "book_id",
    "title",
    "title_normalized",
    "author",
    "bookformat",
    "genre_text",
    "target_categories",
    "desc",
    "img",
    "link",
    "pages",
    "rating",
    "reviews",
    "totalratings",
    "isbn10_clean",
    "isbn13_clean",
    "model_text"
]

website_catalog_columns = [
    column
    for column in website_catalog_columns
    if column in df_books_model.columns
]

website_catalog = df_books_model[
    website_catalog_columns
].copy()

In [ ]:
def serialize_list_value(value):
    if isinstance(value, list):
        return json.dumps(
            value,
            ensure_ascii=False
        )

    return value


if "target_categories" in website_catalog.columns:
    website_catalog["target_categories"] = (
        website_catalog["target_categories"]
        .apply(serialize_list_value)
    )

In [ ]:
# Instal bila belum tersedia.
!pip install -q pyarrow

In [ ]:
website_catalog.to_parquet(
    data_directory / "books_catalog.parquet",
    index=False
)

website_catalog.to_csv(
    data_directory / "books_catalog.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Katalog tersimpan:",
    website_catalog.shape
)

Katalog tersimpan: (4083, 17)


In [ ]:
# ============================================================
# SAVE CORPUS EMBEDDINGS
# ============================================================

if len(website_catalog) != len(
    mpnet_embeddings
):
    raise ValueError(
        "Jumlah baris katalog tidak sama dengan "
        "jumlah corpus embedding."
    )

np.save(
    data_directory / "mpnet_embeddings.npy",
    np.asarray(
        mpnet_embeddings,
        dtype=np.float32
    )
)

print(
    "Shape embedding tersimpan:",
    mpnet_embeddings.shape
)

Shape embedding tersimpan: (4083, 768)


In [ ]:
# ============================================================
# SAVE TITLE SEARCH INDEX
# ============================================================

title_search_index = website_catalog[
    [
        "book_id",
        "title",
        "title_normalized"
    ]
].copy()

title_search_index.to_json(
    data_directory / "title_search_index.json",
    orient="records",
    force_ascii=False,
    indent=2
)

In [ ]:
# ============================================================
# SAVE MODELS LOCALLY
# ============================================================

best_embedding_model.save(
    str(
        model_directory
        / "multilingual_mpnet"
    )
)

optional_reranker.save_pretrained(
    str(
        model_directory
        / "multilingual_cross_encoder"
    ),
    safe_serialization=True
)

print("Model lokal berhasil disimpan.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model lokal berhasil disimpan.


In [ ]:
# ============================================================
# SAVE EVALUATION METRICS
# ============================================================

evaluation_metrics_records = (
    overall_evaluation_summary
    .to_dict(orient="records")
)

with open(
    config_directory / "evaluation_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        evaluation_metrics_records,
        file,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
# ============================================================
# SAVE DEPLOYMENT CONFIGURATION
# ============================================================

deployment_config = {
    "application_name": (
        "Nonfiction Book Recommendation System"
    ),

    "catalog_file": (
        "data/books_catalog.parquet"
    ),

    "embedding_file": (
        "data/mpnet_embeddings.npy"
    ),

    "title_index_file": (
        "data/title_search_index.json"
    ),

    "embedding_model_name": (
        "sentence-transformers/"
        "paraphrase-multilingual-mpnet-base-v2"
    ),

    "embedding_model_local_path": (
        "models/multilingual_mpnet"
    ),

    "reranker_model_name": (
        "cross-encoder/"
        "mmarco-mMiniLMv2-L12-H384-v1"
    ),

    "reranker_model_local_path": (
        "models/multilingual_cross_encoder"
    ),

    "default_use_reranker": False,

    # Retrieval dan pagination
    "top_k_candidates": 50,
    "default_page_size": 10,
    "maximum_page_size": 20,

    # Fuzzy title
    "fuzzy_score_cutoff": 85,
    "fuzzy_suggestion_limit": 5,
    "fuzzy_auto_accept_score": 92,
    "fuzzy_min_length_ratio": 0.65,
    "default_auto_accept_fuzzy": False,

    "embedding_normalized": True,
    "primary_metric": "ndcg_at_10",
    "selected_system": "MPNET_RETRIEVAL"
}

with open(
    config_directory / "deployment_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        deployment_config,
        file,
        ensure_ascii=False,
        indent=2
    )

In [ ]:
# ============================================================
# SAVE BOOK ORDER MANIFEST
# ============================================================

book_order_manifest = website_catalog[
    ["book_id"]
].copy()

book_order_manifest.insert(
    0,
    "corpus_index",
    np.arange(
        len(book_order_manifest)
    )
)

book_order_manifest.to_csv(
    data_directory / "book_order_manifest.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
# ============================================================
# VALIDATE DEPLOYMENT ARTIFACTS
# ============================================================

saved_catalog = pd.read_parquet(
    data_directory / "books_catalog.parquet"
)

saved_embeddings = np.load(
    data_directory / "mpnet_embeddings.npy",
    mmap_mode="r"
)

if len(saved_catalog) != len(saved_embeddings):
    raise ValueError(
        "Katalog dan embedding deployment "
        "tidak memiliki jumlah baris yang sama."
    )

if not saved_catalog["book_id"].is_unique:
    raise ValueError(
        "Book ID pada katalog deployment tidak unik."
    )

print("Artefak deployment valid.")
print("Jumlah buku:", len(saved_catalog))
print("Shape embedding:", saved_embeddings.shape)

Artefak deployment valid.
Jumlah buku: 4083
Shape embedding: (4083, 768)


In [ ]:
# ============================================================
# CREATE DEPLOYMENT ZIP
# ============================================================
import shutil

deployment_zip = shutil.make_archive(
    base_name="book_recommendation_deployment",
    format="zip",
    root_dir=str(deployment_root)
)

print(
    "Deployment ZIP:",
    deployment_zip
)

Deployment ZIP: /content/book_recommendation_deployment.zip


In [ ]:
# from google.colab import files

# files.download(
#     "book_recommendation_deployment.zip"
# )